# Correr cada inicio de sesión

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/isac2026')
for carpeta in ['datos_crudos', 'datos_parquet', 'salidas', 'codigo']:
    (BASE / carpeta).mkdir(parents=True, exist_ok=True)

print(sorted(p.name for p in BASE.iterdir()))

Mounted at /content/drive
['codigo', 'datos_crudos', 'datos_externos', 'datos_parquet', 'salidas']


In [ ]:
from google.colab import userdata

USUARIO = userdata.get('SB_USER')
CLAVE = userdata.get('SB_PASS')
print('Usuario cargado:', bool(USUARIO), '| Clave cargada:', bool(CLAVE))

#Sesión 13

In [ ]:
# [E-00 v1.0] Inspección previa a la re-descarga de 2026/27 (SOLO LECTURA: no escribe nada, no llama a la API, no calcula juego)
# =====================================================================
# Sesión 13 · 2026-10-19 · Preregistro de Almada, sección 4, paso 1 · Buena práctica: inspeccionar antes de asumir
# El código de descarga y conversión de las sesiones 1-3 (celdas D1, conversión y E3) y el de la tabla de técnicos
# (procesar) no están en el proyecto. Antes de escribir la re-descarga hay que saber EXACTAMENTE qué existe hoy:
#   (1) tablas de partidos (crudo y parquet): fechas de foto, columnas, cuántos de 2026/27 están jugados
#   (2) tabla de técnicos (equipo_partido_tecnico y partido_tecnico_periodo): columnas, tipos, filas por temporada
#   (3) calendario de América en 2026/27 según la tabla de técnicos (fechas e ids; NUNCA marcadores ni juego: D-2)
#   (4) qué partidos de 2026/27 tienen crudo (json.gz) y parquet (eventos, alineaciones, 360) y cuáles están en ptp
#   (5) registros de conversión (validacion_*.jsonl) y tablas derivadas con filas de 2026/27 (Elo, Elo de xG, marcador)
#   (6) esquema de un partido de 2026/27 ya convertido (plantilla para que los nuevos queden idénticos)
# Orden: celda 1 (Drive) → E-00. Tarda < 2 min.
# =====================================================================
import os, re, glob, json, gzip
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP, DC, SAL = os.path.join(RAIZ, 'datos_parquet'), os.path.join(RAIZ, 'datos_crudos'), os.path.join(RAIZ, 'salidas')
FW1, FW5, FW6 = (os.path.join(DP, f) for f in ('fw1', 'fw5', 'fw6'))
VIVO, SID_VIVO = '2026/2027', 351
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 60)
mid_de = lambda r: int(re.findall(r'\d+', os.path.basename(r))[0])
def titulo(t): print('\n' + '=' * 78 + f'\n{t}\n' + '=' * 78)

# ---------- (1) Tablas de partidos ----------
titulo('(1) Tablas de partidos (fotos guardadas)')
fotos = sorted(glob.glob(os.path.join(DC, 'matches_73_*')) + glob.glob(os.path.join(DP, 'matches_73_*')))
for f in fotos:
    print(f'  {os.path.relpath(f, RAIZ):60s} {os.path.getsize(f) / 1e6:7.2f} MB')
def leer_foto(f):
    if f.endswith('.parquet'):
        return pd.read_parquet(f)
    with (gzip.open if f.endswith('.gz') else open)(f, 'rt', encoding='utf-8') as h:
        return pd.json_normalize(json.load(h))
pq_fotos = [f for f in fotos if f.endswith('.parquet')]
if pq_fotos:
    M = leer_foto(pq_fotos[-1])
    print(f'\n  Última foto parquet: {os.path.basename(pq_fotos[-1])} · filas {len(M)} · columnas ({M.shape[1]}):')
    print('   ', list(M.columns))
    if 'season_label' in M:
        print('  season_id ↔ season_label:', M.drop_duplicates('season_id').set_index('season_id')['season_label'].to_dict())
v_json = [f for f in fotos if not f.endswith('.parquet') and f'_{SID_VIVO}_' in os.path.basename(f)]
if v_json:
    M3 = leer_foto(v_json[-1])
    jug = M3[M3['home_score'].notna()]
    print(f'\n  Última foto cruda de {VIVO}: {os.path.basename(v_json[-1])} · partidos {len(M3)} · jugados {len(jug)} · '
          f'último jugado {jug.match_date.max() if len(jug) else "—"}')
    print('  match_status:', M3['match_status'].value_counts().to_dict() if 'match_status' in M3 else '—',
          '· match_status_360 (jugados):', jug['match_status_360'].value_counts().to_dict() if 'match_status_360' in jug else '—')
    print('  ¿columnas de managers?', [c for c in M3.columns if 'managers' in c])
    print('  ¿columnas de actualización?', [c for c in M3.columns if 'updated' in c or 'last_' in c])
else:
    M3 = None
    print(f'\n  No hay foto cruda separada de {VIVO} (season {SID_VIVO}).')

# ---------- (2) Tabla de técnicos ----------
titulo('(2) Tabla de técnicos')
for nom in ['equipo_partido_tecnico.parquet', 'partido_tecnico_periodo.parquet', 'etapas_tecnico.parquet']:
    p = os.path.join(DP, nom)
    if not os.path.exists(p):
        print(f'  {nom}: NO EXISTE'); continue
    t = pd.read_parquet(p)
    print(f'\n  {nom}: {len(t)} filas · tipos: {dict(t.dtypes.astype(str))}')
    if 'temporada' in t:
        print('  filas por temporada:', t.groupby('temporada').size().to_dict())
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
v = ptp[ptp.temporada == VIVO].copy(); v['fecha_dt'] = pd.to_datetime(v['fecha'])
print(f'\n  ptp {VIVO}: partidos {v.match_id.nunique()} · primero {v.fecha_dt.min().date()} · último {v.fecha_dt.max().date()} · '
      f'roles {v.rol.value_counts().to_dict()}')
print('  Ejemplo de 2 filas (sin marcador):'); print(v.drop(columns='fecha_dt').head(2).to_string(index=False))

# ---------- (3) Calendario de América en 2026/27 (solo fechas e ids) ----------
titulo(f'(3) América en {VIVO} según ptp (calendario; sin marcador ni juego)')
am = v[v.team.str.startswith('Am')].sort_values('fecha_dt')
print(am[['match_id', 'fecha', 'lado', 'tec_p', 'etapa', 'rol']].to_string(index=False))
print(f'  Partidos de América: {len(am)} · técnicos: {am.tec_p.value_counts().to_dict()} · etapa_id: '
      f'{sorted(set(am.team + " · " + am.etapa.astype(str)))}')
if M3 is not None:
    es_am = M3['home_team.home_team_name'].str.startswith('Am') | M3['away_team.away_team_name'].str.startswith('Am')
    a3 = M3[es_am].sort_values('match_date')
    a3 = a3.assign(jugado=a3['home_score'].notna(), en_ptp=a3['match_id'].isin(set(ptp.match_id)))
    print(f'\n  América en la foto cruda {os.path.basename(v_json[-1])} (calendario completo de la foto):')
    print(a3[['match_id', 'match_date', 'jugado', 'en_ptp'] + [c for c in ['match_status_360', 'competition_stage.name'] if c in a3]]
          .to_string(index=False))

# ---------- (4) Cobertura de archivos por partido de 2026/27 ----------
titulo(f'(4) Archivos por partido de {VIVO}')
ids_ptp = set(v.match_id)
ids_foto = set(M3.loc[M3['home_score'].notna(), 'match_id'].astype(int)) if M3 is not None else set()
crudo = {s: {mid_de(f) for f in glob.glob(os.path.join(DC, 'liga', s, '*.json.gz'))} for s in ['eventos', 'lineups', 'frames360']}
parq = {s: {mid_de(f) for f in glob.glob(os.path.join(DP, 'liga', s, '*.parquet'))} for s in ['eventos', 'lineups', 'frames360', 'jugadores360']}
filas = []
for nom, ids in [('en ptp', ids_ptp), ('jugados en la foto cruda', ids_foto), ('en la foto y NO en ptp', ids_foto - ids_ptp),
                 ('América en ptp', set(am.match_id))]:
    f = {'grupo': nom, 'partidos': len(ids)}
    f.update({f'crudo_{s}': len(ids & c) for s, c in crudo.items()})
    f.update({f'parquet_{s}': len(ids & c) for s, c in parq.items()})
    filas.append(f)
print(pd.DataFrame(filas).to_string(index=False))
print('  Totales en carpetas (todas las temporadas): crudo', {s: len(c) for s, c in crudo.items()},
      '· parquet', {s: len(c) for s, c in parq.items()})

# ---------- (5) Registros y tablas derivadas ----------
titulo('(5) Registros de conversión y tablas derivadas con filas de 2026/27')
for f in sorted(glob.glob(os.path.join(SAL, 'validacion_*.jsonl')) + glob.glob(os.path.join(SAL, 'fallos_descarga_*.csv')) +
                glob.glob(os.path.join(SAL, 'ids_etapa3_*.csv')) + glob.glob(os.path.join(DP, 'resumen_partido_etapa*.parquet'))):
    try:
        t = pd.read_json(f, lines=True) if f.endswith('.jsonl') else (pd.read_csv(f) if f.endswith('.csv') else pd.read_parquet(f))
        n_v = int(t['match_id'].astype(int).isin(ids_ptp | ids_foto).sum()) if 'match_id' in t else '—'
        print(f'  {os.path.relpath(f, RAIZ):55s} filas {len(t):5d} · de 2026/27: {n_v}' +
              (f" · season_id {t['season_id'].value_counts().to_dict()}" if 'season_id' in t else ''))
    except Exception as e:
        print(f'  {os.path.relpath(f, RAIZ)}: no se pudo leer ({type(e).__name__})')
for carpeta in [FW1, FW5]:
    for nom in ['elo_partido_e2.parquet', 'elo_equipo_partido_e2.parquet', 'elo_xg_partido.parquet', 'tiros_xg.parquet']:
        p = os.path.join(carpeta, nom)
        if os.path.exists(p):
            t = pd.read_parquet(p)
            fv = t['match_id'].isin(ids_ptp)
            ult = pd.to_datetime(t.loc[fv, 'fecha']).max().date() if 'fecha' in t and fv.any() else '—'
            print(f'  {os.path.relpath(p, RAIZ):45s} filas {len(t):6d} · partidos de 2026/27 {t.loc[fv, "match_id"].nunique():3d} · último {ult}')
p = os.path.join(DP, 'equipo_partido_marcador.parquet')
t = pd.read_parquet(p, columns=['match_id'])
print(f'  equipo_partido_marcador.parquet: partidos de 2026/27 {t.match_id.isin(ids_ptp).sum() // 2}')
with open(os.path.join(FW5, 'parametros_fw1.json')) as h:
    print('  fw5/parametros_fw1.json temporadas:', json.load(h)['temporadas'])
print('  ¿Existe fw6?', os.path.isdir(FW6), '· ¿salidas/fw6?', os.path.isdir(os.path.join(SAL, 'fw6')))
for s in ['eventos', 'posesiones']:
    print(f'  R1 {s}:', {os.path.basename(d): sorted(os.path.basename(x) for x in glob.glob(os.path.join(d, f'{s}_*.parquet')))
                         for d in [FW1, FW5] if os.path.isdir(d)})

# ---------- (6) Esquema de un partido de 2026/27 ya convertido ----------
titulo('(6) Esquema de un partido ya convertido (plantilla)')
ej = sorted(ids_ptp & parq['eventos'] & parq['frames360'])
ej_prev = sorted((set(ptp.loc[ptp.temporada == '2025/2026', 'match_id'])) & parq['eventos'] & parq['frames360'])
for etiqueta, lst in [(VIVO, ej), ('2025/2026', ej_prev)]:
    if not lst:
        print(f'  {etiqueta}: sin partidos con eventos y 360'); continue
    m = lst[-1]
    print(f'\n  {etiqueta} · match_id {m}')
    for s in ['eventos', 'lineups', 'frames360', 'jugadores360']:
        a = os.path.join(DP, 'liga', s, f'{m}.parquet')
        if os.path.exists(a):
            sch = pq.read_schema(a)
            tipos = dict(zip(sch.names, map(str, sch.types)))
            txt = str(tipos) if s != 'eventos' else f'{len(sch.names)} columnas · v11: ' + str({c: c in tipos for c in ['control_degree', 'is_controlled_possession', 'possession_directness', 'set_piece_phase', 'obv_total_net']})
            print(f'    {s:13s} {txt[:600]}')
print('\nE-00 terminó (no escribió nada, no llamó a la API, no calculó juego).')

In [ ]:
# [E-1 v1.0] Re-descarga de 2026/27 con corte fijo 2026-10-19 (preregistro de Almada, sección 4, paso 1)
# =====================================================================
# Sesión 13 · 2026-10-19 · Base: celda E3 de la sesión 3 (descarga de la etapa 3), misma función bajar() sin cambios.
# Qué hace:
#   (1) Foto de la tabla de partidos de 2026/27 (season 351) con fecha 2026-10-19 en datos_crudos/. Si ya existe, NO se vuelve a
#       pedir: se lee la guardada (la foto queda congelada aunque se corra dos veces).
#   (2) Objetivo = partidos jugados (marcador no nulo) con fecha ≤ 2026-10-19 y match_status 'available' (procesados por el
#       proveedor). Los jugados que aún no están procesados NO entran y se listan.
#   (3) Calendario de América hasta el corte: cuáles entran y cuál falta (y por qué).
#   (4) Descarga crudos (json.gz) en datos_crudos/liga/{lineups,eventos,frames360}/ solo de lo que falta (bajar() salta lo que
#       ya existe: los 88 de la etapa 3 no se tocan). Incluye 360 que antes estaba 'unscheduled' y hoy está disponible.
#   (5) Avisa si el proveedor actualizó partidos que ya teníamos (last_updated posterior al 2026-10-07): NO se re-descargan
#       (se conserva la foto con fecha documentada, regla de la sesión 3); solo se reporta.
# Modo: SOLO_CONTAR = True primero (pide solo la tabla de partidos y muestra el plan); luego SOLO_CONTAR = False para bajar.
# Orden: celda 1 (Drive) → E-1. No escribe en fw1 ni en fw5. Tarda ~1 min contando; ~5-10 min descargando.
# =====================================================================
import os, gzip, json, time, requests
import pandas as pd
from google.colab import userdata

SOLO_CONTAR = False                                   # ← primero True; después False

URL_360     = "https://data.statsbomb.com/api/v2/360-frames/{match_id}"
URL_EVENTOS = "https://data.statsbomb.com/api/v11/events/{match_id}"
URL_MATCHES = "https://data.statsbomb.com/api/v6/competitions/{competition_id}/seasons/{season_id}/matches"
URL_LINEUPS = "https://data.statsbomb.com/api/v5/lineups/{match_id}"

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
assert 'PEGAR' not in URL_MATCHES + URL_LINEUPS + URL_EVENTOS + URL_360, 'Pega las cuatro URLs (las mismas de E3)'
AUTH = (userdata.get('SB_USER'), userdata.get('SB_PASS'))
SID, CORTE, HOY, FOTO_PREV = 351, '2026-10-19', '2026-10-19', '2026-10-07'
CRUDO = os.path.join(RAIZ, 'datos_crudos', 'liga'); SAL = os.path.join(RAIZ, 'salidas')

def bajar(url, ruta):                                # ← copia literal de E3 (sesión 3)
    if os.path.exists(ruta):
        return 'ya existía'
    estado = 'sin intento'
    for _ in range(3):
        try:
            r = requests.get(url, auth=AUTH, timeout=120)
        except Exception as e:
            estado = type(e).__name__; time.sleep(5); continue
        if r.status_code == 200:
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            with gzip.open(ruta, 'wb') as f:
                f.write(r.content)
            return 'ok'
        estado = f'HTTP {r.status_code}'
        if r.status_code == 404:
            break
        time.sleep(5)
    return estado

# ---------- (1) Foto de la tabla de partidos ----------
ruta_foto = os.path.join(RAIZ, 'datos_crudos', f'matches_73_{SID}_{HOY}.json.gz')
if os.path.exists(ruta_foto):
    origen = 'leída de Drive (ya congelada)'
else:
    r = requests.get(URL_MATCHES.format(competition_id=73, season_id=SID), auth=AUTH, timeout=120)
    r.raise_for_status()
    with gzip.open(ruta_foto, 'wb') as f:
        f.write(r.content)
    origen = 'pedida a la API y guardada'
with gzip.open(ruta_foto, 'rt', encoding='utf-8') as f:
    M = pd.json_normalize(json.load(f))
M['match_id'] = M['match_id'].astype(int)
print(f'--- (1) Foto {os.path.basename(ruta_foto)}: {origen} · partidos {len(M)} · jugados {int(M.home_score.notna().sum())} · '
      f'último jugado {M.loc[M.home_score.notna(), "match_date"].max()}')

# ---------- (2) Objetivo ----------
jug = M[M['home_score'].notna() & (M['match_date'] <= CORTE)].copy()
obj = jug[jug['match_status'] == 'available'].copy()
fuera = jug[jug['match_status'] != 'available']
existe = lambda t, m: os.path.exists(os.path.join(CRUDO, t, f'{m}.json.gz'))
obj['ya_eventos'] = [existe('eventos', m) for m in obj.match_id]
obj['ya_360'] = [existe('frames360', m) for m in obj.match_id]
nuevos = obj[~obj['ya_eventos']]
f360 = obj[obj['ya_eventos'] & ~obj['ya_360'] & (obj['match_status_360'] == 'available')]
print(f'--- (2) Jugados hasta el {CORTE}: {len(jug)} · procesados (entran): {len(obj)} · jugados sin procesar (no entran): {len(fuera)}')
print(f'    Ya descargados: {int(obj.ya_eventos.sum())} · nuevos a bajar: {len(nuevos)} (fechas {nuevos.match_date.min()} a {nuevos.match_date.max()}) · '
      f'360 de los nuevos: {nuevos.match_status_360.value_counts().to_dict()}')
print(f'    Partidos viejos cuyo 360 ahora está disponible (se agrega): {len(f360)}')
if len(fuera):
    print('    Sin procesar:', fuera[['match_id', 'match_date', 'match_status']].to_dict('records'))

# ---------- (3) América ----------
es_am = M['home_team.home_team_name'].str.startswith('Am') | M['away_team.away_team_name'].str.startswith('Am')
am = M[es_am & (M['match_date'] <= CORTE)].sort_values('match_date').copy()
am['jugado'] = am['home_score'].notna()
am['entra'] = am['match_id'].isin(obj['match_id'])
am['nota'] = ['entra' if e else ('no jugado / reprogramado' if not j else 'jugado, sin procesar') for e, j in zip(am.entra, am.jugado)]
print(f'--- (3) América hasta el {CORTE} en el calendario: {len(am)} · entran {int(am.entra.sum())} · faltan {int((~am.entra).sum())}')
print(am[['match_id', 'match_date', 'match_status', 'match_status_360', 'nota']].to_string(index=False))

# ---------- (5) Actualizaciones del proveedor sobre lo que ya teníamos ----------
col_upd = 'last_updated' if 'last_updated' in obj else None
if col_upd:
    viejos = obj[obj['ya_eventos']]
    n_upd = int((viejos[col_upd].astype(str) > FOTO_PREV).sum())
    print(f'--- (5) Ya descargados con {col_upd} posterior al {FOTO_PREV}: {n_upd} de {len(viejos)} (no se re-descargan; se reporta)')

# ---------- (4) Descarga ----------
if SOLO_CONTAR:
    print('\nSOLO_CONTAR = True: no se bajó nada (salvo la foto). Revisa el plan y vuelve a correr con SOLO_CONTAR = False.')
else:
    log = []
    for i, row in enumerate(nuevos.itertuples(), 1):
        mid = int(row.match_id)
        log.append((mid, 'lineups', bajar(URL_LINEUPS.format(match_id=mid), os.path.join(CRUDO, 'lineups', f'{mid}.json.gz'))))
        log.append((mid, 'eventos', bajar(URL_EVENTOS.format(match_id=mid), os.path.join(CRUDO, 'eventos', f'{mid}.json.gz'))))
        if row.match_status_360 == 'available':
            log.append((mid, 'frames360', bajar(URL_360.format(match_id=mid), os.path.join(CRUDO, 'frames360', f'{mid}.json.gz'))))
        if i % 25 == 0: print(f'   {i}/{len(nuevos)}')
    for mid in f360.match_id:
        log.append((int(mid), 'frames360', bajar(URL_360.format(match_id=mid), os.path.join(CRUDO, 'frames360', f'{mid}.json.gz'))))
    lg = pd.DataFrame(log, columns=['match_id', 'tipo', 'estado'])
    print('\n--- (4) Resultado de la descarga:'); print(pd.crosstab(lg['tipo'], lg['estado']).to_string() if len(lg) else '  nada que bajar')
    fallos = lg[~lg['estado'].isin(['ok', 'ya existía'])]
    fallos.to_csv(os.path.join(SAL, f'fallos_descarga_{HOY}.csv'), index=False)
    obj.assign(nuevo=~obj['ya_eventos'])[['match_id', 'match_date', 'match_status_360', 'nuevo']].to_csv(
        os.path.join(SAL, f'ids_e1_corte_{CORTE}.csv'), index=False)
    print(f'Fallos: {len(fallos)} · guardado salidas/ids_e1_corte_{CORTE}.csv ({len(obj)} partidos) y salidas/fallos_descarga_{HOY}.csv')
    am_ok = [m for m in am.loc[am.entra, 'match_id'] if existe('eventos', m)]
    print(f'América: partidos con eventos crudos en Drive hasta el corte: {len(am_ok)} de {int(am.entra.sum())} que entran')
print('\nE-1 terminó.')

In [ ]:
# [E-2 v1.0] Conversión a parquet de los partidos nuevos de 2026/27 (preregistro de Almada, sección 4, paso 1)
# =====================================================================
# Sesión 13 · 2026-10-19 · Funciones de conversión: COPIA LITERAL de la celda de conversión de la sesión 1 (etapa 1/2):
# leer, a_texto, anidado_a_texto, separar_coordenadas, aplanar_eventos, guardar, convertir_360, convertir_lineups, goles_equipos.
# Qué hace:
#   (A) PRUEBA DE REPRODUCCIÓN (buena práctica 13): reconvierte desde el crudo 2 partidos de 2026/27 que YA están en parquet
#       (incluido uno de América) en una carpeta temporal de Colab y exige que salgan IDÉNTICOS a los de Drive en las cuatro
#       tablas. Si no, se detiene sin escribir nada.
#   (B) Convierte SOLO los partidos nuevos de E-1 (salidas/ids_e1_corte_2026-10-19.csv, nuevo = True) a
#       datos_parquet/liga/{eventos,lineups,frames360,jugadores360}/. Nunca sobrescribe un parquet existente.
#       Registro propio: salidas/validacion_e1_2026-10-19.jsonl (no se mezcla con los de las etapas 1 y 2).
#   (C) Chequeos: goles de eventos = marcador, 2 Starting XI, frames 360 enlazados, columnas nuevas frente a las ya existentes.
# Orden: celda 1 (Drive) → E-2. No usa la API. Tarda ~1-2 min.
# =====================================================================
import os, gzip, json, time, glob
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
BASE_ = Path(RAIZ)
RAW = BASE_ / 'datos_crudos' / 'liga'
OUT = BASE_ / 'datos_parquet' / 'liga'
CORTE, SID = '2026-10-19', 351
LOG = BASE_ / 'salidas' / f'validacion_e1_{CORTE}.jsonl'
PRUEBA = Path('/content/e2_prueba')
PRUEBA_IDS = [4063611, 4063612]                     # 4063611 = América 28-sep (con 360); 4063612 = otro de 2026/27 ya convertido

# ---------- Funciones: copia literal de la sesión 1 (solo leer() recibe RAW igual que antes) ----------
def leer(tipo, mid):
    with gzip.open(RAW / tipo / f'{mid}.json.gz', 'rt', encoding='utf-8') as f:
        return json.load(f)

def a_texto(v):
    if v is None or (isinstance(v, float) and v != v):
        return None
    return json.dumps(v, ensure_ascii=False)

def anidado_a_texto(df):
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]):
            if df[c].apply(lambda v: isinstance(v, (list, dict))).any():
                df[c] = df[c].apply(a_texto)
    return df

def separar_coordenadas(df, col, pref):
    if col not in df.columns:
        return
    for i, eje in enumerate(['x', 'y', 'z']):
        vals = df[col].apply(lambda a: a[i] if isinstance(a, list) and len(a) > i else np.nan)
        if eje == 'z' and vals.isna().all():
            continue
        df[f'{pref}_{eje}'] = vals.astype('float32')
    df.drop(columns=[col], inplace=True)

def aplanar_eventos(raw, mid):
    df = pd.json_normalize(raw)
    df.insert(0, 'match_id', mid)
    for col, pref in [('location', 'loc'), ('pass.end_location', 'pass_end'),
                      ('carry.end_location', 'carry_end'), ('shot.end_location', 'shot_end')]:
        separar_coordenadas(df, col, pref)
    return anidado_a_texto(df)

def guardar(df, ruta):
    try:
        df.to_parquet(ruta, index=False, compression='zstd')
        return 'ok'
    except Exception:
        d = df.copy()
        for c in d.columns:
            if pd.api.types.is_object_dtype(d[c]):
                d[c] = d[c].apply(lambda v: None if v is None or (isinstance(v, float) and v != v) else str(v))
        d.to_parquet(ruta, index=False, compression='zstd')
        return 'respaldo_texto'

def convertir_360(raw, mapa, mid):
    fr, jg = [], []
    for f in raw:
        uuid = f.get('event_uuid')
        idx = mapa.get(uuid, -1)
        ff = f.get('freeze_frame') or []
        va = f.get('visible_area') or []
        mates = sum(1 for p in ff if p.get('teammate'))
        fr.append({
            'match_id': mid, 'event_index': idx, 'event_uuid': uuid,
            'line_breaking_pass': f.get('line_breaking_pass'),
            'num_defenders_on_goal_side_of_actor': f.get('num_defenders_on_goal_side_of_actor'),
            'distance_to_nearest_defender': f.get('distance_to_nearest_defender'),
            'ball_receipt_in_space': f.get('ball_receipt_in_space'),
            'ball_receipt_exceeds_distance': f.get('ball_receipt_exceeds_distance'),
            'n_visibles': len(ff), 'n_visibles_companeros': mates,
            'n_visibles_rivales': len(ff) - mates,
            'tiene_actor': any(p.get('actor') for p in ff),
            'area_visible_vacia': len(va) == 0,
            'visible_area': a_texto(va) if va else None,
            'visible_player_counts': a_texto(f.get('visible_player_counts')),
        })
        for p in ff:
            loc = p.get('location') or [np.nan, np.nan]
            jg.append((mid, idx, bool(p.get('teammate')), bool(p.get('actor')),
                       bool(p.get('keeper')), loc[0], loc[1]))
    frames = pd.DataFrame(fr)
    jug = pd.DataFrame(jg, columns=['match_id', 'event_index', 'teammate', 'actor', 'keeper', 'x', 'y'])
    jug = jug.astype({'match_id': 'int32', 'event_index': 'int32', 'x': 'float32', 'y': 'float32'})
    return frames, jug

def convertir_lineups(raw, mid):
    filas = []
    for eq in raw:
        for p in eq.get('lineup') or []:
            r = {'match_id': mid, 'team_id': eq.get('team_id'), 'team_name': eq.get('team_name'),
                 'equipo_formations': a_texto(eq.get('formations')),
                 'equipo_events': a_texto(eq.get('events'))}
            r.update(p)
            filas.append(r)
    return anidado_a_texto(pd.DataFrame(filas))

def goles_equipos(ev):
    e = ev[ev['period'] < 5]
    g = {}
    if 'shot.outcome.name' in e.columns:
        s = e[(e['type.name'] == 'Shot') & (e['shot.outcome.name'] == 'Goal')]
        for t, n in s['team.id'].value_counts().items():
            g[int(t)] = g.get(int(t), 0) + int(n)
    og = e[e['type.name'] == 'Own Goal For']
    for t, n in og['team.id'].value_counts().items():
        g[int(t)] = g.get(int(t), 0) + int(n)
    return g

def convertir_partido(mid, dest):
    """Mismo flujo de la sesión 1: eventos → mapa id→index → alineaciones → 360 (si hay crudo)."""
    ev = aplanar_eventos(leer('eventos', mid), mid)
    r_ev = guardar(ev, dest / 'eventos' / f'{mid}.parquet')
    mapa = dict(zip(ev['id'], ev['index']))
    lu = convertir_lineups(leer('lineups', mid), mid)
    guardar(lu, dest / 'lineups' / f'{mid}.parquet')
    tiene360 = (RAW / 'frames360' / f'{mid}.json.gz').exists()
    frames = None
    if tiene360:
        frames, jug = convertir_360(leer('frames360', mid), mapa, mid)
        guardar(frames, dest / 'frames360' / f'{mid}.parquet')
        guardar(jug, dest / 'jugadores360' / f'{mid}.parquet')
    return ev, lu, frames, r_ev

# ---------- (A) Prueba de reproducción ----------
print('--- (A) Reproducción: reconvertir partidos ya convertidos y comparar con Drive (deben ser idénticos) ---')
for c in ['eventos', 'frames360', 'jugadores360', 'lineups']:
    (PRUEBA / c).mkdir(parents=True, exist_ok=True)
ok_rep = True
for mid in PRUEBA_IDS:
    convertir_partido(mid, PRUEBA)
    for c in ['eventos', 'lineups', 'frames360', 'jugadores360']:
        a, b = OUT / c / f'{mid}.parquet', PRUEBA / c / f'{mid}.parquet'
        if not a.exists() and not b.exists():
            print(f'  {mid} {c:13s} sin archivo en ninguno (sin 360): ok'); continue
        A, Bn = pd.read_parquet(a), pd.read_parquet(b)
        try:
            pd.testing.assert_frame_equal(A, Bn, check_dtype=False, check_like=True)
            print(f'  {mid} {c:13s} idéntico · {A.shape[0]} filas × {A.shape[1]} columnas')
        except AssertionError as e:
            ok_rep = False
            print(f'  {mid} {c:13s} DISTINTO · columnas solo en Drive {sorted(set(A.columns) - set(Bn.columns))[:5]} · '
                  f'solo en la nueva {sorted(set(Bn.columns) - set(A.columns))[:5]} · {str(e)[:200]}')
assert ok_rep, 'La reconversión NO reproduce los parquet existentes: no se convierte nada. Manda esta salida a Claude.'

# ---------- (B) Conversión de los nuevos ----------
ids = pd.read_csv(BASE_ / 'salidas' / f'ids_e1_corte_{CORTE}.csv')
nuevos = ids.loc[ids['nuevo'].astype(bool), 'match_id'].astype(int).tolist()
with gzip.open(BASE_ / 'datos_crudos' / f'matches_73_{SID}_{CORTE}.json.gz', 'rt', encoding='utf-8') as f:
    M = pd.json_normalize(json.load(f)).set_index('match_id')
hechos = set()
if LOG.exists():
    hechos = {json.loads(l)['match_id'] for l in open(LOG) if l.strip()}
pend = [m for m in nuevos if m not in hechos and not (OUT / 'eventos' / f'{m}.parquet').exists()]
print(f'\n--- (B) Nuevos en E-1: {len(nuevos)} · ya convertidos: {len(nuevos) - len(pend)} · a convertir ahora: {len(pend)}')
errores, respaldo = [], 0
t0 = time.time()
with open(LOG, 'a') as flog:
    for mid in pend:
        try:
            ev, lu, frames, r_ev = convertir_partido(mid, OUT)
            respaldo += r_ev != 'ok'
            m = M.loc[mid]
            g = goles_equipos(ev)
            gl, gv = g.get(int(m['home_team.home_team_id']), 0), g.get(int(m['away_team.away_team_id']), 0)
            fila = {'match_id': mid, 'season_id': SID, 'n_eventos': int(len(ev)), 'n_columnas': int(ev.shape[1]),
                    'n_starting_xi': int((ev['type.name'] == 'Starting XI').sum()),
                    'periodo_max': int(ev['period'].max()),
                    'n_lineup_min_equipo': int(lu.groupby('team_id').size().min()) if len(lu) else 0,
                    'tiene_360': frames is not None, 'n_frames': 0 if frames is None else int(len(frames)),
                    'frames_sin_evento': 0 if frames is None else int((frames['event_index'] < 0).sum()),
                    'coincide_marcador': bool(gl == int(m['home_score']) and gv == int(m['away_score']))}
            flog.write(json.dumps(fila) + '\n'); flog.flush()
        except Exception as e:
            errores.append((mid, type(e).__name__, str(e)[:150]))
print(f'  convertidos en {time.time() - t0:.0f} s · errores {len(errores)} · eventos con respaldo de texto {respaldo}')
for e in errores[:10]: print('   ', e)

# ---------- (C) Chequeos ----------
val = pd.read_json(LOG, lines=True).drop_duplicates('match_id', keep='last') if LOG.exists() else pd.DataFrame()
print(f'\n--- (C) Registro {LOG.name}: {len(val)} partidos')
if len(val):
    print(f"  goles de eventos ≠ marcador: {int((~val.coincide_marcador).sum())} · Starting XI ≠ 2: {int((val.n_starting_xi != 2).sum())} · "
          f"periodo máximo {val.periodo_max.value_counts().to_dict()} · con 360: {int(val.tiene_360.sum())} · "
          f"frames 360 sin evento enlazado: {int(val.frames_sin_evento.sum())} de {int(val.n_frames.sum())} · mín. jugadores por equipo {int(val.n_lineup_min_equipo.min())}")
viejos = sorted(set(ids.loc[~ids['nuevo'].astype(bool), 'match_id'].astype(int)))[:40]
col_v = set().union(*[set(pq.read_schema(OUT / 'eventos' / f'{m}.parquet').names) for m in viejos])
tip_v = {}
for m in viejos:
    s = pq.read_schema(OUT / 'eventos' / f'{m}.parquet'); tip_v.update({n: str(t) for n, t in zip(s.names, s.types) if str(t) != 'null'})
col_n, conf = set(), {}
for m in nuevos:
    p = OUT / 'eventos' / f'{m}.parquet'
    if not p.exists(): continue
    s = pq.read_schema(p); col_n |= set(s.names)
    for n, t in zip(s.names, s.types):
        if str(t) != 'null' and n in tip_v and tip_v[n] != str(t): conf[n] = (tip_v[n], str(t))
print(f'  Columnas de eventos: nuevas que no estaban en 40 partidos previos de 2026/27: {sorted(col_n - col_v)[:10]} · '
      f'conflictos de tipo: {conf if conf else "ninguno"}')
am = [4063616, 4063629]
print('  América nuevos (11-oct, 18-oct):', {m: {c: (OUT / c / f'{m}.parquet').exists() for c in ['eventos', 'lineups', 'frames360']} for m in am})
print('\nE-2 terminó.')

In [ ]:
# [E-3 v1.0] Tabla de técnicos con 2026/27 al corte 2026-10-19, en datos_parquet/fw6/ (preregistro de Almada, sección 4, paso 2)
# =====================================================================
# Sesión 13 · 2026-10-19 · Funciones: COPIA LITERAL de la sesión 1: lado() (equipo_partido_tecnico) y procesar() (D-5: suspensión
# ≤ 5 partidos A-B-A → titular; interino ≤ 3 entre dos técnicos distintos; la última racha nunca es interina).
# Qué hace:
#   (1) Filas de 2021/22 a 2025/26: las de equipo_partido_tecnico.parquet tal cual. Filas de 2026/27: rehechas con lado() desde la
#       foto del 2026-10-19 (E-1). Compara las de 2026/27 que ya existían (técnico registrado) con las nuevas.
#   (2) procesar() sobre todo, igual que en la sesión 1.
#   (3) VALIDACIÓN OBLIGATORIA: en 2021/22-2025/26 la tabla nueva debe ser IDÉNTICA a la de Drive (cada columna, cada equipo-partido).
#       En los 88 partidos de 2026/27 ya existentes se reporta cualquier cambio (puede pasar: una racha que era la última deja de serlo).
#   (4) Escribe SOLO en datos_parquet/fw6/: partido_tecnico_periodo.parquet, etapas_tecnico.parquet, equipo_partido_tecnico.parquet.
#       La tabla de Drive (datos_parquet/partido_tecnico_periodo.parquet) no se toca.
# Orden: celda 1 (Drive) → E-1 ya corrido → E-3. No usa la API. Tarda < 30 s.
# =====================================================================
import os, gzip, json
import numpy as np
import pandas as pd

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW6 = os.path.join(DP, 'fw6')
VIVO, SID, CORTE = '2026/2027', 351, '2026-10-19'
UMBRAL_SUSP, UMBRAL_INT = 5, 3                    # D-5 (sesión 1)
pd.set_option('display.width', 220)

# ---------- lado(): copia literal de la sesión 1 ----------
def n_tec(m):
    try: return len(m)
    except Exception: return 0

def campo(m, k):
    try: return m[0][k] if len(m) > 0 else None
    except Exception: return None

def lado(df, s):
    o = pd.DataFrame({
        'match_id': df['match_id'], 'fecha': df['match_date'],
        'temporada': df['season_label'], 'etapa': df['competition_stage.name'],
        'jugado': df['home_score'].notna(),
        'team': df[f'{s}_team.{s}_team_name'],
        'n_tec': df[f'{s}_team.managers'].apply(n_tec),
        'tec_id': df[f'{s}_team.managers'].apply(lambda m: campo(m, 'id')),
        'tec': df[f'{s}_team.managers'].apply(lambda m: campo(m, 'name')),
    })
    o['lado'] = s
    return o

# ---------- procesar(): copia literal de la sesión 1 ----------
def procesar(g):
    g = g.sort_values(['fecha', 'match_id']).reset_index(drop=True)
    g['tec_id_p'] = g['tec_id']
    g['tec_p'] = g['tec']
    g['rol'] = 'titular'
    while True:
        g['etapa'] = (g['tec_id_p'] != g['tec_id_p'].shift()).cumsum()
        r = g.groupby('etapa').agg(n=('match_id', 'count'), tid=('tec_id_p', 'first'))
        ids = r.index.tolist()
        movido = False
        for k in range(1, len(ids) - 1):
            a, b, c = r.iloc[k - 1], r.iloc[k], r.iloc[k + 1]
            if a['tid'] == c['tid'] and b['tid'] != a['tid'] and b['n'] <= UMBRAL_SUSP:
                m = g['etapa'] == ids[k]
                nombre = g.loc[g['etapa'] == ids[k - 1], 'tec_p'].iloc[0]
                g.loc[m, 'rol'] = 'sustituto_suspension'
                g.loc[m, 'tec_id_p'] = a['tid']
                g.loc[m, 'tec_p'] = nombre
                movido = True
                break
        if not movido:
            break
    tam = g.groupby('etapa')['match_id'].transform('count')
    ultima = g['etapa'].max()
    g.loc[(tam <= UMBRAL_INT) & (g['etapa'] < ultima) & (g['rol'] == 'titular'), 'rol'] = 'interino'
    return g

# ---------- (1) Filas ----------
tm_old = pd.read_parquet(os.path.join(DP, 'equipo_partido_tecnico.parquet'))
with gzip.open(os.path.join(RAIZ, 'datos_crudos', f'matches_73_{SID}_{CORTE}.json.gz'), 'rt', encoding='utf-8') as f:
    M = pd.json_normalize(json.load(f))
M['season_label'] = VIVO
M = M[M['match_date'] <= CORTE]                                       # corte fijo (los programados después no entran)
tm_new = pd.concat([lado(M, 'home'), lado(M, 'away')], ignore_index=True)
tm_new['match_id'] = tm_new['match_id'].astype('int64'); tm_new['tec_id'] = tm_new['tec_id'].astype(float)
prev = tm_old[tm_old.temporada == VIVO]
cmp = prev[prev.jugado].merge(tm_new, on=['match_id', 'team'], how='left', suffixes=('_ant', '_nue'))
dif = cmp[(cmp.tec_id_ant != cmp.tec_id_nue) | (cmp.n_tec_ant != cmp.n_tec_nue)]
print(f'--- (1) 2026/27: filas jugadas en la tabla anterior {int(prev.jugado.sum())} · en la nueva (corte {CORTE}) {int(tm_new.jugado.sum())} · '
      f'técnico registrado distinto en filas que ya existían: {len(dif)}')
if len(dif): print(dif[['match_id', 'fecha_ant', 'team', 'tec_ant', 'tec_nue']].to_string(index=False))
tm = pd.concat([tm_old[tm_old.temporada != VIVO], tm_new], ignore_index=True)

# ---------- (2) procesar ----------
j = tm[tm['jugado'] & (tm['n_tec'] > 0)].copy()
jj = pd.concat([procesar(g) for _, g in j.groupby('team')], ignore_index=True)
ep = (jj.groupby(['team', 'etapa'])
        .agg(tec=('tec_p', 'first'), partidos=('match_id', 'count'),
             de_sustituto=('rol', lambda s: (s == 'sustituto_suspension').sum()),
             desde=('fecha', 'min'), hasta=('fecha', 'max'),
             rol=('rol', lambda s: 'interino' if (s == 'interino').any() else 'titular'))
        .reset_index())

# ---------- (3) Validación ----------
ptp_d = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
k = ['match_id', 'team']; cols = [c for c in ptp_d.columns if c not in k]
J = ptp_d.merge(jj, on=k, how='outer', suffixes=('_d', '_n'), indicator=True)
hist = J['temporada_d'].fillna(J['temporada_n']) != VIVO
dist = pd.Series(False, index=J.index)
for c in cols:
    a, b = J[f'{c}_d'], J[f'{c}_n']
    dist |= ~((a == b) | (a.isna() & b.isna()))
h = J[hist]
print(f'\n--- (3) Validación 2021/22-2025/26 · filas Drive {int((ptp_d.temporada != VIVO).sum())} · nueva {int((jj.temporada != VIVO).sum())} · '
      f'solo en una {int((h._merge != "both").sum())} · con alguna columna distinta {int(dist[hist & (J._merge == "both")].sum())} (las dos deben ser 0)')
OK = bool((h._merge == 'both').all() and not dist[hist & (J._merge == 'both')].any())
v88 = J[~hist & (J._merge == 'both')]
cam = v88[dist[~hist & (J._merge == 'both')]]
print(f'2026/27 · equipo-partidos ya existentes {len(v88)} · con cambios {len(cam)} · nuevos {int((J[~hist]._merge == "right_only").sum())} · '
      f'desaparecidos {int((J[~hist]._merge == "left_only").sum())}')
if len(cam): print(cam[['match_id', 'team', 'fecha_d', 'tec_p_d', 'tec_p_n', 'etapa_d', 'etapa_n', 'rol_d', 'rol_n']].to_string(index=False))
v = jj[jj.temporada == VIVO]
print(f'2026/27 nueva · partidos {v.match_id.nunique()} · último {v.fecha.max()} · roles {v.rol.value_counts().to_dict()}')
am = v[v.team.str.startswith('Am')].sort_values('fecha')
print(f"América 2026/27 · partidos {len(am)} · {am.tec_p.value_counts().to_dict()} · etapa_id {sorted(set(am.team + ' · ' + am.etapa.astype(str)))} · "
      f"roles {am.rol.value_counts().to_dict()} · fechas {am.fecha.min()} a {am.fecha.max()}")

# ---------- (4) Guardar en fw6 ----------
assert OK, 'La tabla nueva NO reproduce la histórica: no se escribe nada. Manda esta salida a Claude.'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
os.makedirs(FW6, exist_ok=True)
jj.to_parquet(os.path.join(FW6, 'partido_tecnico_periodo.parquet'))
ep.to_parquet(os.path.join(FW6, 'etapas_tecnico.parquet'))
tm.to_parquet(os.path.join(FW6, 'equipo_partido_tecnico.parquet'))
print(f'\nVALIDACIÓN OK · guardado en datos_parquet/fw6/: partido_tecnico_periodo ({len(jj)} filas), etapas_tecnico ({len(ep)}), '
      f'equipo_partido_tecnico ({len(tm)}). La tabla de Drive no se tocó.')

In [ ]:
# [E-4 v1.0] R1 (objetos base) de 2026/27 al corte 2026-10-19, escrito en datos_parquet/fw6/ (preregistro, sección 4, paso 2)
# =====================================================================
# Sesión 13 · 2026-10-19 · procesar_partido(), COLS y BOOLS: COPIA LITERAL de claude/R1_objetos_base_framework_v1.py.
# Parámetros: se LEEN de datos_parquet/fw1/parametros_fw1.json (lo que escribió R0), sin volver a correr R0 (R0 escribe en fw1).
# Qué hace:
#   (A) PRUEBA DE REPRODUCCIÓN: procesa 2 partidos de 2026/27 que ya están en fw1/eventos_2026-2027 y exige que eventos y
#       posesiones salgan IDÉNTICOS. Si no, se detiene sin escribir.
#   (B) Procesa los 106 partidos de 2026/27 de la tabla de técnicos de fw6 y escribe fw6/eventos_2026-2027.parquet y
#       fw6/posesiones_2026-2027.parquet. fw1 no se toca.
#   (C) Validación completa: en los 88 partidos que ya existían, fw6 debe ser idéntico a fw1. Resumen compacto de los 18 nuevos.
# Orden: celda 1 (Drive) → E-2 y E-3 ya corridos → E-4. Tarda ~2-4 min.
# =====================================================================
import os, glob, re, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); LIGA = os.path.join(DP, 'liga')
FW1, FW6 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw6')
VIVO = '2026/2027'; T_ = VIVO.replace('/', '-')
PRUEBA_IDS = [4063611, 4063612]
assert os.path.exists(os.path.join(FW6, 'partido_tecnico_periodo.parquet')), 'Falta fw6/partido_tecnico_periodo: corre E-3 primero'

with open(os.path.join(FW1, 'parametros_fw1.json')) as f:          # lo que fijó R0 (sesión 4)
    CFG = json.load(f)
P = CFG['P']
REANUD_PASE, REANUD_TIRO = CFG['reanud_pase'], CFG['reanud_tiro']
PATRON_RECUPERACION, TIPOS_CONTRAGOLPE = CFG['patron_recuperacion'], CFG['tipos_contragolpe']

# ---------- COPIA LITERAL DE R1 (COLS, BOOLS, procesar_partido) ----------
COLS = ["index", "period", "timestamp", "minute", "second", "duration", "type.name",
        "possession", "possession_team.id", "possession_team.name", "play_pattern.name",
        "team.id", "team.name", "player.id", "position.id", "loc_x", "loc_y",
        "under_pressure", "counterpress", "set_piece_phase", "location_bucket",
        "control_degree", "is_controlled_possession", "possession_directness",
        "pass.length", "pass.angle", "pass.type.name", "pass.outcome.name", "pass.height.name",
        "pass.recipient.id", "pass.cross", "pass.switch", "pass.pass_success_probability",
        "pass_end_x", "pass_end_y", "carry_end_x", "carry_end_y", "ball_receipt.outcome.name",
        "shot.statsbomb_xg", "shot.type.name", "shot.outcome.name",
        "duel.type.name", "duel.outcome.name", "interception.outcome.name",
        "ball_recovery.recovery_failure",
        "foul_committed.offensive", "foul_committed.card.name", "bad_behaviour.card.name",
        "substitution.replacement.id", "substitution.outcome.name", "tactics.formation",
        "out", "off_camera", "obv_total_net"]
BOOLS = ["under_pressure", "counterpress", "pass.cross", "pass.switch",
         "ball_recovery.recovery_failure", "foul_committed.offensive", "out", "off_camera"]


def procesar_partido(mid, ruta):
    nombres = set(pq.read_schema(ruta).names)
    ev = pd.read_parquet(ruta, columns=[c for c in COLS if c in nombres])
    for c in COLS:
        if c not in ev.columns:
            ev[c] = np.nan
    ev = ev.sort_values("index").reset_index(drop=True)
    ev.insert(0, "match_id", mid)
    ev["v11"] = "control_degree" in nombres
    for c in BOOLS:
        ev[c] = ev[c].eq(True)
    ev["t"] = pd.to_timedelta(ev["timestamp"]).dt.total_seconds()   # segundos dentro del periodo
    ev = ev.drop(columns="timestamp")
    tipo = ev["type.name"]

    # O-2 · reanudaciones (pases y tiros)
    ev["es_reanudacion"] = ev["pass.type.name"].isin(REANUD_PASE) | ev["shot.type.name"].isin(REANUD_TIRO)

    # O-1 + S4-05 · el balón parado es del equipo que lo ejecuta y acaba cuando el rival controla
    marca = ev["set_piece_phase"].notna()
    dueno = ev["team.id"].where(ev["es_reanudacion"] & marca)
    ev["bp_equipo"] = dueno.groupby(ev["period"]).ffill()
    ev["bp_sin_dueno"] = marca & ev["bp_equipo"].isna()
    ev["es_bp"] = marca & ((ev["possession_team.id"] == ev["bp_equipo"]) | ev["bp_sin_dueno"])

    # O-3 / O-12 · tabla de posesiones y su tiempo (B decidido; C sensibilidad)
    cb = ev[ev["loc_x"].notna()]
    g = cb.groupby("possession")
    pos = pd.DataFrame({
        "period": g["period"].first(),
        "equipo_id": g["possession_team.id"].first(),
        "equipo": g["possession_team.name"].first(),
        "patron": g["play_pattern.name"].first(),
        "primer_tipo": g["type.name"].first(),
        "primer_reanudacion": g["es_reanudacion"].first().astype(bool),
        "inicia_bp": g["es_bp"].first().astype(bool),
        "t0": g["t"].min(),
        "t1": (cb["t"] + cb["duration"].fillna(0)).groupby(cb["possession"]).max(),
        "controlada": g["is_controlled_possession"].first(),
        "directness": g["possession_directness"].first(),
    })
    pos["dur_B"] = (pos["t1"] - pos["t0"]).clip(lower=0)
    t_sig = g["t"].shift(-1)
    dt = (t_sig - cb["t"]).clip(lower=0)
    ultimo = t_sig.isna()
    dt = dt.where(~ultimo, cb["duration"].fillna(0))
    dt = dt.where(ultimo | (dt <= P["tope_intervalo_C_s"]), 0)
    pos["dur_C"] = dt.groupby(cb["possession"]).sum()

    # O-6 · posesión que nace de una recuperación en juego abierto
    pos["nace_rec"] = (pos["patron"].isin(PATRON_RECUPERACION)
                       & ~pos["primer_reanudacion"] & (pos["period"] <= 4))

    # O-7 · transición (se guarda t_pos para poder probar 15 s sin recalcular)
    ev = ev.join(pos[["t0", "nace_rec"]], on="possession")
    ev["nace_rec"] = ev["nace_rec"].eq(True)
    ev["t_pos"] = ev["t"] - ev["t0"]
    ev["en_transicion"] = ev["nace_rec"] & ev["t_pos"].between(0, P["transicion_s"])

    # O-11 · jerarquía de fases: balón parado > transición > establecido
    ev["fase"] = np.select([ev["es_bp"], ev["en_transicion"]], ["bp", "transicion"], default="establecido")

    # O-8 · contragolpe
    propio = ev["team.id"] == ev["possession_team.id"]
    m_cg = (ev["en_transicion"] & propio & (ev["loc_x"] >= P["tercio_ultimo_x"])
            & tipo.isin(TIPOS_CONTRAGOLPE))
    pos["contragolpe"] = pos.index.isin(ev.loc[m_cg, "possession"].unique()) & pos["nace_rec"]

    # xG sin penales y tiros por posesión
    tiro = (tipo == "Shot") & (ev["shot.type.name"] != "Penalty")
    pos["xg_np"] = ev.loc[tiro].groupby("possession")["shot.statsbomb_xg"].sum().reindex(pos.index).fillna(0)
    pos["tiros"] = ev.loc[tiro].groupby("possession").size().reindex(pos.index).fillna(0).astype(int)

    # O-9 · pausa desde la presión anterior del mismo equipo en la misma posesión rival
    pr = ev.loc[tipo == "Pressure"]
    ev["presion_gap_prev"] = pr.groupby(["possession", "team.id"])["t"].diff()

    # O-13 · diferencia de goles ANTES del evento, vista desde el equipo del evento
    gol = (((tipo == "Shot") & (ev["shot.outcome.name"] == "Goal") & (ev["period"] <= 4))
           | (tipo == "Own Goal For"))
    equipos = list(ev["team.id"].dropna().unique())
    if len(equipos) == 2:
        a, b = equipos
        prev = {}
        for e in equipos:
            ge = (gol & (ev["team.id"] == e)).astype(int)
            prev[e] = ge.cumsum() - ge
        ev["dif_antes"] = np.where(ev["team.id"] == a, prev[a] - prev[b],
                                   np.where(ev["team.id"] == b, prev[b] - prev[a], np.nan))
    else:
        ev["dif_antes"] = np.nan

    # P-9 · ventana de 5 minutos (0-8 primer tiempo con añadido en la 8; 9-17 segundo; 18 = añadido 2T)
    m = ev["minute"]
    ev["ventana"] = np.select(
        [ev["period"] == 1, ev["period"] == 2],
        [(m // 5).clip(upper=8), np.where(m >= 90, 18, ((m - 45) // 5).clip(lower=0, upper=8) + 9)],
        default=-1)

    pos = pos.reset_index()
    pos.insert(0, "match_id", mid)
    ev = ev.drop(columns=["t0"])
    return ev, pos
# ---------- FIN DE LA COPIA LITERAL ----------

def igual(A, B):
    try:
        pd.testing.assert_frame_equal(A.reset_index(drop=True), B.reset_index(drop=True), check_dtype=False, check_like=True)
        return True, ''
    except AssertionError as e:
        return False, str(e)[:250]

ruta = lambda m: os.path.join(LIGA, 'eventos', f'{m}.parquet')
E1 = pd.read_parquet(os.path.join(FW1, f'eventos_{T_}.parquet'))
P1 = pd.read_parquet(os.path.join(FW1, f'posesiones_{T_}.parquet'))

# ---------- (A) Reproducción ----------
print('--- (A) Reproducción contra fw1 (deben ser idénticos) ---')
ok = True
for mid in PRUEBA_IDS:
    e, p = procesar_partido(mid, ruta(mid))
    r1, m1 = igual(E1[E1.match_id == mid].sort_values('index'), e.sort_values('index'))
    r2, m2 = igual(P1[P1.match_id == mid].sort_values('possession'), p.sort_values('possession'))
    ok &= r1 and r2
    print(f'  {mid}: eventos {"idénticos" if r1 else "DISTINTOS " + m1} · posesiones {"idénticas" if r2 else "DISTINTAS " + m2}')
assert ok, 'R1 copiado NO reproduce fw1: no se escribe nada. Manda esta salida a Claude.'

# ---------- (B) Todos los partidos de 2026/27 de fw6 ----------
ptp6 = pd.read_parquet(os.path.join(FW6, 'partido_tecnico_periodo.parquet'))
ids = sorted(ptp6.loc[ptp6.temporada == VIVO, 'match_id'].unique())
evs, poss, fallos = [], [], []
t0 = time.time()
for mid in ids:
    try:
        e, p = procesar_partido(int(mid), ruta(int(mid))); evs.append(e); poss.append(p)
    except Exception as err:
        fallos.append((mid, repr(err)[:160]))
E6, P6 = pd.concat(evs, ignore_index=True), pd.concat(poss, ignore_index=True)
print(f'\n--- (B) 2026/27: {len(evs)} de {len(ids)} partidos · fallos {len(fallos)} · {len(E6):,} eventos · {len(P6):,} posesiones · {time.time() - t0:.0f} s')
for f_ in fallos[:5]: print('   FALLO', f_)
assert not fallos, 'Hubo fallos: no se escribe nada. Manda esta salida a Claude.'

# ---------- (C) Validación completa contra fw1 y resumen ----------
viejos = sorted(set(E1.match_id) & set(ids))
faltan = sorted(set(E1.match_id) - set(ids))
print(f'Partidos de fw1 que NO están en la tabla de fw6: {len(faltan)} {faltan[:5]} (debe ser 0)')
rE, mE = igual(E1[E1.match_id.isin(viejos)].sort_values(['match_id', 'index']), E6[E6.match_id.isin(viejos)].sort_values(['match_id', 'index']))
rP, mP = igual(P1[P1.match_id.isin(viejos)].sort_values(['match_id', 'possession']), P6[P6.match_id.isin(viejos)].sort_values(['match_id', 'possession']))
print(f'--- (C) Los {len(viejos)} partidos que ya estaban en fw1: eventos {"idénticos" if rE else "DISTINTOS " + mE} · '
      f'posesiones {"idénticas" if rP else "DISTINTAS " + mP}')
assert rE and rP and not faltan, 'fw6 no coincide con fw1 en los partidos viejos: no se escribe nada. Manda esta salida a Claude.'
nv = E6[~E6.match_id.isin(viejos)]; pn = P6[~P6.match_id.isin(viejos)]
est = nv[(nv.period <= 2) & nv.loc_x.notna()]
fases = (est.fase.value_counts(normalize=True) * 100).round(1).to_dict()
pm = pn[pn.period <= 2].groupby('match_id').dur_B.sum() / 60
print(f'  Nuevos: {nv.match_id.nunique()} partidos · eventos por partido {len(nv) / max(1, nv.match_id.nunique()):.0f} · % eventos por fase {fases} · '
      f'posesión B por partido {pm.mean():.1f} min · v11 en todos: {bool(nv.groupby("match_id").v11.first().all())} · '
      f'bp sin dueño {int(nv.bp_sin_dueno.sum())}')
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
E6.to_parquet(os.path.join(FW6, f'eventos_{T_}.parquet'), index=False)
P6.to_parquet(os.path.join(FW6, f'posesiones_{T_}.parquet'), index=False)
print(f'\nVALIDACIÓN OK · guardado fw6/eventos_{T_}.parquet y fw6/posesiones_{T_}.parquet. fw1 no se tocó.')

In [ ]:
# [E-5 v1.0] Elo (E-2) y Elo de xG extendidos a los partidos de 2026/27 hasta el 2026-10-19, en datos_parquet/fw6/
# =====================================================================
# Sesión 13 · 2026-10-19 · Preregistro de Almada, sección 4, paso 3 · Base: R3 (sesión 5) y E11b (sesión 9).
# COPIAS LITERALES de R3: _leer_matches, _col, CANDIDATAS, cargar_matches (R3-0); armado de la tabla de partidos (R3a);
# correr_elo (R3-0, versión de E11b con arranque como argumento); lectura de tiros, _dist_goles y correr_elo_s (R3e).
# Configuración congelada (bitácora v8 §3): K 20 · H 75 · R verano 0.25 · R invierno 0 · arranque por tabla.
# El arranque se toma del propio archivo de fw1 (Elo con el que cada equipo de 2021/22 jugó su primer partido), como en E11b.
# Qué hace:
#   (A) Arma la tabla de partidos con la tabla de técnicos de fw6 (1,807 partidos) y la foto de partidos más reciente.
#   (B) REPRODUCCIÓN: el Elo y el Elo de xG recalculados deben ser IDÉNTICOS a los de fw1 en los 1,789 partidos que ya estaban
#       (diferencia máxima < 1e-6). Si no, se detiene sin escribir.
#   (C) Escribe en fw6: elo_partido_e2, elo_equipo_partido_e2, elo_xg_partido, tiros_xg. fw1 no se toca.
# REGLA DE HOLDOUT (R3): no imprime resultados de 2025/26 ni de 2026/27; solo conteos y diferencias.
# Orden: celda 1 (Drive) → E-3 ya corrido → E-5. Tarda ~1-2 min.
# =====================================================================
import os, re, glob, json, gzip, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
BASE = RAIZ
DP = os.path.join(BASE, 'datos_parquet'); DC = os.path.join(BASE, 'datos_crudos')
FW1, FW6 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw6')
E2 = dict(K=20, H=75, R_ver=0.25, R_inv=0.0)
INICIO = 1500.0
CALENTAMIENTO = pd.Timestamp('2021-10-01')
VIVO = '2026/2027'
t0 = time.time()

# ---------- Copia literal de R3-0: carga de partidos ----------
def _leer_matches(ruta):
    if ruta.endswith('.parquet'):
        df = pd.read_parquet(ruta)
        obj = [c for c in df.columns if df[c].dtype == object]
        if any(df[c].map(lambda v: isinstance(v, dict)).any() for c in obj):
            df = pd.json_normalize(df.to_dict('records'))
    else:
        abrir = gzip.open if ruta.endswith('.gz') else open
        with abrir(ruta, 'rt', encoding='utf-8') as f:
            df = pd.json_normalize(json.load(f))
    m = re.search(r'(\d{4}-\d{2}-\d{2})', os.path.basename(ruta))
    df['_fecha_archivo'] = m.group(1) if m else '0000-00-00'
    df['_archivo'] = os.path.basename(ruta)
    return df

def _col(df, opciones):
    for o in opciones:
        if o in df.columns:
            return o
    return None

CANDIDATAS = {
    'match_id': ['match_id'],
    'fecha_m': ['match_date'],
    'hora': ['kick_off'],
    'local_id': ['home_team.home_team_id', 'home_team_id'],
    'local': ['home_team.home_team_name', 'home_team_name'],
    'visita_id': ['away_team.away_team_id', 'away_team_id'],
    'visita': ['away_team.away_team_name', 'away_team_name'],
    'gl': ['home_score'], 'gv': ['away_score'],
    'estadio': ['stadium.name', 'stadium_name', 'stadium'],
}

def cargar_matches():
    rutas = sorted(glob.glob(os.path.join(DC, 'matches_73_*.json*')) +
                   glob.glob(os.path.join(DP, 'matches_73_*.parquet')))
    piezas = []
    for r in rutas:
        try:
            df = _leer_matches(r)
        except Exception as e:
            print(f'  aviso: no se pudo leer {os.path.basename(r)}: {e}')
            continue
        cols = {k: _col(df, v) for k, v in CANDIDATAS.items()}
        faltan = [k for k in ['match_id', 'local', 'visita', 'gl', 'gv'] if cols[k] is None]
        if faltan:
            print(f'  aviso: {os.path.basename(r)} sin columnas {faltan}; se omite')
            continue
        out = pd.DataFrame({k: (df[c] if c else np.nan) for k, c in cols.items()})
        if out['estadio'].map(lambda v: isinstance(v, dict)).any():
            out['estadio'] = out['estadio'].map(lambda v: v.get('name') if isinstance(v, dict) else v)
        out['_fecha_archivo'] = df['_fecha_archivo']; out['_archivo'] = df['_archivo']
        piezas.append(out)
    M = pd.concat(piezas, ignore_index=True)
    M = (M.sort_values('_fecha_archivo', ascending=False)
           .drop_duplicates('match_id', keep='first'))
    return M, rutas

# ---------- Copia literal de E11b: correr_elo con arranque como argumento (devuelve también post) ----------
def correr_elo(P, K, H, R_ver, R_inv, ini):
    rating = dict(ini); temp_act = torneo_act = None
    pre_l = np.empty(len(P)); pre_v = np.empty(len(P)); post_l = np.empty(len(P)); post_v = np.empty(len(P))
    for i, r in enumerate(P.itertuples(index=False)):
        if temp_act is not None and r.temporada != temp_act:
            for t in rating: rating[t] = INICIO + (1 - R_ver) * (rating[t] - INICIO)
        elif torneo_act is not None and r.torneo != torneo_act:
            for t in rating: rating[t] = INICIO + (1 - R_inv) * (rating[t] - INICIO)
        temp_act, torneo_act = r.temporada, r.torneo
        rl = rating.get(r.local_id, INICIO); rv = rating.get(r.visita_id, INICIO)
        E = 1.0 / (1.0 + 10 ** (-(rl + H - rv) / 400))
        dif = r.gl - r.gv
        S = 1.0 if dif > 0 else (0.5 if dif == 0 else 0.0)
        n = abs(dif)
        G = 1.0 if n <= 1 else (1.5 if n == 2 else (11 + n) / 8)
        delta = K * G * (S - E)
        rating[r.local_id] = rl + delta; rating[r.visita_id] = rv - delta
        pre_l[i], pre_v[i], post_l[i], post_v[i] = rl, rv, rl + delta, rv - delta
    return pre_l, pre_v, post_l, post_v

# ---------- (A) Tabla de partidos: copia literal de R3a con la tabla de técnicos de fw6 ----------
ptp = pd.read_parquet(os.path.join(FW6, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp.columns:
    ptp = ptp[ptp['jugado'].astype(bool)]
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
uni = (ptp.groupby('match_id').agg(fecha=('fecha', 'first'), temporada=('temporada', 'first'),
                                   n_filas=('team', 'size')).reset_index())
loc_ptp = ptp[ptp['lado'] == 'home'][['match_id', 'team']].rename(columns={'team': 'local_ptp'})
M, rutas = cargar_matches()
P = uni.merge(M, on='match_id', how='left').merge(loc_ptp, on='match_id', how='left')
P['hora'] = P['hora'].fillna('00:00:00.000')
P = P.sort_values(['fecha', 'hora', 'match_id']).reset_index(drop=True)
P['torneo'] = np.where(P['fecha'].dt.month >= 7, 'Apertura', 'Clausura')
orden = P.groupby('temporada')['fecha'].min().sort_values().index.tolist()
P['sidx'] = P['temporada'].map({s: i for i, s in enumerate(orden)})
P['t'] = (P['fecha'] - CALENTAMIENTO).dt.days / 365.25
P['y'] = np.where(P.gl > P.gv, 2, np.where(P.gl == P.gv, 1, 0))
for c in ['local_id', 'visita_id']:
    if P[c].isna().all():
        P[c] = P[c.replace('_id', '')]
P['estadio'] = P['estadio'].fillna('(sin estadio)')
print(f'--- (A) Fotos leídas: {len(rutas)} · partidos {len(P)} · de {VIVO}: {int((P.temporada == VIVO).sum())} · '
      f'sin marcador: {int(P[["gl", "gv"]].isna().any(axis=1).sum())} · filas de técnicos ≠ 2: {int((P.n_filas != 2).sum())} · '
      f'local distinto entre partidos y técnicos: {int((P["local"] != P["local_ptp"]).sum())}')
assert not P[['gl', 'gv']].isna().any().any(), 'Hay partidos sin marcador: avisar a Claude'

# ---------- (B) Reproducción contra fw1 ----------
F1 = pd.read_parquet(os.path.join(FW1, 'elo_partido_e2.parquet'))
viejo = P['match_id'].isin(set(F1['match_id']))
orden_ok = P.loc[viejo, 'match_id'].tolist() == F1['match_id'].tolist()
lado = pd.concat([F1[['sidx', 'local_id', 'elo_local_pre']].set_axis(['sidx', 'id', 'pre'], axis=1),
                  F1[['sidx', 'visita_id', 'elo_visita_pre']].set_axis(['sidx', 'id', 'pre'], axis=1)])
lado = lado.reset_index().sort_values(['index'])
prim = lado.groupby('id').first()
INI = prim.loc[prim['sidx'] == 0, 'pre'].to_dict()                      # arranque por tabla (E-14), como en E11b
pl, pv, ql, qv = correr_elo(P, **E2, ini=INI)
P['elo_local_pre'], P['elo_visita_pre'], P['elo_local_post'], P['elo_visita_post'] = pl, pv, ql, qv
P['H'] = float(E2['H'])
P['d'] = P['elo_local_pre'] + P['H'] - P['elo_visita_pre']
cmp = P[viejo].merge(F1, on='match_id', suffixes=('', '_f1'))
dif = max((cmp[c] - cmp[c + '_f1']).abs().max() for c in ['elo_local_pre', 'elo_visita_pre', 'elo_local_post', 'elo_visita_post'])
print(f'--- (B) Elo · partidos que ya estaban: {int(viejo.sum())} de {len(F1)} · mismo orden de partidos: {orden_ok} · '
      f'diferencia máxima con fw1: {dif:.2e} (debe ser < 1e-6) · arranque por tabla: {len(INI)} equipos')

# Elo de xG (R3e, copia literal de la lectura de tiros y del cálculo)
T1 = pd.read_parquet(os.path.join(FW1, 'tiros_xg.parquet'))
DIR_EV = os.path.join(DP, 'liga', 'eventos')
COLS_T = ['type.name', 'team.name', 'period', 'possession', 'shot.statsbomb_xg', 'shot.type.name']
piezas = []
for mid in sorted(set(P['match_id']) - set(T1['match_id'])):
    ruta = os.path.join(DIR_EV, f'{mid}.parquet')
    disp = set(pq.read_schema(ruta).names)
    d = pd.read_parquet(ruta, columns=[c for c in COLS_T if c in disp])
    for c in COLS_T:
        if c not in d.columns: d[c] = np.nan
    d = d[(d['type.name'] == 'Shot') & d['period'].isin([1, 2, 3, 4])]
    piezas.append(pd.DataFrame({'match_id': mid, 'team': d['team.name'].values, 'possession': d['possession'].values,
                                'xg': d['shot.statsbomb_xg'].astype(float).values,
                                'penal': (d['shot.type.name'] == 'Penalty').values}))
T = pd.concat([T1] + piezas, ignore_index=True)
Tp = (T.dropna(subset=['xg']).assign(q=lambda d: np.log1p(-d.xg.clip(0, 0.999)))
        .groupby(['match_id', 'team', 'possession'])['q'].sum().reset_index())
Tp['p_gol'] = 1 - np.exp(Tp['q'])
def _dist_goles(ps):
    d = np.array([1.0])
    for p in ps: d = np.convolve(d, [1 - p, p])
    return d
grupos = {k: g['p_gol'].values for k, g in Tp.groupby(['match_id', 'team'])}
xs = []
for r in P.itertuples(index=False):
    dl = _dist_goles(grupos.get((r.match_id, r.local), []))
    dv = _dist_goles(grupos.get((r.match_id, r.visita), []))
    M_ = np.outer(dl, dv)
    p_l = np.tril(M_, -1).sum(); p_e = np.trace(M_)
    xs.append((p_l, p_e, 1 - p_l - p_e,
               sum(grupos.get((r.match_id, r.local), [])), sum(grupos.get((r.match_id, r.visita), []))))
X3 = pd.DataFrame(xs, columns=['xp_local_gana', 'xp_empate', 'xp_visita_gana', 'xg_local', 'xg_visita'])
for c in X3: P[c] = X3[c].values
P['S_xg'] = P['xp_local_gana'] + 0.5 * P['xp_empate']
P.loc[~P['match_id'].isin(T['match_id'].unique()), 'S_xg'] = np.nan
def correr_elo_s(P, S, K, H, R_ver, R_inv, ini):
    rating = dict(ini); temp_act = torneo_act = None
    pre_l = np.empty(len(P)); pre_v = np.empty(len(P)); post_l = np.empty(len(P)); post_v = np.empty(len(P))
    for i, r in enumerate(P.itertuples(index=False)):
        if temp_act is not None and r.temporada != temp_act:
            for t in rating: rating[t] = INICIO + (1 - R_ver) * (rating[t] - INICIO)
        elif torneo_act is not None and r.torneo != torneo_act:
            for t in rating: rating[t] = INICIO + (1 - R_inv) * (rating[t] - INICIO)
        temp_act, torneo_act = r.temporada, r.torneo
        rl = rating.get(r.local_id, INICIO); rv = rating.get(r.visita_id, INICIO)
        E = 1.0 / (1.0 + 10 ** (-(rl + H - rv) / 400))
        delta = 0.0 if np.isnan(S[i]) else K * (S[i] - E)
        rating[r.local_id] = rl + delta; rating[r.visita_id] = rv - delta
        pre_l[i], pre_v[i], post_l[i], post_v[i] = rl, rv, rl + delta, rv - delta
    return pre_l, pre_v, post_l, post_v
xl, xv, xlp, xvp = correr_elo_s(P, P['S_xg'].values, **E2, ini=INI)
P['eloxg_local_pre'], P['eloxg_visita_pre'], P['eloxg_local_post'], P['eloxg_visita_post'] = xl, xv, xlp, xvp
X1 = pd.read_parquet(os.path.join(FW1, 'elo_xg_partido.parquet'))
cx = P.merge(X1, on='match_id', suffixes=('', '_f1'))
dx = max((cx[c] - cx[c + '_f1']).abs().max() for c in ['eloxg_local_pre', 'eloxg_visita_pre', 'S_xg'])
print(f'    Elo de xG · partidos comparados {len(cx)} · diferencia máxima con fw1 (pre y S_xg): {dx:.2e} (debe ser < 1e-6) · '
      f'tiros leídos de partidos nuevos: {sum(len(p) for p in piezas)} en {len(piezas)} partidos')
OK = orden_ok and dif < 1e-6 and dx < 1e-6 and len(cx) == len(X1)
nuevos = P[~viejo]
am = nuevos[nuevos['local'].str.startswith('Am') | nuevos['visita'].str.startswith('Am')]
print(f'    Partidos nuevos con Elo: {len(nuevos)} ({nuevos.fecha.min().date()} a {nuevos.fecha.max().date()}) · de América: {len(am)} · '
      f'nulos en Elo o Elo de xG: {int(nuevos[["elo_local_pre", "elo_visita_pre", "eloxg_local_pre", "eloxg_visita_pre"]].isna().sum().sum())}')

# ---------- (C) Guardar en fw6 (mismas columnas que R3a y R3e) ----------
assert OK, 'El Elo recalculado NO reproduce fw1: no se escribe nada. Manda esta salida a Claude.'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
cols_out = ['match_id', 'fecha', 'temporada', 'sidx', 'torneo', 'local', 'visita', 'local_id', 'visita_id',
            'estadio', 'gl', 'gv', 'elo_local_pre', 'elo_visita_pre', 'elo_local_post', 'elo_visita_post', 'H', 'd']
P[cols_out].to_parquet(os.path.join(FW6, 'elo_partido_e2.parquet'), index=False)
largo = pd.concat([
    P.assign(team=P.local, lado='home', elo_pre=P.elo_local_pre, elo_post=P.elo_local_post, elo_rival_pre=P.elo_visita_pre),
    P.assign(team=P.visita, lado='away', elo_pre=P.elo_visita_pre, elo_post=P.elo_visita_post, elo_rival_pre=P.elo_local_pre)
])[['match_id', 'fecha', 'temporada', 'sidx', 'team', 'lado', 'elo_pre', 'elo_post', 'elo_rival_pre']]
largo.to_parquet(os.path.join(FW6, 'elo_equipo_partido_e2.parquet'), index=False)
cols_x = ['match_id', 'fecha', 'temporada', 'sidx', 'local', 'visita', 'xg_local', 'xg_visita', 'xp_local_gana',
          'xp_empate', 'xp_visita_gana', 'S_xg', 'eloxg_local_pre', 'eloxg_visita_pre', 'eloxg_local_post', 'eloxg_visita_post']
P[cols_x].to_parquet(os.path.join(FW6, 'elo_xg_partido.parquet'), index=False)
T.to_parquet(os.path.join(FW6, 'tiros_xg.parquet'), index=False)
print(f'\nVALIDACIÓN OK · guardado en fw6: elo_partido_e2 ({len(P)}), elo_equipo_partido_e2 ({len(largo)}), elo_xg_partido ({len(P)}), '
      f'tiros_xg ({len(T)}) · {time.time() - t0:.0f} s. fw1 no se tocó.')

# Validación Almada (S.13.5)

In [ ]:
# [X6-0 v1.1] Carpeta espejo para reajustar todo el pipeline con 2026/27 agregada (preregistro de Almada, §4 paso 4; patrón X5 / D-11)
# =====================================================================
# Sesión 13 · 2026-10-20 · Preregistro, Aclaración 1 (2026-10-20 11:35).
# v1.1 (14:30): (1) YA NO BORRA la carpeta espejo: se puede volver a correr en cualquier momento (p. ej. tras un reinicio de Colab) y
#   solo rehace lo que falta. (2) Recupera de Drive los resultados respaldados con X6-g (datos_parquet/fw6_x6/), así un reinicio no
#   cuesta nada. (3) Incluye los reintentos ante errores de lectura de Drive (antes en X6-r). (4) Dice en qué paso vas.
# Idea (igual que X5): NO se modifica ninguna celda validada. La carpeta /content/isac_fw6/ tiene ENLACES (solo lectura) a Drive:
#   · tabla de técnicos, R1 de 2026/27, Elo y Elo de xG → los de datos_parquet/fw6/ (sesión 13: E-3, E-4, E-5)
#   · parametros_fw1.json → el de fw5 con 2026/27 agregada a las temporadas de ajuste
#   · datos_parquet/fw5 local = MISMA carpeta que datos_parquet/fw1 local (enlace)
#   · ninguna caché de resultados de fw1/fw5 se enlaza; solo cachés de columnas crudas por temporada (r6*_extra_*, r8_extra_*) hasta 2025/26
#   · NINGÚN archivo con "2026-2027" en el nombre se toma de fw1 o fw5 (allí solo hay 88 partidos): solo de fw6.
# Fija BASE, RAIZ_PRUEBA y RAIZ_X6 = carpeta espejo. X6-9 los borra al final.
# =====================================================================
import os, json, glob, shutil, time
import pandas as pd
import pyarrow.parquet as pq

DRIVE = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
LOCAL = '/content/isac_fw6/'
VIVO, T_ = '2026/2027', '2026-2027'
D_DP = os.path.join(DRIVE, 'datos_parquet')
D_FW1, D_FW5, D_FW6 = (os.path.join(D_DP, f) for f in ('fw1', 'fw5', 'fw6'))
CK = os.path.join(D_DP, 'fw6_x6')                                   # respaldo de X6-g
L_DP = os.path.join(LOCAL, 'datos_parquet'); L_FW = os.path.join(L_DP, 'fw1')
L_SAL, L_SAL5 = os.path.join(LOCAL, 'salidas'), os.path.join(LOCAL, 'salidas', 'fw5')
nueva = not os.path.isdir(L_FW)
os.makedirs(L_FW, exist_ok=True); os.makedirs(L_SAL5, exist_ok=True)
if not os.path.lexists(os.path.join(L_DP, 'fw5')): os.symlink(L_FW, os.path.join(L_DP, 'fw5'))   # fw5 local = fw1 local

# 0. Reintentos de lectura ante fallos momentáneos de Drive (OSError, Errno 5). No cambian ningún cálculo.
def _con_reintentos(f, nombre):
    if getattr(f, '_reintentos', False): return f
    def g(*a, **k):
        for i in range(6):
            try:
                return f(*a, **k)
            except OSError as e:
                if i == 5: raise
                print(f'  [reintento {i + 1}/5 de {nombre}: {type(e).__name__} {str(e)[:60]}]'); time.sleep(5 * (i + 1))
    g._reintentos = True
    return g
pd.read_parquet = _con_reintentos(pd.read_parquet, 'read_parquet')
pq.read_schema = _con_reintentos(pq.read_schema, 'read_schema')
pq.read_table = _con_reintentos(pq.read_table, 'read_table')

def enlazar(src, dst):
    if os.path.exists(src) and not os.path.lexists(dst):
        os.symlink(src, dst); return True
    return os.path.islink(dst) and os.path.realpath(dst) == os.path.realpath(src)

# 1. Raíz y datos_parquet: todo menos lo que se reemplaza
TABLAS_FW6 = ['partido_tecnico_periodo.parquet', 'etapas_tecnico.parquet', 'equipo_partido_tecnico.parquet']
for e in os.listdir(DRIVE):
    if e not in {'datos_parquet', 'salidas'}: enlazar(os.path.join(DRIVE, e), os.path.join(LOCAL, e))
for e in os.listdir(D_DP):
    if e not in {'fw1', 'fw5', 'fw6', 'fw6_x6'} | set(TABLAS_FW6): enlazar(os.path.join(D_DP, e), os.path.join(L_DP, e))
for f in TABLAS_FW6: assert enlazar(os.path.join(D_FW6, f), os.path.join(L_DP, f)), f'Falta fw6/{f}: corre E-3'

# 2. fw1 local: entradas
enl = {}
for f in ['elo_equipo_partido_e2.parquet', 'elo_partido_e2.parquet', 'elo_xg_partido.parquet', 'tiros_xg.parquet',
          f'eventos_{T_}.parquet', f'posesiones_{T_}.parquet']:
    enl[f] = 'fw6' if enlazar(os.path.join(D_FW6, f), os.path.join(L_FW, f)) else 'FALTA'
PATRONES = ['eventos_*.parquet', 'posesiones_*.parquet', 'componentes_*.parquet', 'presion_*.parquet',
            'r6_extra_*.parquet', 'r6b_extra_*.parquet', 'r6c_extra_*.parquet', 'r8_extra_*.parquet']
for origen, carpeta in [('fw1', D_FW1), ('fw5', D_FW5)]:
    for p in PATRONES:
        for r in sorted(glob.glob(os.path.join(carpeta, p))):
            f = os.path.basename(r)
            if T_ in f or os.path.islink(r) or f in enl: continue
            if enlazar(r, os.path.join(L_FW, f)): enl[f] = origen
# 3. Parámetros: los de fw5 con 2026/27 agregada al ajuste
with open(os.path.join(D_FW5, 'parametros_fw1.json')) as f:
    PRM_ = json.load(f)
PRM_['temporadas']['entrenamiento'] = sorted(set(PRM_['temporadas']['entrenamiento']) | {VIVO})
PRM_['_fw6'] = 'copia de fw5 con 2026/27 agregada (sesión 13, X6; preregistro de Almada §4 paso 4)'
with open(os.path.join(L_FW, 'parametros_fw1.json'), 'w') as f:
    json.dump(PRM_, f, indent=1)
TEMPS = PRM_['temporadas']['entrenamiento']
# 4. R2a y R2b (componentes_, presion_) solo se usan en chequeos de coherencia: vacíos donde no existan
for t in TEMPS:
    for tipo in ['componentes', 'presion']:
        p = os.path.join(L_FW, f"{tipo}_{t.replace('/', '-')}.parquet")
        if not os.path.lexists(p):
            pd.DataFrame({'match_id': pd.Series(dtype='int64'), 'equipo': pd.Series(dtype='object')}).to_parquet(p, index=False)

# 5. Recuperar lo respaldado con X6-g (solo archivos que no estén ya en la carpeta local)
rec = 0
for sub, dst in [('fw1', L_FW), ('salidas', L_SAL), ('salidas_fw5', L_SAL5)]:
    src = os.path.join(CK, sub)
    if not os.path.isdir(src): continue
    for f in os.listdir(src):
        d = os.path.join(dst, f)
        if not os.path.lexists(d): shutil.copy2(os.path.join(src, f), d); rec += 1

# 6. Chequeos y en qué paso vas
ptp_ = pd.read_parquet(os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))
ev26 = pd.read_parquet(os.path.join(L_FW, f'eventos_{T_}.parquet'), columns=['match_id'])['match_id'].nunique()
elo_ = pd.read_parquet(os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'), columns=['match_id'])['match_id']
m26 = set(ptp_.loc[ptp_.temporada == VIVO, 'match_id'])
falta = [f for t in TEMPS for f in [f"eventos_{t.replace('/', '-')}.parquet", f"posesiones_{t.replace('/', '-')}.parquet"]
         if not os.path.exists(os.path.join(L_FW, f))] + [k for k, v in enl.items() if v == 'FALTA']
# las cachés de resultados solo pueden venir del respaldo de X6 (nunca enlaces a fw1/fw5)
mal = [f for f in os.listdir(L_FW) if os.path.islink(os.path.join(L_FW, f)) and f.split('_')[0] in
       {'r4', 'r5', 'r6', 'r6b', 'r6c', 'r7', 'r7u', 'r7b', 'r7c', 'r8', 'r8p', 'r8z'} and 'extra' not in f]
print(f'--- (X6-0 v1.1) Carpeta espejo {"CREADA" if nueva else "YA EXISTÍA (no se borró nada)"} en {LOCAL} · archivos recuperados del respaldo: {rec}')
print(f'Temporadas de ajuste: {TEMPS}')
print(f'Tabla de técnicos (fw6): partidos de {VIVO} {len(m26)} · R1 {VIVO}: {ev26} · con Elo: {elo_.isin(m26).sum() // 2} (los tres deben ser 106)')
print('Faltantes:', falta or 'ninguno', '· resultados enlazados a fw1/fw5 (debe ser ninguno):', mal or 'ninguno')
assert not falta and not mal and len(m26) == ev26 == 106, 'Algo no cuadra: NO seguir; manda esta salida a Claude'
hay = lambda *fs: all(os.path.exists(os.path.join(L_FW, f)) or os.path.exists(os.path.join(L_SAL, f)) or os.path.exists(os.path.join(L_SAL5, f)) for f in fs)
PASOS = [('2 · R4a', ['r4_tramos_v21.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet']),
         ('2 · R4b', ['r4_residuos.parquet', 'r4_perfiles_etapa.parquet', 'r4_coeficientes.csv']),
         ('3 · R5', ['r5_perfiles_contraidos.parquet', 'r5_confiabilidad.csv']),
         ('4 · R6a', ['r6_tramos.parquet']), ('4 · R6b', ['r6b_tramos.parquet']), ('4 · R6c', ['r6c_tramos.parquet'])] + \
        [(f'5 · R6r {p}', [f'{p}_perfiles_contraidos.parquet', f'{p}_confiabilidad.csv']) for p in ['r6', 'r6b', 'r6c']] + \
        [('6 · R7a', ['r7_equipo_partido.parquet', 'r7_jugador_partido.parquet', 'r7_subs.parquet']),
         ('7 · R7u-0', ['r7u_tramos.parquet']), ('7 · R6r r7u', ['r7u_perfiles_contraidos.parquet', 'r7u_confiabilidad.csv']),
         ('8 · R7b-0', ['r7b_tramos.parquet']), ('8 · R6r r7b', ['r7b_perfiles_contraidos.parquet', 'r7b_confiabilidad.csv']),
         ('9 · R7c-0', ['r7c_tramos.parquet']), ('9 · R6r r7c', ['r7c_perfiles_contraidos.parquet', 'r7c_confiabilidad.csv']),
         ('10 · X6-R8a-1', ['r8_bp.parquet', 'r8_penales.parquet']), ('10 · X6-R8p-1', ['r8p_corners.parquet']),
         ('10 · X6-R8z-1', ['r8z_marcaje.parquet']), ('10 · X6-R8u-0', ['r8_tramos.parquet']),
         ('10 · R6r r8', ['r8_perfiles_contraidos.parquet', 'r8_confiabilidad.csv'])]
hechos = [n for n, fs in PASOS if hay(*fs)]
sig = next((n for n, fs in PASOS if not hay(*fs)), '11 · X6-9')
print('Pasos con sus resultados presentes:', hechos or 'ninguno')
print('→ SIGUIENTE PASO:', sig, '(los pasos 2, 4, 6 y las celdas de R8 cargan su caché si ya existe y terminan rápido)')
BASE = LOCAL; RAIZ_PRUEBA = LOCAL; RAIZ_X6 = LOCAL
print('BASE = RAIZ_PRUEBA = RAIZ_X6 =', LOCAL, '· reintentos de lectura activos')

# =====================================================================
# HOJA DE RUTA X6 (misma sesión de Colab; no correr R0 ni ninguna celda *-r0 / *-r9 / X5)
#  1. celda 1 (Drive) → X6-0   (si Colab se reinicia: celda 1 → X6-0 y seguir desde el "SIGUIENTE PASO" que imprime)
#  2. R4a y R4b → X6-g          3. R5 → X6-g          4. R6a, R6b, R6c → X6-g
#  5. Para cada PREFIJO en r6, r6b, r6c: X6-p → R6r-1 → R6r-2 → X6-g
#  6. R7a → X6-g
#  7. R7u-0 → X6-p 'r7u' → R6r-1 → R6r-2 → X6-g
#  8. R7b-0 → X6-p 'r7b' → R6r-1 → R6r-2 → X6-g
#  9. R7c-0 → X6-p 'r7c' → R6r-1 → R6r-2 → X6-g          (no correr R7c-9)
# 10. R8a-0 → X6-R8a-1 · R8p-0 → X6-R8p-1 · R8z-0 → X6-R8z-1 · X6-R8u-0 → X6-p 'r8' → R6r-1 → R6r-2 → X6-g
# 11. X6-9
# X6-p:   assert globals().get('BASE') == '/content/isac_fw6/', 'BASE no apunta al espejo: corre X6-0'
#         PREFIJO = 'r6'
# =====================================================================

In [ ]:
#X6-p
assert globals().get('BASE') == '/content/isac_fw6/', 'BASE no apunta al espejo: corre X6-0'
PREFIJO = 'r8'

In [ ]:
# [X6-r v1.0] Reintentos de lectura ante errores de Drive (OSError / Errno 5 "Input/output error")
# =====================================================================
# Sesión 13 · 2026-10-20 · Correr UNA vez en la sesión de X6 (después de X6-0, antes de volver a correr R7a).
# Qué hace: si una lectura de parquet desde Drive falla por un error de entrada/salida (fallo momentáneo de la conexión de Drive,
# no de los datos), la repite hasta 6 veces con espera creciente. NO cambia ningún cálculo: el archivo leído es el mismo.
# Aplica a pd.read_parquet y a pq.read_schema en TODAS las celdas siguientes de esta sesión.
# =====================================================================
import time
import pandas as pd
import pyarrow.parquet as pq

def _con_reintentos(f, nombre):
    if getattr(f, '_reintentos', False): return f
    def g(*a, **k):
        for i in range(6):
            try:
                return f(*a, **k)
            except OSError as e:
                if i == 5: raise
                print(f'  [reintento {i + 1}/5 de {nombre}: {type(e).__name__} {str(e)[:60]}]'); time.sleep(5 * (i + 1))
    g._reintentos = True
    return g
pd.read_parquet = _con_reintentos(pd.read_parquet, 'read_parquet')
pq.read_schema = _con_reintentos(pq.read_schema, 'read_schema')
pq.read_table = _con_reintentos(pq.read_table, 'read_table')
assert globals().get('BASE') == '/content/isac_fw6/', 'BASE no apunta al espejo: corre X6-0'
print('Reintentos activos para read_parquet, read_schema y read_table · BASE =', BASE, '→ vuelve a correr R7a.')

In [ ]:
# [X6-g v1.0] Respaldo en Drive de lo que X6 lleva hecho (para que un reinicio de Colab no cueste nada)
# =====================================================================
# Sesión 13 · 2026-10-20 · Correr después de cada bloque de la hoja de ruta X6 (ver X6-0). Tarda segundos a ~2 min.
# Copia a Drive, en datos_parquet/fw6_x6/ (carpeta de trabajo, NO es fw6), solo los archivos que X6 generó (no los enlaces) y solo
# los nuevos o cambiados. X6-0 v1.1 los recupera si la carpeta local se pierde. fw1, fw5 y fw6 no se tocan.
# =====================================================================
import os, shutil
DRIVE, LOCAL = '/content/drive/MyDrive/isac2026/', '/content/isac_fw6/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
assert os.path.isdir(os.path.join(LOCAL, 'datos_parquet', 'fw1')), 'No hay carpeta espejo: corre X6-0'
CK = os.path.join(DRIVE, 'datos_parquet', 'fw6_x6')
nuevos = []
for sub, src in [('fw1', os.path.join(LOCAL, 'datos_parquet', 'fw1')), ('salidas', os.path.join(LOCAL, 'salidas')),
                 ('salidas_fw5', os.path.join(LOCAL, 'salidas', 'fw5'))]:
    dst = os.path.join(CK, sub); os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        s, d = os.path.join(src, f), os.path.join(dst, f)
        if not os.path.isfile(s) or os.path.islink(s): continue
        if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s) and os.path.getmtime(d) >= os.path.getmtime(s): continue
        shutil.copy2(s, d); nuevos.append(f)
tot = sum(len(os.listdir(os.path.join(CK, s))) for s in ['fw1', 'salidas', 'salidas_fw5'])
print(f'--- (X6-g) Respaldados ahora: {len(nuevos)} · total en datos_parquet/fw6_x6: {tot}')
print('  nuevos:', sorted(nuevos)[:30], '…' if len(nuevos) > 30 else '')

In [ ]:
# [X6-R8a-1] = R8a-1 v1.0 (claude/codigo/sesion11/R8a_base.py) con UNA línea cambiada para X6 (preregistro, Aclaración 1 punto 5):
#   original:  RAIZ = '/content/drive/MyDrive/isac2026/'
#   X6:        RAIZ = RAIZ_X6   (carpeta espejo /content/isac_fw6/; se agrega un freno que lo verifica)
# Todo lo demás es copia literal hasta el bloque (1). Se omiten las impresiones descriptivas (2)-(6) del original: no escriben nada
# y mostrarían cifras de 2026/27. Antes: X6-0 y R8a-0 (sin cambios) en esta sesión.
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = RAIZ_X6                                                       # X6: única línea cambiada
assert RAIZ == '/content/isac_fw6/', 'Freno X6: RAIZ no es la carpeta espejo; corre X6-0'   # X6: freno agregado
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'bp_de_partido' in globals(), 'Corre primero R8a-0'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1'); FW5 = os.path.join(DP, 'fw5'); LIGA = os.path.join(DP, 'liga')
REHACER = False
CORTE_C1, CORTE_C3 = 25, 70                                          # cuartos de R4 (A2)
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
TEMPS = sorted(Mm['temporada'].unique())
COLS_R1 = ['match_id', 'index', 'period', 'minute', 't', 'type.name', 'team.name', 'possession', 'possession_team.name', 'player.id',
           'loc_x', 'loc_y', 'pass_end_x', 'pass_end_y', 'pass.type.name', 'pass.length', 'ball_receipt.outcome.name', 'shot.type.name',
           'shot.outcome.name', 'shot.statsbomb_xg', 'set_piece_phase', 'fase', 'en_transicion', 'pass.technique.name', 'foul_committed.penalty']
OPC_CRUDOS = ['pass.technique.name', 'foul_committed.penalty', 'pass.length', 'ball_receipt.outcome.name', 'second']
RUTAS = {k: os.path.join(FW5, f'r8_{k}.parquet') for k in ['bp', 'penales', 'reparto']}

# HOPS: ¿viene en los eventos? (nombres reales en 20 partidos crudos)
nom = set()
for m in Mm['match_id'].iloc[::85][:20]: nom |= set(pq.read_schema(os.path.join(LIGA, 'eventos', f'{m}.parquet')).names)
print('--- (0) Columnas tipo HOPS o probabilidad de duelo en los crudos:', sorted(c for c in nom if any(s in c.lower() for s in ['hops', 'win_prob', 'rating'])) or 'ninguna')

def extra_crudos(temp, falt):
    ruta = os.path.join(FW5, f"r8_extra_{temp.replace('/', '-')}.parquet")
    if os.path.exists(ruta) and not REHACER:
        X = pd.read_parquet(ruta)
        if all(c in X for c in falt): return X
    piezas = []
    for mid in Mm.loc[Mm.temporada == temp, 'match_id']:
        r = os.path.join(LIGA, 'eventos', f'{mid}.parquet')
        if not os.path.exists(r): continue
        disp = set(pq.read_schema(r).names)
        d = pd.read_parquet(r, columns=['index'] + [c for c in falt if c in disp])
        for c in falt:
            if c not in d: d[c] = np.nan
        d['match_id'] = mid; piezas.append(d)
    X = pd.concat(piezas, ignore_index=True); X.to_parquet(ruta, index=False); return X

if all(os.path.exists(r) for r in RUTAS.values()) and not REHACER:
    BP, PE, RE = (pd.read_parquet(RUTAS[k]) for k in ['bp', 'penales', 'reparto'])
    print('Cargado de caché:', len(BP), 'balones parados')
else:
    bps, pes, res, chk = [], [], [], []
    for temp in TEMPS:
        t0 = time.time()
        ruta_e = os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"); cols_r1 = set(pq.read_schema(ruta_e).names)
        E = pd.read_parquet(ruta_e, columns=[c for c in COLS_R1 if c in cols_r1])
        E = E[E['match_id'].isin(loc.index)]
        falt = [c for c in OPC_CRUDOS if c not in E.columns and (c != 'second' or 't' not in E.columns)]
        if falt:
            E = E.merge(extra_crudos(temp, falt)[['match_id', 'index'] + falt], on=['match_id', 'index'], how='left')
        if 't' not in E.columns: E['t'] = E['minute'] * 60 + E['second']
        for c in COLS_R1:
            if c not in E.columns: E[c] = np.nan
        E = E.sort_values(['match_id', 'index']).reset_index(drop=True)
        ok_t = ((E['t'] - 60 * E['minute']).between(-0.01, 60.01)).mean()
        # Marcador: copia literal de R6a (goles de los periodos 1-4: tiro convertido o autogol a favor; offset de liguilla)
        E['local'] = E['match_id'].map(loc['local']); E['ev_local'] = E['team.name'] == E['local']
        es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) | (E['type.name'] == 'Own Goal For'))
        gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
        E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
        E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset']); E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
        E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < CORTE_C1, 1, 2), np.where(E['minute'] < CORTE_C3, 3, 4))
        gg = pd.DataFrame({'gh': gh.groupby(E['match_id']).sum(), 'ga': ga.groupby(E['match_id']).sum()})
        cuadra = ((gg.gh == loc.loc[gg.index, 'gl']) & (gg.ga == loc.loc[gg.index, 'gv'])).mean()
        E = E[E['period'].between(1, 4)]
        # Reparto de tiros, xG y goles por tipo de jugada (periodos 1-4; sin tandas)
        sh = E[E['type.name'] == 'Shot'].copy()
        sh['tipo'] = np.select([sh['shot.type.name'] == 'Penalty', sh['fase'] == 'bp', sh['fase'] == 'transicion'], ['penal', 'balon_parado', 'transicion'], 'establecido')
        r = sh.groupby(['match_id', 'team.name', 'tipo']).agg(tiros=('tipo', 'size'), xg=('shot.statsbomb_xg', 'sum'),
                                                              goles=('shot.outcome.name', lambda s: int((s == 'Goal').sum()))).reset_index()
        og = E[E['type.name'] == 'Own Goal For'].groupby(['match_id', 'team.name']).size().rename('goles').reset_index().assign(tipo='autogol', tiros=0, xg=0.0)
        res.append(pd.concat([r, og], ignore_index=True).rename(columns={'team.name': 'team'}))
        n360 = 0
        for mid, M in E.groupby('match_id', sort=False):
            J = None; rj = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
            ic = M.loc[(M['pass.type.name'] == 'Corner'), 'index'].values
            if os.path.exists(rj) and len(ic):
                nec = set(np.concatenate([ic + k for k in range(16)]))
                jj = pd.read_parquet(rj, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y'])
                jj = jj[jj.event_index.isin(nec)]; J = {k: g for k, g in jj.groupby('event_index')}; n360 += 1
            for f in bp_de_partido(M, J): f['match_id'] = mid; bps.append(f)
            for f in penales_de_partido(M): f['match_id'] = mid; pes.append(f)
        chk.append(dict(temporada=temp, partidos=E.match_id.nunique(), con_360=n360, t_coherente_pct=round(100 * ok_t, 2),
                        marcador_cuadra_pct=round(100 * cuadra, 2), segundos=round(time.time() - t0)))
        print(f'  {temp}: {chk[-1]}')
    BP, PE, RE = pd.DataFrame(bps), pd.DataFrame(pes), pd.concat(res, ignore_index=True)
    for d_, k in [(BP, 'bp'), (PE, 'penales'), (RE, 'reparto')]: d_.to_parquet(RUTAS[k], index=False)
    print('Chequeos por temporada (t_coherente y marcador_cuadra deben ser ~100):'); print(pd.DataFrame(chk).to_string(index=False))

BP['temporada'] = BP.match_id.map(loc['temporada']); RE['temporada'] = RE.match_id.map(loc['temporada'])
pc = lambda s: round(100 * float(np.mean(s)), 1) if len(s) else np.nan
co = BP[BP.kind == 'corner']
print(f'\n--- (1) Balones parados: {BP.kind.value_counts().to_dict()} · córners por partido {len(co) / BP.match_id.nunique():.1f}')
print('  Ruta del córner (%):', co.ruta.value_counts(normalize=True).mul(100).round(1).to_dict())
print(f'\nX6-R8a-1 terminó en {time.time() - t_ini:.0f} s. Guardado (espejo): r8_bp, r8_penales, r8_reparto (+ r8_extra_2026-2027).')

In [ ]:
# [X6-R8p-1] = R8p-1 v1.1 (claude/codigo/sesion11/R8p_playbooks.py) para X6, con los cambios de la Aclaración 1 (puntos 3 y 5):
#   (a) RAIZ = RAIZ_X6 (carpeta espejo) en lugar de '/content/drive/MyDrive/isac2026/' + freno.
#   (b) NO se vuelve a agrupar: cada córner se asigna al playbook más cercano de los CENTROIDES de fw5 (salidas/fw5/r8p_km_*.csv,
#       lectura de Drive). Es lo mismo que KMeans.predict con el modelo de la sesión 11.
#   (c) Se omiten la elección de k, los placebos y las tablas descriptivas (no escriben nada que use R8u).
# Copia literal del original: bloque 1 (primer contacto) y bloque 2 (huellas 360). Mismo archivo de salida: r8p_corners.
# Validación obligatoria: en los córners de 2021/22-2025/26, huellas idénticas a fw5/r8p_huellas y etiquetas idénticas a fw5/r8p_corners.
# Antes: X6-0, X6-R8a-1 y R8p-0 (sin cambios) en esta sesión.
import os, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
RAIZ = RAIZ_X6                                                       # X6 (a)
assert RAIZ == '/content/isac_fw6/', 'Freno X6: RAIZ no es la carpeta espejo; corre X6-0'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'huella' in globals() and 'ZONA_CELDA' in globals(), 'Corre primero R8p-0 (define huella)'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
D_FW5, D_SAL5 = '/content/drive/MyDrive/isac2026/datos_parquet/fw5', '/content/drive/MyDrive/isac2026/salidas/fw5'   # X6 (b): solo lectura
REHACER = False
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
C = BP[(BP.kind == 'corner') & (BP.ruta == 'directo') & BP['s_frame'].astype('boolean').fillna(False)].copy()
C['temporada'] = C.match_id.map(loc['temporada'])
C['rival'] = np.where(C.team == C.match_id.map(loc['local']), C.match_id.map(loc['visita']), C.match_id.map(loc['local']))
ruta_h = os.path.join(FW5, 'r8p_huellas.parquet')
if os.path.exists(ruta_h) and not REHACER:
    H = pd.read_parquet(ruta_h); print('Huellas de caché:', len(H))
else:
    # 1. Primer contacto de cada córner directo (misma definición que R8a: siguiente evento sin presiones ni recepciones fallidas, ≤ 5 s)
    piezas = []
    for temp, g in C.groupby('temporada'):
        E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                            columns=['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'ball_receipt.outcome.name'])
        E = E[E.match_id.isin(set(g.match_id))]
        F = E[(E['type.name'] != 'Pressure') & ~((E['type.name'] == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())]
        F = F.rename(columns={'index': 'c_index', 'period': 'c_period', 't': 'c_t', 'type.name': 'c_tipo', 'team.name': 'c_team'})
        F['index'] = F['c_index']
        x = pd.merge_asof(g.sort_values('index'), F.sort_values('index')[['match_id', 'index', 'c_index', 'c_period', 'c_t', 'c_tipo', 'c_team']],
                          on='index', by='match_id', direction='forward', allow_exact_matches=False)
        val = (x.c_period == x.period) & (x.c_t <= x.t + 5)
        x.loc[~val, ['c_index', 'c_tipo', 'c_team']] = np.nan
        piezas.append(x)
    C = pd.concat(piezas, ignore_index=True)
    igual = (C.c_tipo.fillna('—') == C.contacto_tipo.fillna('—')).mean()
    print(f'--- (0) Coherencia con R8a: tipo de contacto idéntico en {100 * igual:.2f}% de {len(C):,} córners (debe ser 100)')
    # 2. Huellas desde las fotos 360
    filas = []
    for mid, g in C.groupby('match_id'):
        r = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
        if not os.path.exists(r): continue
        nec = set(g['index']) | set(g['c_index'].dropna().astype(int))
        jj = pd.read_parquet(r, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y']); jj = jj[jj.event_index.isin(nec)]
        JJ = {k: v for k, v in jj.groupby('event_index')}
        for _, c in g.iterrows():
            if c['index'] not in JJ: continue
            A, D, peg = huella(JJ[c['index']], bool(c.lado_bajo))
            f = dict(match_id=mid, index=c['index'], pegados_portero=peg, **{f'sa{i}': A[i] for i in range(20)}, **{f'sd{i}': D[i] for i in range(20)})
            ci = c['c_index']
            if pd.notna(ci) and int(ci) in JJ:
                A2, _, _ = huella(JJ[int(ci)], bool(c.lado_bajo), girar=c['c_team'] != c['team'], sin_actor=False)
                f.update({f'ca{i}': A2[i] for i in range(20)})
            filas.append(f)
    H = pd.DataFrame(filas).merge(C[['match_id', 'index', 'team', 'rival', 'temporada', 'zona', 'tecnica', 'xg1', 'xg2', 'goles', 'caida_gana',
                                     'contacto_clase', 's_marc2', 's_marc_den']], on=['match_id', 'index'])
    H.to_parquet(ruta_h, index=False)
SA, SD, CA = [f'sa{i}' for i in range(20)], [f'sd{i}' for i in range(20)], [f'ca{i}' for i in range(20)]
print(f'--- (1) Córners directos con foto del saque: {len(H):,} · con foto del contacto {100 * H[CA[0]].notna().mean():.1f}%'
      f' · con portero visible {100 * H.pegados_portero.notna().mean():.1f}% ({time.time() - t_ini:.0f} s) · por temporada {H.temporada.value_counts().sort_index().to_dict()}')

# X6 (b) · Asignación a los centroides congelados de fw5 (playbook más cercano, distancia euclidiana = KMeans.predict)
def asigna(X, cen):
    return ((X[:, None, :] - cen[None, :, :]) ** 2).sum(-1).argmin(1)
cen_o = pd.read_csv(os.path.join(D_SAL5, 'r8p_km_ofensivo.csv'))[SA + CA].to_numpy(float)
cen_d = pd.read_csv(os.path.join(D_SAL5, 'r8p_km_defensivo.csv'))[SD].to_numpy(float)
Ho = H[H[CA[0]].notna()].reset_index(drop=True)
Ho['pb_of'] = asigna(Ho[SA + CA].to_numpy(float), cen_o)
H['pb_def'] = asigna(H[SD].to_numpy(float), cen_d)
H = H.merge(Ho[['match_id', 'index', 'pb_of']], on=['match_id', 'index'], how='left')
print(f'--- (2) Playbooks con centroides de fw5: ofensivos k = {len(cen_o)} · defensivos k = {len(cen_d)}')

# Validación contra fw5 (Drive, solo lectura)
k_ = ['match_id', 'index']
h5 = pd.read_parquet(os.path.join(D_FW5, 'r8p_huellas.parquet'))
c5 = pd.read_parquet(os.path.join(D_FW5, 'r8p_corners.parquet'), columns=k_ + ['pb_of', 'pb_def'])
num = ['pegados_portero'] + SA + SD + CA
j = h5[k_ + num].merge(H[k_ + num], on=k_, how='left', suffixes=('_5', '_6'), indicator=True)
dif_h = max(float(np.nanmax(np.abs(j[c + '_5'] - j[c + '_6']))) if j[c + '_5'].notna().any() else 0.0 for c in num)
nul_h = int(sum(((j[c + '_5'].isna()) != (j[c + '_6'].isna())).sum() for c in num))
jc = c5.merge(H[k_ + ['pb_of', 'pb_def']], on=k_, how='left', suffixes=('_5', '_6'))
ig_of = ((jc.pb_of_5 == jc.pb_of_6) | (jc.pb_of_5.isna() & jc.pb_of_6.isna())).mean()
ig_def = (jc.pb_def_5 == jc.pb_def_6).mean()
print(f'--- (3) Validación (córners de fw5: {len(h5):,}) · en X6: {int((j._merge == "both").sum()):,} · huellas: diferencia máxima {dif_h:.2e} · '
      f'nulos distintos {nul_h} · etiqueta ofensiva igual {100 * ig_of:.2f}% · defensiva igual {100 * ig_def:.2f}% (todo debe ser 0 / 100)')
nuevos = H[~H.set_index(k_).index.isin(h5.set_index(k_).index)]
print(f'  Córners nuevos (2026/27): {len(nuevos)} · con contacto (tienen playbook ofensivo) {int(nuevos.pb_of.notna().sum())}')
assert (j._merge == 'both').all() and dif_h < 1e-9 and nul_h == 0 and ig_of == 1 and ig_def == 1, \
    'X6-R8p no reproduce fw5: NO seguir; manda esta salida a Claude'
H.to_parquet(os.path.join(FW5, 'r8p_corners.parquet'), index=False)
print(f'\nX6-R8p-1 terminó en {time.time() - t_ini:.0f} s. VALIDACIÓN OK · guardado (espejo): r8p_huellas, r8p_corners.')


In [ ]:
# [X6-R8z-1] = R8z-1 v1.1 (claude/codigo/sesion11/R8z_marcaje.py) con UNA línea cambiada para X6 (preregistro, Aclaración 1 punto 5):
#   original:  RAIZ = '/content/drive/MyDrive/isac2026/'
#   X6:        RAIZ = RAIZ_X6   (carpeta espejo; se agrega un freno que lo verifica)
# Todo lo demás es copia literal, salvo que se omiten las impresiones descriptivas finales del bloque (2) y el bloque (3)
# (no escriben nada que use R8u). Antes: X6-0, X6-R8a-1, X6-R8p-1 y R8z-0 (sin cambios) en esta sesión.
import os, time
RAIZ = RAIZ_X6                                                       # X6: única línea cambiada
assert RAIZ == '/content/isac_fw6/', 'Freno X6: RAIZ no es la carpeta espejo; corre X6-0'   # X6: freno agregado
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert all(f in globals() for f in ['jugadores', 'marcaje', 'reparto', 'pendiente']), 'Corre primero R8z-0 (define las funciones)'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
REHACER = False
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
C = BP[(BP.kind == 'corner') & (BP.ruta == 'directo') & BP['s_frame'].astype('boolean').fillna(False)].copy()
C['temporada'] = C.match_id.map(loc['temporada'])
C['rival'] = np.where(C.team == C.match_id.map(loc['local']), C.match_id.map(loc['visita']), C.match_id.map(loc['local']))
ruta_z = os.path.join(FW5, 'r8z_marcaje.parquet')
if os.path.exists(ruta_z) and not REHACER:
    Z = pd.read_parquet(ruta_z); print('Marcaje de caché:', len(Z))
else:
    # Primer contacto: copia literal de R8p-1 (misma definición que R8a)
    piezas = []
    for temp, g in C.groupby('temporada'):
        E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                            columns=['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'ball_receipt.outcome.name'])
        E = E[E.match_id.isin(set(g.match_id))]
        F = E[(E['type.name'] != 'Pressure') & ~((E['type.name'] == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())]
        F = F.rename(columns={'index': 'c_index', 'period': 'c_period', 't': 'c_t', 'type.name': 'c_tipo', 'team.name': 'c_team'})
        F['index'] = F['c_index']
        x = pd.merge_asof(g.sort_values('index'), F.sort_values('index')[['match_id', 'index', 'c_index', 'c_period', 'c_t', 'c_tipo', 'c_team']],
                          on='index', by='match_id', direction='forward', allow_exact_matches=False)
        val = (x.c_period == x.period) & (x.c_t <= x.t + 5)
        x.loc[~val, ['c_index', 'c_tipo', 'c_team']] = np.nan
        piezas.append(x)
    C = pd.concat(piezas, ignore_index=True)
    print(f'--- (0) Coherencia con R8a: tipo de contacto idéntico en {100 * (C.c_tipo.fillna("—") == C.contacto_tipo.fillna("—")).mean():.2f}% (debe ser 100)')
    filas = []
    for mid, g in C.groupby('match_id'):
        r = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
        if not os.path.exists(r): continue
        nec = set(g['index']) | set(g['c_index'].dropna().astype(int))
        jj = pd.read_parquet(r, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y']); jj = jj[jj.event_index.isin(nec)]
        JJ = {k: v for k, v in jj.groupby('event_index')}
        for _, c in g.iterrows():
            if c['index'] not in JJ: continue
            ax, ay, dx, dy = jugadores(JJ[c['index']], bool(c.lado_bajo))
            f = dict(match_id=mid, index=c['index'], team=c.team, rival=c.rival, temporada=c.temporada, n_atac=len(ax), n_def=len(dx),
                     **marcaje(ax, ay, dx, dy), **{f'pa_{z}': v for z, v in zip(ZONAS, reparto(ax, ay))},
                     **{f'pd_{z}': v for z, v in zip(ZONAS, reparto(dx, dy))})
            ci = c['c_index']
            if pd.notna(ci) and int(ci) in JJ:
                bx, by, ex_, ey_ = jugadores(JJ[int(ci)], bool(c.lado_bajo), girar=c['c_team'] != c['team'], sin_actor=False)
                f['c_pct_marca_b'] = marcaje(bx, by, ex_, ey_)['pct_marca_b']
            filas.append(f)
    Z = pd.DataFrame(filas); Z.to_parquet(ruta_z, index=False)

print(f'--- (1) Córners: {len(Z):,} · atacantes en el área (mediana) {Z.n_atac.median():.0f} · defensores {Z.n_def.median():.0f} ({time.time() - t_ini:.0f} s)')
for k, (u1, u2) in UMBRALES.items():
    print(f'  Umbrales {u1}/{u2} yd: % de atacantes con marca · cuantiles 10/25/50/75/90 {Z[f"pct_marca_{k}"].quantile([.1, .25, .5, .75, .9]).round(0).tolist()}'
          f' · defensores zonales {Z[f"zonales_{k}"].quantile([.1, .25, .5, .75, .9]).tolist()}')
print('  Histograma del % de atacantes con marca (umbral principal, % de córners por tramo de 20 puntos):',
      pd.cut(Z.pct_marca_b, [-1, 20, 40, 60, 80, 101]).value_counts(normalize=True).sort_index().mul(100).round(1).to_dict())
zc = Z.dropna(subset=['c_pct_marca_b'])
print(f'  Contacto: % de atacantes con marca saque → contacto {zc.pct_marca_b.mean():.1f} → {zc.c_pct_marca_b.mean():.1f}'
      f' · correlación entre córners {zc[["pct_marca_b", "c_pct_marca_b"]].corr().iloc[0, 1]:.2f}')

# 2. Por equipo defensor y temporada: marca media, zonales y pendiente de respuesta al rival (con mitades para su confiabilidad)
rng = np.random.default_rng(SEMILLA)
PA_c, PD_c = [f'pa_{z}' for z in ZONAS], [f'pd_{z}' for z in ZONAS]
filas = []
for (eq, temp), g in Z.groupby(['rival', 'temporada']):
    if len(g) < 40: continue
    m = rng.random(len(g)) < 0.5
    filas.append(dict(equipo=eq, temporada=temp, corners=len(g), pct_marca=g.pct_marca_b.mean(), zonales=g.zonales_b.mean(),
                      pendiente=pendiente(g[PA_c].to_numpy(), g[PD_c].to_numpy()),
                      pend_m1=pendiente(g[m][PA_c].to_numpy(), g[m][PD_c].to_numpy()), pend_m2=pendiente(g[~m][PA_c].to_numpy(), g[~m][PD_c].to_numpy()),
                      marca_m1=g[m].pct_marca_b.mean(), marca_m2=g[~m].pct_marca_b.mean()))
T = pd.DataFrame(filas)
sb = lambda r: 2 * r / (1 + r)
print(f'\n--- (2) Equipos defensores (equipo-temporada con ≥ 40 córners): {len(T)}')
print('  % de atacantes con marca: cuantiles 10/50/90', T.pct_marca.quantile([.1, .5, .9]).round(1).tolist(),
      f'· confiabilidad entre mitades {sb(T[["marca_m1", "marca_m2"]].corr().iloc[0, 1]):.2f}')
print('  Pendiente de respuesta al rival: cuantiles 10/50/90', T.pendiente.quantile([.1, .5, .9]).round(2).tolist(),
      f'· confiabilidad entre mitades {sb(T[["pend_m1", "pend_m2"]].corr().iloc[0, 1]):.2f}')
T.to_csv(os.path.join(SAL5, 'r8z_equipos.csv'), index=False)
print(f'\nX6-R8z-1 terminó en {time.time() - t_ini:.0f} s. Guardado (espejo): r8z_marcaje · salidas/fw5/r8z_equipos.csv.')


In [ ]:
# [X6-R8u-0] = R8u-0 v1.0 (claude/codigo/sesion11/R8u_rasgos.py), parte "Tramos reales", con UNA línea cambiada para X6
# (preregistro, Aclaración 1 punto 5):
#   original:  RAIZ = '/content/drive/MyDrive/isac2026/'
#   X6:        RAIZ = RAIZ_X6   (carpeta espejo; se agrega un freno que lo verifica)
# Se incluyen sin cambios las funciones del original (e5, estado, ZON, pendiente_partes). Las pruebas T1-T3 del original no cambian
# nada y no se repiten. Todo lo demás es copia literal. Antes: X6-0, X6-R8a-1, X6-R8p-1, X6-R8z-1 en esta sesión.
import os, time
import numpy as np
import pandas as pd

def e5(m):                                  # copia de R4a
    m = np.asarray(m, dtype=float)
    return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)
def estado(df, defensa=False):
    """e5, e5b desde el equipo que cobra (defensa=False) o desde el que defiende (defensa=True). m_home viene de R8a (marcador local + offset)."""
    s = np.where(df['ev_local'].astype(bool), 1.0, -1.0) * (-1.0 if defensa else 1.0)
    return e5(s * df['m_home'].astype(float)), e5(s * df['m_home_b'].astype(float))
ZON = ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']
def pendiente_partes(Z):
    """Numerador y denominador por córner, centrando las proporciones por equipo defensor y temporada."""
    PA, PD = Z[[f'pa_{z}' for z in ZON]], Z[[f'pd_{z}' for z in ZON]]
    ok = PA.notna().all(axis=1) & PD.notna().all(axis=1); Z = Z[ok].copy()
    g = Z.groupby(['rival', 'temporada'])
    A = Z[[f'pa_{z}' for z in ZON]] - g[[f'pa_{z}' for z in ZON]].transform('mean')
    D = Z[[f'pd_{z}' for z in ZON]] - g[[f'pd_{z}' for z in ZON]].transform('mean')
    Z['num'] = (A.values * D.values).sum(1); Z['den'] = (A.values ** 2).sum(1)
    return Z

# --- Tramos reales ---
RAIZ = RAIZ_X6                                                       # X6: única línea cambiada
assert RAIZ == '/content/isac_fw6/', 'Freno X6: RAIZ no es la carpeta espejo; corre X6-0'   # X6: freno agregado
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
t0 = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
BP['rival'] = np.where(BP.team == BP.match_id.map(loc['local']), BP.match_id.map(loc['visita']), BP.match_id.map(loc['local']))
BP['temporada'] = BP.match_id.map(loc['temporada'])
B = BP[BP.period.isin([1, 2]) & BP.match_id.isin(loc.index)].copy()                     # D-9
B['e5'], B['e5b'] = estado(B); B['e5_d'], B['e5b_d'] = estado(B, defensa=True)
piezas = []
def agrega(d, num, nombre, den=1.0, defensa=False):
    d = d.assign(num=np.asarray(num, float), den=np.asarray(den, float) if np.ndim(den) else den)
    t = pd.DataFrame({'metrica': nombre, 'match_id': d.match_id.values, 'team': (d.rival if defensa else d.team).values,
                      'e5': (d.e5_d if defensa else d.e5).values, 'e5b': (d.e5b_d if defensa else d.e5b).values,
                      'cuarto': d.cuarto.astype(int).values, 'num': d.num.values, 'den': d.den.values})
    piezas.append(t.dropna(subset=['num', 'den']))

co = B[B.kind == 'corner']; cor = co[co.ruta.isin(['via_corto', 'reciclado'])]
env = co[co.ruta.isin(['directo', 'via_corto']) & co.zona.isin(ZON)]
dtec = co[(co.ruta == 'directo') & co.tecnica.notna()]
seg = co[co.caida_gana.notna() & (co.contacto_clase != 'control_atac')]
s360 = co[(co.ruta == 'directo') & co['s_frame'].astype('boolean').fillna(False)]
agrega(co, 100.0 * co.ruta.isin(['via_corto', 'reciclado']), 'corto_pct')
agrega(cor, 100.0 * (cor.ruta == 'via_corto'), 'corto_a_envio')
for z in ZON: agrega(env, 100.0 * (env.zona == z), f'envio_{z}')
agrega(dtec, 100.0 * (dtec.tecnica == 'Inswinging'), 'cerrado_pct')
agrega(co, 100.0 * (co.xg1 + co.xg2), 'xg_corner'); agrega(co, 100.0 * co.xg2, 'xg2_corner')
agrega(seg, 100.0 * seg.caida_gana.astype(bool), 'segunda_ganada')
agrega(s360, s360.s_atac_frontal, 'atac_frontal_360'); agrega(s360, s360.s_atac_area, 'atac_area_360')
sb = B[B.kind == 'saque_banda']; tl = B[B.kind == 'tiro_libre']
agrega(sb, 100.0 * (sb.ruta == 'directo'), 'saque_banda_largo')
agrega(tl, 100.0 * (tl.ruta == 'tiro_directo'), 'tl_directo'); agrega(tl, 100.0 * tl.ruta.isin(['via_corto', 'reciclado']), 'tl_corto')
# Defensa (córner del rival, asignado al que defiende)
agrega(co, 100.0 * (co.xg1 + co.xg2), 'xg_concedido_corner', defensa=True)
agrega(seg, 100.0 * (1 - seg.caida_gana.astype(bool)), 'segunda_recuperada', defensa=True)
agrega(s360, s360.s_def_area, 'def_area_360', defensa=True)
# Playbooks (R8p) y marcaje (R8z): se les pega el estado desde r8_bp
llave = ['match_id', 'index']
P = pd.read_parquet(os.path.join(FW5, 'r8p_corners.parquet'), columns=llave + ['pb_of', 'pegados_portero']).merge(B[B.kind == 'corner'], on=llave)
po = P[P.pb_of.notna()]
for k in range(int(po.pb_of.max()) + 1): agrega(po, 100.0 * (po.pb_of == k), f'pb_of_{k}')
pp = P[P.pegados_portero.notna()]; agrega(pp, pp.pegados_portero, 'pegados_portero')
Z = pd.read_parquet(os.path.join(FW5, 'r8z_marcaje.parquet')).drop(columns=['team', 'rival', 'temporada'], errors='ignore').merge(B[B.kind == 'corner'], on=llave)
zm = Z[Z.pct_marca_b.notna()]
agrega(zm, zm.pct_marca_b, 'pct_marca', defensa=True); agrega(zm, zm.zonales_b, 'zonales', defensa=True)
Zp = pendiente_partes(Z); agrega(Zp, Zp.num, 'pendiente_marca', den=Zp.den.values, defensa=True)
print(f'\n--- (0) Enlaces: playbooks {len(P):,} de {len(pd.read_parquet(os.path.join(FW5, "r8p_corners.parquet"), columns=["index"])):,} · marcaje {len(Z):,} ({time.time() - t0:.0f} s)')

# Peso del balón parado (por tiro, sin penales): marcador reconstruido con la copia literal de R6a sobre tiros y autogoles
pesos, ch_t, ch_p = [], [], []
OFF = {1: 0, 2: 45, 3: 90, 4: 105}
PE = pd.read_parquet(os.path.join(FW5, 'r8_penales.parquet'))
cor_t = BP[BP.kind == 'corner'][['match_id', 'index', 'period', 't', 'team']]
for temp in sorted(Mm.temporada.unique()):
    E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                        columns=['match_id', 'index', 'period', 'minute', 't', 'type.name', 'team.name', 'shot.type.name', 'shot.statsbomb_xg',
                                 'shot.outcome.name', 'fase'])
    E = E[E.match_id.isin(loc.index)].sort_values(['match_id', 'index'])
    ok_t = ((E.t - 60 * (E.minute - E.period.map(OFF))).between(-0.01, 60.01) | E.period.eq(5)).mean()
    ch_t.append((temp, round(100 * ok_t, 2)))
    pm = PE[PE.match_id.isin(set(E.match_id))].merge(E[['match_id', 'index', 't']], on=['match_id', 'index'], how='left')
    ch_p.append(pm)
    E = E[E['type.name'].isin(['Shot', 'Own Goal For'])].copy()
    E['local'] = E['match_id'].map(loc['local']); E['ev_local'] = E['team.name'] == E['local']
    es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) | (E['type.name'] == 'Own Goal For'))
    gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
    E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
    E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset']); E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
    E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < 25, 1, 2), np.where(E['minute'] < 70, 3, 4))
    sh = E[(E['type.name'] == 'Shot') & (E['shot.type.name'] != 'Penalty') & E.period.isin([1, 2]) & (E['shot.statsbomb_xg'] > 0)]
    sh = sh.rename(columns={'team.name': 'team'}); sh['e5'], sh['e5b'] = estado(sh)
    agrega(sh, 100.0 * sh['shot.statsbomb_xg'] * (sh.fase == 'bp'), 'peso_bp', den=sh['shot.statsbomb_xg'].values)
print('--- (1) Chequeo de t: % de eventos con t = segundos desde el inicio del PERIODO (minuto − 0/45/90/105):', dict(ch_t), '(debe ser ~100)')

TR = pd.concat(piezas, ignore_index=True).groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
TR = TR[TR.den != 0]
TR.to_parquet(os.path.join(FW5, 'r8_tramos.parquet'), index=False)
res = TR.groupby('metrica').apply(lambda d: pd.Series({'equipo_partidos': d[['match_id', 'team']].drop_duplicates().shape[0],
                                                       'media_liga': d.num.sum() / d.den.sum()}), include_groups=False)
print(f'--- (3) Rasgos ({TR.metrica.nunique()}). Media de la liga en su unidad:'); print(res.round(3).to_string())
print(f'X6-R8u-0 terminó en {time.time() - t0:.0f} s. Guardado (espejo): r8_tramos. Siguiente: X6-p con PREFIJO = "r8" → R6r-1 → R6r-2.')


In [ ]:
# [X6-9 v1.0] Validación de X6 contra fw5 y copia a Drive (preregistro de Almada, §4 paso 4; Aclaración 1)
# =====================================================================
# Sesión 13 · 2026-10-20 · Correr AL FINAL de la hoja de ruta X6, en la misma sesión de Colab. Tarda ~1-3 min.
# (1) Copia a Drive todo lo que X6 generó (no los enlaces): datos_parquet/fw6/ y salidas/fw6/. fw1 y fw5 no se tocan.
# (2) Reproducción (como X5-1): en los partidos de 2021/22-2025/26, los TRAMOS de los ocho bloques deben ser idénticos a fw5
#     (mismo código, mismos datos; solo cambia que hay una temporada más).
# (3) Validación obligatoria del preregistro: perfiles de las etapas ya existentes (mismos partidos que en fw5) con correlación
#     ≥ 0.99 por rasgo frente a fw5, en los 65 rasgos de C-1. Las etapas que ganaron partidos en 2026/27 se reportan aparte.
# (4) Escribe datos_parquet/fw6/validacion_x6.json (V-1 no corre si ok = false) y regresa BASE a Drive.
# NO imprime ningún perfil de equipo ni de técnico: solo conteos y correlaciones.
# =====================================================================
import os, json, shutil
import numpy as np
import pandas as pd

DRIVE = '/content/drive/MyDrive/isac2026/'
LOCAL = '/content/isac_fw6/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
assert globals().get('RAIZ_X6') == LOCAL, 'Corre X6-9 en la misma sesión que X6-0 y toda la hoja de ruta'
L_FW = os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_DP = os.path.join(DRIVE, 'datos_parquet'); D_FW5, D_FW6 = os.path.join(D_DP, 'fw5'), os.path.join(D_DP, 'fw6')
D_SAL5, D_SAL6 = os.path.join(DRIVE, 'salidas', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw6')
os.makedirs(D_FW6, exist_ok=True); os.makedirs(D_SAL6, exist_ok=True)
PREF = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8']
sal = lambda p: 'r5' if p == 'r4' else p

# ---------- (1) Copia ----------
n = []
for src_dir, dst_dir in [(L_FW, D_FW6), (os.path.join(LOCAL, 'salidas'), D_SAL6), (os.path.join(LOCAL, 'salidas', 'fw5'), D_SAL6)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print(f'--- (1) Copiados a Drive: {len(n)} archivos → datos_parquet/fw6/ y salidas/fw6/')
falta = [f'{p}_perfiles_contraidos' for p in [sal(x) for x in PREF] if not os.path.exists(os.path.join(D_FW6, f'{p}_perfiles_contraidos.parquet'))]
falta += [f'{sal(p)}_confiabilidad.csv' for p in PREF if not os.path.exists(os.path.join(D_SAL6, f'{sal(p)}_confiabilidad.csv'))]
print('Insumos de la compatibilidad que faltan en fw6:', falta or 'ninguno')

# ---------- (2) Reproducción de tramos en 2021/22-2025/26 ----------
mid5 = set(pd.read_parquet(os.path.join(D_FW5, 'r4_partidos.parquet'))['match_id'])
filas = []
for arch in ['r4_tramos_v21', 'r6_tramos', 'r6b_tramos', 'r6c_tramos', 'r7u_tramos', 'r7b_tramos', 'r7c_tramos', 'r8_tramos']:
    a = pd.read_parquet(os.path.join(D_FW5, f'{arch}.parquet'))
    b = pd.read_parquet(os.path.join(D_FW6, f'{arch}.parquet'))
    k = [c for c in ['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'] if c in a.columns and c in b.columns]
    j = a.merge(b[b['match_id'].isin(mid5)], on=k, how='outer', suffixes=('_5', '_6'), indicator=True)
    igual = np.isclose(j['num_5'].fillna(-1), j['num_6'].fillna(-1)) & np.isclose(j['den_5'].fillna(-1), j['den_6'].fillna(-1))
    filas.append(dict(archivo=arch, tramos_fw5=len(a), solo_en_uno=int((j['_merge'] != 'both').sum()), iguales_pct=100 * igual.mean(),
                      partidos_2026_27=int(b.loc[~b['match_id'].isin(mid5), 'match_id'].nunique())))
V = pd.DataFrame(filas)
print('\n--- (2) Reproducción de tramos en las cinco temporadas previas (solo_en_uno = 0 e iguales_pct = 100 en los ocho):')
print(V.round(3).to_string(index=False))
OK_TR = bool((V['solo_en_uno'] == 0).all() and (V['iguales_pct'] > 99.999).all() and (V['partidos_2026_27'] > 90).all())

# ---------- (3) Perfiles de las etapas existentes frente a fw5 ----------
RAS = pd.read_csv(os.path.join(D_SAL5, 'c_mapa.csv'))['rasgo'].tolist()           # los 65 de C-1 (congelados)
ptp5 = pd.read_parquet(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'))
ptp6 = pd.read_parquet(os.path.join(D_FW6, 'partido_tecnico_periodo.parquet'))
with open(os.path.join(D_FW5, 'parametros_fw1.json')) as f:
    T5 = set(json.load(f)['temporadas']['entrenamiento'])
eid = lambda d: d['team'] + ' · ' + d['etapa'].astype(str)
s5 = ptp5[ptp5.temporada.isin(T5) & ptp5.jugado.astype(bool)].assign(e=eid).groupby('e')['match_id'].apply(frozenset)
s6 = ptp6[ptp6.jugado.astype(bool)].assign(e=eid).groupby('e')['match_id'].apply(frozenset)
igual_e = [e for e in s5.index if e in s6.index and s5[e] == s6[e]]
crece = [e for e in s5.index if e in s6.index and s5[e] != s6[e]]
print(f'\n--- (3) Etapas en fw5: {len(s5)} · con los mismos partidos en X6: {len(igual_e)} · que ganaron partidos en 2026/27: {len(crece)} · '
      f'etapas nuevas en 2026/27: {len(set(s6.index) - set(s5.index))}')
res = []
for p in PREF:
    a = pd.read_parquet(os.path.join(D_FW5, f'{sal(p)}_perfiles_contraidos.parquet'))
    b = pd.read_parquet(os.path.join(D_FW6, f'{sal(p)}_perfiles_contraidos.parquet'))
    j = a.merge(b, on=['metrica', 'etapa_id'], suffixes=('_5', '_6'))
    for met, g in j.groupby('metrica'):
        r = f'{p}:{met}'
        if r not in RAS: continue
        def c(gg, col):
            m = np.isfinite(gg[col + '_5']) & np.isfinite(gg[col + '_6'])
            return np.corrcoef(gg.loc[m, col + '_5'], gg.loc[m, col + '_6'])[0, 1] if m.sum() >= 5 else np.nan
        gi, g15 = g[g.etapa_id.isin(igual_e)], g[g.etapa_id.isin(igual_e) & (g.partidos_5 >= 15)]
        gc = g[g.etapa_id.isin(crece)]
        res.append(dict(rasgo=r, etapas=len(gi), corr_perfil=c(gi, 'perfil_r4'), corr_mostrado=c(gi, 'perfil_mostrado'),
                        corr_15=c(g15, 'perfil_r4'), dif_max_z=np.nanmax(np.abs(gi.perfil_mostrado_5 - gi.perfil_mostrado_6)) /
                        max(1e-12, float(pd.read_csv(os.path.join(D_SAL5, f'{sal(p)}_confiabilidad.csv')).set_index('metrica').loc[met, 'tau'])),
                        corr_etapas_que_crecen=c(gc, 'perfil_r4') if len(gc) >= 5 else np.nan))
R = pd.DataFrame(res)
cf6 = pd.concat([pd.read_csv(os.path.join(D_SAL6, f'{sal(p)}_confiabilidad.csv')).assign(rasgo=lambda d: p + ':' + d.metrica) for p in PREF])
cf5 = pd.concat([pd.read_csv(os.path.join(D_SAL5, f'{sal(p)}_confiabilidad.csv')).assign(rasgo=lambda d: p + ':' + d.metrica) for p in PREF])
cc = cf5.set_index('rasgo')[['conf', 'tau']].join(cf6.set_index('rasgo')[['conf', 'tau']], lsuffix='_5', rsuffix='_6').reindex(RAS)
print(f'Rasgos de C-1 encontrados en X6: {len(R)} de {len(RAS)} (deben ser {len(RAS)})')
print(f'Correlación por rasgo (etapas con los mismos partidos), perfil ajustado: mínima {R.corr_perfil.min():.4f} · mediana {R.corr_perfil.median():.4f} · '
      f'rasgos < 0.99: {int((R.corr_perfil < 0.99).sum())}')
print(f'  perfil mostrado (tras P-5): mínima {R.corr_mostrado.min():.4f} · solo etapas de 15+: mínima {R.corr_15.min():.4f} · '
      f'cambio máximo en z (perfil mostrado ÷ τ de fw5): mediana {R.dif_max_z.median():.3f} · máximo {R.dif_max_z.max():.3f}')
print(f'  etapas que ganaron partidos (informativo): correlación mínima {R.corr_etapas_que_crecen.min():.3f}')
if (R.corr_perfil < 0.99).any(): print(R[R.corr_perfil < 0.99].round(4).to_string(index=False))
print(f'Confiabilidad X6 frente a fw5 en los 65: cambio máximo {np.nanmax(np.abs(cc.conf_6 - cc.conf_5)):.3f} · '
      f'rasgos que en X6 quedarían < 0.5: {int((cc.conf_6 < 0.5).sum())} (la lista sigue siendo la congelada; solo se reporta) · '
      f'τ X6/fw5: mediana {np.nanmedian(cc.tau_6 / cc.tau_5):.3f}')
OK_PF = bool(len(R) == len(RAS) and (R.corr_perfil >= 0.99).all())

# ---------- (4) Marca y regreso a Drive ----------
OK = OK_TR and OK_PF and not falta
with open(os.path.join(D_FW6, 'validacion_x6.json'), 'w') as f:
    json.dump(dict(ok=OK, tramos=OK_TR, perfiles=OK_PF, faltan=falta, tabla_tramos=V.to_dict('records'),
                   corr_min=float(R.corr_perfil.min()), rasgos_bajo_099=R.loc[R.corr_perfil < 0.99, 'rasgo'].tolist()), f, indent=1, default=str)
R.to_csv(os.path.join(D_SAL6, 'x6_validacion_perfiles.csv'), index=False)
for v in ['RAIZ_PRUEBA', 'RAIZ_X6']: globals().pop(v, None)
BASE = DRIVE
print('\nVALIDACIÓN X6:', 'OK → se puede correr V-1' if OK else 'FALLÓ → NO correr V-1; manda esta salida a Claude',
      '· BASE =', BASE, '(RAIZ_PRUEBA y RAIZ_X6 borrados)')


In [ ]:
# [V-1 v1.0] Evaluación preregistrada de Almada en América: P1-P4 (preregistro §3-§4 y Aclaraciones 1 y 2)
# =====================================================================
# Sesión 13 · 2026-10-20 · Antes: celda 1 (Drive) → C-0 (claude/codigo/sesion12/C_0_prueba.py, SIN CAMBIOS: define las funciones y
# debe terminar con todas sus pruebas PASAN) → X6 completo con "VALIDACIÓN X6: OK" → V-1. Si X6-9 se corrió en otra sesión, basta
# con celda 1 → C-0 → V-1. Tarda ~3-6 min. No usa C-1 ni C-3 de la sesión 12 en memoria: los rehace aquí.
# Código: las secciones 0, 1, 3 y 4 de C-1 y el bucle de C-3 se copian SIN CAMBIOS DE LÓGICA dentro de construir() y ensayo(); lo único
# distinto es de dónde leen (fw5 o fw6) y que la lista de rasgos es la congelada de C-1 (Aclaración 1, punto 1).
# Pasos (cada uno se detiene si falla):
#   (A) REPRODUCCIÓN sobre fw5: las 27 llegadas de C-3 con K = 9, 12 y 15 deben dar los mismos números que salidas/fw5/
#       c_ensayo_llegadas.csv (compatibilidad, cambio, percentil y puesto de identificación). Si no, no se evalúa a Almada.
#   (B) X6: mismas 27 llegadas (las de 2026/27 no entran al ensayo) → umbral de P1 = percentil 75 del cambio con K = 11.
#   (C) Almada: zt (Pachuca · 2), zc (últimos 18 de América antes de su llegada), zn (sus K partidos de 2026/27), pesos sin sus parejas.
#   (D) P1 a P4 por separado (cumple / no cumple / no evaluado) con su valor · cambios de zt y zc > 0.1 z frente a fw5 · sensibilidad fw5.
# Escribe (con freno de Drive) salidas/fw6/v1_resultado.csv, v1_rasgos.csv, v1_p4_ranking.csv, v1_reproduccion.csv.
# =====================================================================
import os, json, time
from types import SimpleNamespace
import numpy as np
import pandas as pd

DRIVE = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 y confirma que todas sus pruebas PASAN'
assert 'RAIZ_PRUEBA' not in globals() and 'RAIZ_X6' not in globals(), 'Quedan variables de X6: corre X6-9 (las borra) o reinicia y corre celda 1 → C-0'
DP = os.path.join(DRIVE, 'datos_parquet')
VERS = {'fw5': dict(FW=os.path.join(DP, 'fw5'), SAL=os.path.join(DRIVE, 'salidas', 'fw5'), PTP=os.path.join(DP, 'partido_tecnico_periodo.parquet')),
        'fw6': dict(FW=os.path.join(DP, 'fw6'), SAL=os.path.join(DRIVE, 'salidas', 'fw6'), PTP=os.path.join(DP, 'fw6', 'partido_tecnico_periodo.parquet'))}
with open(os.path.join(VERS['fw6']['FW'], 'validacion_x6.json')) as f:
    VX6 = json.load(f)
assert VX6.get('ok'), 'X6 no pasó su validación (validacion_x6.json): no se evalúa'
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8']
ETAPA_MIN, NUEVA_MIN = 15, 10                      # C-1
VENTANA, VENT_MIN = 18, 10                         # S12-05
CONF_MIN = 0.5
RELOJ_P4 = 20                                      # C-3
FOCO = {'P2a': 'r8:def_area_360', 'P2b': 'r8:zonales', 'P3': 'r6:seg_hasta_ultimo'}
INICIO_VIVO = pd.Timestamp('2026-07-01')           # llegadas de 2026/27: no entran al ensayo (las 27 son históricas)
RAS = pd.read_csv(os.path.join(VERS['fw5']['SAL'], 'c_mapa.csv'))['rasgo'].tolist()           # los 65 congelados de C-1
CUAD = pd.read_csv(os.path.join(VERS['fw5']['SAL'], 'c_mapa.csv')).set_index('rasgo')['cuadrante']
sal = lambda p: 'r5' if p == 'r4' else p
corto = lambda s: ' '.join(s.split()[:1] + [s.split()[-2] if len(s.split()) >= 3 else s.split()[-1]]) if isinstance(s, str) and s.strip() else s
t0 = time.time(); pd.set_option('display.width', 250)


def construir(nombre):
    """C-1 secciones 0, 1, 3 y 4 (sin cambios de lógica) sobre la versión indicada."""
    FW5, SAL5, PTPF = VERS[nombre]['FW'], VERS[nombre]['SAL'], VERS[nombre]['PTP']
    REQ = {'contraidos': ['metrica', 'etapa_id', 'perfil_r4', 'perfil_mostrado'], 'confiabilidad': ['metrica', 'conf', 'tau', 'sigma_partido'],
           'perfiles_etapa': ['etapa_id', 'metrica', 'tau_estado'], 'residuos': ['metrica', 'match_id', 'team', 'e', 'den', 'res']}
    CONT, CF, TAUE, RES, S2 = [], [], {}, {}, {}
    for p in PREFIJOS:
        fuentes = {'contraidos': os.path.join(FW5, f'{sal(p)}_perfiles_contraidos.parquet'), 'confiabilidad': os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'),
                   'perfiles_etapa': os.path.join(FW5, f'{p}_perfiles_etapa.parquet'), 'residuos': os.path.join(FW5, f'{p}_residuos.parquet')}
        D = {k: (pd.read_csv(v) if v.endswith('.csv') else pd.read_parquet(v)) for k, v in fuentes.items()}
        faltan = {k: [c for c in REQ[k] if c not in D[k].columns] for k in D}
        assert not any(faltan.values()), f'{nombre}: columnas faltantes en {p}: {faltan}'
        CONT.append(D['contraidos'].assign(rasgo=lambda d: p + ':' + d.metrica))
        CF.append(D['confiabilidad'].assign(rasgo=lambda d: p + ':' + d.metrica, prefijo=p))
        TAUE.update({p + ':' + m: v for m, v in D['perfiles_etapa'].groupby('metrica')['tau_estado'].first().items()})
        rs = D['residuos'][REQ['residuos']]
        S2.update({p + ':' + m: float((g.den * g.res ** 2).sum() / (len(g) - 19)) for m, g in rs.groupby('metrica')})
        RES[p] = {t: g for t, g in rs.groupby('team')}
    CONT = pd.concat(CONT, ignore_index=True); CF = pd.concat(CF, ignore_index=True).set_index('rasgo')
    RASGOS_SEL = [r for r in CF.index if CF.loc[r, 'conf'] >= CONF_MIN and CF.loc[r, 'tau'] > 0]
    faltan = [r for r in RAS if r not in CF.index or not CF.loc[r, 'tau'] > 0]
    assert not faltan, f'{nombre}: rasgos congelados sin dato: {faltan}'
    RASGOS = list(RAS)                                                       # Aclaración 1 (1): lista congelada
    with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
        TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
    ptp = pd.read_parquet(PTPF)
    if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
    ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
    ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
    ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                     ini=('fecha', 'min'), fin=('fecha', 'max')).reset_index()
    ETAPAS = ET.etapa_id.tolist(); IX = {e: i for i, e in enumerate(ETAPAS)}
    tau = CF.loc[RASGOS, 'tau'].values; conf = CF.loc[RASGOS, 'conf'].values
    ZR = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_r4').reindex(index=ETAPAS, columns=RASGOS) / tau).values
    ZM = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_mostrado').reindex(index=ETAPAS, columns=RASGOS) / tau).values
    L = ET[ET.partidos >= ETAPA_MIN]
    pt, ct = [], []
    for tec, d in L.groupby('tecnico'):
        ix = [IX[e] for e in d.etapa_id]
        for i in range(len(ix)):
            for j in range(i + 1, len(ix)):
                if ET.team[ix[i]] != ET.team[ix[j]]: pt.append((ix[i], ix[j])); ct.append(tec)
    pt, ct = np.array(pt), np.array(ct)
    MIN_ = pd.read_parquet(os.path.join(FW5, 'r4_minutos_v21.parquet'))
    W3 = MIN_.assign(e3=np.sign(MIN_.e5)).groupby('e3')['minutos'].sum(); W3v = (W3 / W3.sum()).reindex([-1, 0, 1]).fillna(0).values

    def perfil_club(team, partidos, contraido=True, rasgos=None):
        rasgos = RASGOS if rasgos is None else rasgos
        z = np.full(len(rasgos), np.nan)
        for j, r in enumerate(rasgos):
            p, m = r.split(':', 1); g = RES[p].get(team)
            if g is None: continue
            g = g[(g.metrica == m) & g.match_id.isin(partidos)]
            n = g.match_id.nunique()
            if n == 0: continue
            pf = perfil_ventana(g.e.values, g.den.values, g.res.values, S2[r], TAUE[r], W3v)
            z[j] = (contraer(pf, n, CF.loc[r, 'tau'], CF.loc[r, 'sigma_partido']) if contraido else pf) / CF.loc[r, 'tau']
        return z
    def n_con_dato(team, partidos, r):
        p, m = r.split(':', 1); g = RES[p].get(team)
        return 0 if g is None else int(g[(g.metrica == m) & g.match_id.isin(partidos)].match_id.nunique())
    def ventana(team, antes_de):
        m = ptp[(ptp.team == team) & (ptp.fecha < antes_de)].sort_values('fecha').tail(VENTANA)
        return set(m.match_id), len(m)
    def pesos_sin(tec):
        keep = ct != tec
        r = r_pares(ZR, pt[keep]) / np.clip(conf, .05, None)
        return np.clip(np.nan_to_num(r), 0, PESO_MAX)
    def candidatos(antes_de, excluir_tec, excluir_team):
        c = L[(L.ini < antes_de) & (L.tecnico != excluir_tec) & (L.team != excluir_team)].sort_values('ini').groupby('tecnico').tail(1)
        return [IX[e] for e in c.etapa_id], c.tecnico.tolist()
    filas = []
    for r_ in ET[ET.partidos >= NUEVA_MIN].itertuples(index=False):
        ant = L[(L.tecnico == r_.tecnico) & (L.team != r_.team) & (L.fin < r_.ini)]
        if not len(ant): continue
        prev = ant.sort_values('fin').etapa_id.iloc[-1]
        vp, nv = ventana(r_.team, r_.ini)
        if nv < VENT_MIN: filas.append(dict(etapa_nueva=r_.etapa_id, tecnico=corto(r_.tecnico), previa=prev, ventana=nv, nota='ventana corta')); continue
        zc = perfil_club(r_.team, vp); zt = ZM[IX[prev]]; w = pesos_sin(r_.tecnico)
        cix, _ = candidatos(r_.ini, r_.tecnico, r_.team)
        d = distancia(zt, zc, w); comp = percentil(d, [distancia(ZM[k], zc, w) for k in cix])
        zn = ZM[IX[r_.etapa_id]]
        filas.append(dict(etapa_nueva=r_.etapa_id, tecnico=corto(r_.tecnico), tec_full=r_.tecnico, ini=r_.ini, previa=prev, ventana=nv,
                          partidos_nueva=r_.partidos, candidatos=len(cix), compat=comp, distancia=d, cambio=distancia(zc, zn, w),
                          v_zc=zc, v_zt=zt, v_zn=zn, v_w=w, v_cix=cix, nota=''))
    LL = pd.DataFrame(filas)
    ok = LL[LL.nota == ''].copy()
    return SimpleNamespace(**{k: v for k, v in locals().items() if k != 'filas'})


def ensayo(S, K_LIST):
    """Bucle de C-3 (sin cambios de lógica): cambio con los primeros K partidos e identificación a ciegas. Solo llegadas históricas."""
    reloj = S.CF.loc[S.RASGOS, 'reloj'].values; legible = reloj <= RELOJ_P4
    filas = []
    for f in S.ok[S.ok.ini < INICIO_VIVO].to_dict('records'):
        e_new = f['etapa_nueva']; k_new = S.IX[e_new]; team = S.ET.team[k_new]; tec_full = S.ET.tecnico[k_new]
        w = S.pesos_sin(tec_full); zc, zt = f['v_zc'], f['v_zt']
        mids = S.ptp[S.ptp.etapa_id == e_new].sort_values('fecha').match_id.tolist()
        for K in K_LIST:
            if len(mids) < K: continue
            zk = S.perfil_club(team, set(mids[:K]))
            cam_k = distancia(zc, zk, w)
            wl = w * legible
            d_real = distancia(zk, zt, wl)
            d_cand = [distancia(zk, S.ZM[j], wl) for j in f['v_cix']]
            filas.append(dict(etapa_nueva=e_new, tecnico=f['tecnico'], K=K, compat=f['compat'], cambio_completo=f['cambio'], cambio_K=cam_k,
                              id_percentil=percentil(d_real, d_cand), id_puesto=1 + int(np.sum(np.array(d_cand) < d_real)), candidatos=len(d_cand) + 1))
    return pd.DataFrame(filas)


# ---------- (A) Reproducción sobre fw5 ----------
S5 = construir('fw5')
assert S5.RASGOS == [r for r in S5.RASGOS_SEL], 'fw5: la lista congelada no coincide con la selección de C-1 (orden o contenido)'
EN5 = ensayo(S5, [9, 12, 15])
REF = pd.read_csv(os.path.join(VERS['fw5']['SAL'], 'c_ensayo_llegadas.csv'))
J = REF.merge(EN5, on=['etapa_nueva', 'K'], how='outer', suffixes=('_ref', '_v1'), indicator=True)
COLS_R = ['compat', 'cambio_completo', 'cambio_K', 'id_percentil', 'id_puesto', 'candidatos']
difs = {c: float(np.nanmax(np.abs(J[c + '_ref'] - J[c + '_v1']))) for c in COLS_R}
OK_REP = bool((J._merge == 'both').all() and max(difs.values()) < 1e-6)
print(f'--- (A) Reproducción de C-3 sobre fw5: llegadas {EN5.etapa_nueva.nunique()} · filas (llegada × K) {len(EN5)} de {len(REF)} · '
      f'solo en una {int((J._merge != "both").sum())} · diferencia máxima {max(difs.values()):.2e} (debe ser < 1e-6) · {time.time() - t0:.0f} s')
print('  por columna:', {k: f'{v:.1e}' for k, v in difs.items()})
assert OK_REP, 'V-1 NO reproduce C-3: no se evalúa a Almada. Manda esta salida a Claude.'

# ---------- (B) X6: ensayo con K = 11 → umbral de P1 ----------
S6 = construir('fw6')
lleg5, lleg6 = set(S5.ok.etapa_nueva), set(S6.ok[S6.ok.ini < INICIO_VIVO].etapa_nueva)
EN6 = ensayo(S6, [11]); EN5_11 = ensayo(S5, [11])
UMBRAL_P1, UMBRAL_P1_FW5 = float(EN6.cambio_K.quantile(.75)), float(EN5_11.cambio_K.quantile(.75))
print(f'\n--- (B) X6 · llegadas históricas {len(lleg6)} (iguales a las de fw5: {lleg6 == lleg5}) · llegadas de 2026/27 fuera del ensayo: '
      f'{int((S6.ok.ini >= INICIO_VIVO).sum())} · con 11+ partidos para K = 11: {len(EN6)}')
print(f'  Cambio con K = 11 (z): mediana {EN6.cambio_K.median():.2f} · p75 {UMBRAL_P1:.3f} ← UMBRAL DE P1 · p90 {EN6.cambio_K.quantile(.9):.2f} '
      f'· (fw5 con K = 11: p75 {UMBRAL_P1_FW5:.3f}; indicativos del preregistro K = 9 → 1.01, K = 12 → 1.06)')
print(f'  Identificación a ciegas en las llegadas, K = 11: en el 10% más parecido {100 * (EN6.id_percentil >= 90).mean():.0f}% (ensayo K = 12: 37%) · '
      f'percentil del técnico real: p25 {EN6.id_percentil.quantile(.25):.0f} · mediana {EN6.id_percentil.median():.0f} · p75 {EN6.id_percentil.quantile(.75):.0f} (azar = 50)')
assert lleg6 == lleg5, 'Las llegadas históricas de X6 no son las mismas 27 de fw5: avisar a Claude'

# ---------- (C) Almada ----------
def almada(S):
    et = S.ET
    e_am = et[et.team.str.startswith('Am') & et.tecnico.str.contains('Almada', na=False)]
    e_pa = et[et.team.str.startswith('Pachuca') & et.tecnico.str.contains('Almada', na=False)].sort_values('partidos')
    return (e_am.etapa_id.iloc[0] if len(e_am) else None), e_pa.etapa_id.iloc[-1], et
e_am, e_pa, ET6 = almada(S6)
assert e_am is not None, 'No está la etapa de Almada en América en X6'
r_am = ET6.set_index('etapa_id').loc[e_am]; team_am, tec_alm, ini_alm = r_am.team, r_am.tecnico, r_am.ini
mids_alm = S6.ptp[S6.ptp.etapa_id == e_am].sort_values('fecha').match_id.tolist(); K = len(mids_alm)
vp, nv = ventana_ = S6.ventana(team_am, ini_alm)
fin_v = S6.ptp[S6.ptp.match_id.isin(vp) & (S6.ptp.team == team_am)].fecha.max().date()
zt, zc, zn = S6.ZM[S6.IX[e_pa]], S6.perfil_club(team_am, vp), S6.perfil_club(team_am, set(mids_alm))
w = S6.pesos_sin(tec_alm)
print(f'\n--- (C) Almada · etapa nueva {e_am}: K = {K} partidos ({pd.Timestamp(ini_alm).date()} a {S6.ptp[S6.ptp.etapa_id == e_am].fecha.max().date()}) · '
      f'técnico: {e_pa} ({int(ET6.set_index("etapa_id").loc[e_pa, "partidos"])} partidos; preregistro: Pachuca · 2, 139) · '
      f'club: últimos {nv} de {team_am} hasta {fin_v} (preregistro: 18, hasta 2026-05-11) · rasgos con peso > 0: {int((w > 0).sum())}')

# ---------- (D) Predicciones ----------
R = S6.RASGOS; ix = {r: i for i, r in enumerate(R)}
liga6 = S6.CONT.groupby('rasgo')['liga'].first().reindex(R).values; tau6 = S6.CF.loc[R, 'tau'].values
def cob(i): return (zn[i] - zc[i]) / (zt[i] - zc[i]) if np.isfinite(zt[i] - zc[i]) and zt[i] != zc[i] else np.nan
filas = []
# P1
d1 = distancia(zc, zn, w)
filas.append(dict(id='P1', prediccion='América cambia más que una llegada típica', valor=round(d1, 3), umbral=f'> {UMBRAL_P1:.3f} (p75, K = {K})',
                  resultado='cumple' if d1 > UMBRAL_P1 else 'no cumple'))
# P2 y P3
det = []
for clave, r in FOCO.items():
    i = ix[r]; c = cob(i); n = S6.n_con_dato(team_am, set(mids_alm), r)
    det.append(dict(pred=clave, rasgo=r, partidos_con_dato=n, z_america_antes=zc[i], z_almada=zt[i], z_nuevo=zn[i], cobertura=c,
                    nat_america_antes=liga6[i] + zc[i] * tau6[i], nat_almada=liga6[i] + zt[i] * tau6[i], nat_nuevo=liga6[i] + zn[i] * tau6[i],
                    liga=liga6[i], peso=w[i]))
DET = pd.DataFrame(det)
c2 = DET.set_index('pred').cobertura
p2_ok = c2[['P2a', 'P2b']].notna().all()
filas.append(dict(id='P2', prediccion='Lo del técnico viaja (defensores en el área en córner y zonales)',
                  valor=f"{c2['P2a']:.2f} y {c2['P2b']:.2f}", umbral='≥ 0.5 en los dos',
                  resultado=('cumple' if (c2[['P2a', 'P2b']] >= 0.5).all() else 'no cumple') if p2_ok else 'no evaluado'))
filas.append(dict(id='P3', prediccion='Lo del club no viaja (segundos hasta el último tercio)', valor=f"{c2['P3']:.2f}", umbral='< 0.5',
                  resultado=('cumple' if c2['P3'] < 0.5 else 'no cumple') if np.isfinite(c2['P3']) else 'no evaluado'))
# P4: candidatos del ranking de C-1 (incluye a Almada en Pachuca), perfiles con partidos anteriores a su llegada (Aclaraciones 1 y 2)
RK = pd.read_csv(os.path.join(VERS['fw5']['SAL'], 'c_ranking_america.csv'), index_col=0)
assert e_pa in set(RK.etapa), 'Almada en Pachuca no está en el ranking de C-1'
reloj6 = S6.CF.loc[R, 'reloj'].values; reloj5 = S5.CF.loc[R, 'reloj'].values
wl = w * (reloj6 <= RELOJ_P4)
def perfil_antes(S, e):
    tm = S.ET.set_index('etapa_id').loc[e, 'team']
    m = set(S.ptp[(S.ptp.etapa_id == e) & (S.ptp.fecha < ini_alm)].match_id)
    return S.perfil_club(tm, m), len(m), int(S.ET.set_index('etapa_id').loc[e, 'partidos'])
P4 = []
for e in RK.etapa:
    z_e, n_antes, n_total = perfil_antes(S6, e)
    P4.append(dict(etapa=e, tecnico=corto(S6.ET.set_index('etapa_id').loc[e, 'tecnico']), partidos_antes=n_antes, partidos_X6=n_total,
                   real=(e == e_pa), distancia=distancia(zn, z_e, wl)))
P4 = pd.DataFrame(P4).sort_values('distancia').reset_index(drop=True); P4.index += 1
d_real = float(P4.loc[P4.real, 'distancia'].iloc[0]); d_cand = P4.loc[~P4.real, 'distancia'].values
pct4, puesto4 = percentil(d_real, d_cand), 1 + int(np.sum(d_cand < d_real))
filas.append(dict(id='P4', prediccion='Identificación a ciegas: los datos señalan a Almada', valor=f'percentil {pct4:.0f} (puesto {puesto4} de {len(P4)})',
                  umbral='percentil ≥ 90', resultado='cumple' if pct4 >= 90 else 'no cumple'))
RES_ = pd.DataFrame(filas)
print(f'\n--- (D) RESULTADO (lectura literal, E-16) · K = {K} · versión X6 ---')
print(RES_.to_string(index=False))
print('\nDetalle P2-P3 (z = desviaciones típicas entre técnicos; nat = unidad del rasgo; cobertura 0 = no se movió, 1 = igual a Almada):')
print(DET.round(2).to_string(index=False))
print(f'\nP4 · rasgos usados: {int((wl > 0).sum())} (reloj ≤ {RELOJ_P4} y peso > 0; con el reloj de fw5 serían {int(((w > 0) & (reloj5 <= RELOJ_P4)).sum())}) · '
      f'candidatos con partidos después del 19-jul recortados: {int((P4.partidos_X6 > P4.partidos_antes).sum())} · los 8 más parecidos:')
print(P4.head(8)[['tecnico', 'etapa', 'distancia', 'real']].round(3).to_string())

# Cambios de zt y zc frente a fw5 (preregistro §4 paso 4: > 0.1 z en un rasgo de P2-P4 se reporta)
zt5, zc5 = S5.ZM[S5.IX[e_pa]], S5.perfil_club(team_am, vp)
usados = sorted(set(FOCO.values()) | {R[i] for i in np.where(wl > 0)[0]})
CH = pd.DataFrame([dict(rasgo=r, dz_almada=zt[ix[r]] - zt5[ix[r]], dz_america=zc[ix[r]] - zc5[ix[r]]) for r in usados])
grandes = CH[(CH.dz_almada.abs() > 0.1) | (CH.dz_america.abs() > 0.1)]
print(f'\nCambios de zt y zc (X6 − fw5) en los {len(usados)} rasgos de P2-P4: máximo |Δ| Almada {CH.dz_almada.abs().max():.3f} · América {CH.dz_america.abs().max():.3f} · '
      f'rasgos con > 0.1 z: {len(grandes)}')
if len(grandes): print(grandes.round(3).to_string(index=False))
# Sensibilidad fw5 (pesos, umbral, zt y zc de fw5; zn solo existe en X6)
w5 = S5.pesos_sin(tec_alm); d1_5 = distancia(zc5, zn, w5)
cob5 = {k: (zn[ix[r]] - zc5[ix[r]]) / (zt5[ix[r]] - zc5[ix[r]]) for k, r in FOCO.items()}
print(f'\nSensibilidad fw5 · P1: {d1_5:.3f} frente a {UMBRAL_P1_FW5:.3f} → {"cumple" if d1_5 > UMBRAL_P1_FW5 else "no cumple"} · '
      f'P2: {cob5["P2a"]:.2f} y {cob5["P2b"]:.2f} → {"cumple" if min(cob5["P2a"], cob5["P2b"]) >= 0.5 else "no cumple"} · '
      f'P3: {cob5["P3"]:.2f} → {"cumple" if cob5["P3"] < 0.5 else "no cumple"}')

# ---------- Guardar ----------
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
S6_SAL = VERS['fw6']['SAL']
RES_.assign(K=K, umbral_p1=UMBRAL_P1, umbral_p1_fw5=UMBRAL_P1_FW5).to_csv(os.path.join(S6_SAL, 'v1_resultado.csv'), index=False)
DET.to_csv(os.path.join(S6_SAL, 'v1_rasgos.csv'), index=False); P4.to_csv(os.path.join(S6_SAL, 'v1_p4_ranking.csv'))
J.to_csv(os.path.join(S6_SAL, 'v1_reproduccion.csv'), index=False); CH.to_csv(os.path.join(S6_SAL, 'v1_cambios_zt_zc.csv'), index=False)
print(f'\nV-1 terminó en {time.time() - t0:.0f} s. Guardado: salidas/fw6/v1_resultado, v1_rasgos, v1_p4_ranking, v1_reproduccion, v1_cambios_zt_zc.')

# Sección 14 (Entregables)

In [ ]:
# [G-0 v1.0] Cadena de relevos de los técnicos del selector (SOLO LECTURA: no escribe nada)
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Decisiones S14-04 (selector) y S14-06 (cadena de relevos)
# Para cada etapa de Jardine, Larcamón y Ortiz en las cinco temporadas (fw5) imprime:
#   (1) quiénes son en la tabla de técnicos (nombres exactos) y sus etapas (club, partidos, fechas, rol);
#   (2) quién dirigía antes en ese club y si la ventana del club (últimos 18 partidos antes de llegar, mínimo 10; S12-05)
#       cabe dentro de los datos; qué técnicos componen esa ventana;
#   (3) nivel y resultados como CONTEXTO (no se interpretan como efecto del técnico): Elo medio de la etapa y su percentil
#       entre las etapas de 15+ partidos, Elo medio de la ventana del club, puntos por partido y diferencia de goles por partido;
#   (4) si la etapa tiene perfil en fw5 (r4_perfiles_etapa) y si entra en el mapa de C-1 como etapa de 15+.
# Orden: celda 1 (Drive) → G-0. Tarda < 1 min. No toca fw1, fw5 ni fw6.
# =====================================================================
import os, json
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
VENTANA, VENTANA_MIN, ETAPA_MIN = 18, 10, 15                      # S12-05 y P-5
BUSCAR = {'Jardine': 'Jardine', 'Larcamón': 'Larcam', 'Ortiz': 'Ortiz'}
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 80)

with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
print('Temporadas de fw5:', TEMPS)

# ---------- Tabla de técnicos (mismo identificador que C-00, R4b, R5: etapa_id = equipo · etapa) ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max'),
                                 rol=('rol', lambda s: s.mode().iat[0])).reset_index()
PRIMERA = ptp.groupby('team')['fecha'].min()                       # primer partido de cada club en los datos

# ---------- (1) Nombres y etapas ----------
print('\n--- (1) Nombres en la tabla de técnicos que contienen cada búsqueda (revisar que sean las personas correctas) ---')
SEL = {}
for nombre, patron in BUSCAR.items():
    cand = ET[ET.tecnico.astype(str).str.contains(patron, case=False)]
    print(f'  {nombre}: ' + (' | '.join(f"{t} → {', '.join(sorted(g.team))}" for t, g in cand.groupby('tecnico')) or 'NINGUNO'))
    if nombre == 'Ortiz':                                          # el Ortiz de América (puede haber homónimos)
        ok = cand.groupby('tecnico').team.apply(lambda s: s.str.startswith('Am').any())
        cand = cand[cand.tecnico.isin(ok[ok].index)]
    if cand.tecnico.nunique() != 1:
        print(f'  AVISO: {nombre} coincide con {cand.tecnico.nunique()} técnicos distintos; se listan todos y Claude revisa.')
    SEL[nombre] = cand.sort_values('ini')

# ---------- Elo y resultados por equipo-partido (fw1 tiene las cinco temporadas: 1,789 partidos) ----------
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre']]
par = pd.read_parquet(os.path.join(FW1, 'elo_partido_e2.parquet'))[['match_id', 'local', 'visita', 'gl', 'gv']]
res = pd.concat([par.assign(team=par.local, gf=par.gl, gc=par.gv), par.assign(team=par.visita, gf=par.gv, gc=par.gl)])
res['pts'] = np.select([res.gf > res.gc, res.gf == res.gc], [3, 1], 0)
EP = ptp[['match_id', 'team', 'fecha', 'etapa_id', 'tec_p']].merge(elo, on=['match_id', 'team'], how='left') \
        .merge(res[['match_id', 'team', 'gf', 'gc', 'pts']], on=['match_id', 'team'], how='left')
print(f'\nChequeo de cruce: equipo-partidos {len(EP)} · sin Elo {int(EP.elo_pre.isna().sum())} · sin marcador {int(EP.pts.isna().sum())} '
      f'(ambos deben ser 0 o casi 0; si no, los nombres de equipo no coinciden entre tablas)')
ELO_ET = EP.groupby('etapa_id').elo_pre.mean()
ref15 = ELO_ET[ELO_ET.index.isin(ET.loc[ET.partidos >= ETAPA_MIN, 'etapa_id'])]

perf_ok = set()
ruta_p = os.path.join(FW5, 'r4_perfiles_etapa.parquet')
if os.path.exists(ruta_p): perf_ok = set(pd.read_parquet(ruta_p, columns=['etapa_id']).etapa_id)

# ---------- (2) y (3) Cadena por etapa ----------
filas = []
for nombre, etapas in SEL.items():
    for e in etapas.itertuples(index=False):
        club = EP[(EP.team == e.team) & (EP.fecha < e.ini)].sort_values('fecha')
        ven = club.tail(VENTANA)
        prev = ET[(ET.team == e.team) & (ET.fin < e.ini)].sort_values('ini')
        ant = prev.iloc[-1] if len(prev) else None
        ant_largo = prev[(prev.partidos > 3) & (prev.rol != 'interino')]
        ant_largo = ant_largo.iloc[-1] if len(ant_largo) else None
        d = EP[EP.etapa_id == e.etapa_id]
        filas.append({
            'técnico': nombre, 'etapa_id': e.etapa_id, 'partidos': e.partidos, 'rol': e.rol,
            'del': e.ini.date(), 'al': e.fin.date(),
            'llega con datos previos del club': 'sí' if e.ini > PRIMERA[e.team] else 'NO (empieza con los datos)',
            'antecesor inmediato': '—' if ant is None else f'{ant.tecnico} ({ant.partidos}, {ant.rol})',
            'último antecesor no interino': '—' if ant_largo is None else f'{ant_largo.tecnico} ({ant_largo.partidos})',
            'partidos del club en la ventana': len(ven),
            'ventana válida (≥ 10)': len(ven) >= VENTANA_MIN,
            'técnicos en la ventana': ', '.join(f'{t} ({n})' for t, n in ven.tec_p.value_counts().items()) or '—',
            'Elo medio etapa': round(d.elo_pre.mean(), 0),
            'percentil Elo (etapas 15+)': round(100 * (ref15 < d.elo_pre.mean()).mean(), 0),
            'Elo medio ventana del club': round(ven.elo_pre.mean(), 0) if len(ven) else np.nan,
            'puntos por partido': round(d.pts.mean(), 2),
            'dif. goles por partido': round((d.gf - d.gc).mean(), 2),
            'perfil en fw5': e.etapa_id in perf_ok, 'en el mapa (15+)': e.partidos >= ETAPA_MIN})
G = pd.DataFrame(filas)
print('\n--- (2) Cadena de relevos: quién estaba antes y si la ventana del club cabe en los datos ---')
print(G[['técnico', 'etapa_id', 'partidos', 'rol', 'del', 'al', 'llega con datos previos del club', 'antecesor inmediato',
         'último antecesor no interino', 'partidos del club en la ventana', 'ventana válida (≥ 10)']].to_string(index=False))
print('\nTécnicos que componen la ventana del club (últimos 18 partidos antes de llegar):')
print(G[['etapa_id', 'técnicos en la ventana']].to_string(index=False))
print('\n--- (3) Nivel y resultados (CONTEXTO, no efecto del técnico) ---')
print(G[['técnico', 'etapa_id', 'Elo medio etapa', 'percentil Elo (etapas 15+)', 'Elo medio ventana del club',
         'puntos por partido', 'dif. goles por partido']].to_string(index=False))
print(f'Referencia: Elo medio de las {len(ref15)} etapas de 15+: mediana {ref15.median():.0f} · p10 {ref15.quantile(.1):.0f} · p90 {ref15.quantile(.9):.0f}')
print('\n--- (4) ¿Tiene perfil en fw5 y entra al mapa? ---')
print(G[['etapa_id', 'perfil en fw5', 'en el mapa (15+)']].to_string(index=False))
print('\nG-0 terminó (no escribió nada). Pega la salida completa en el chat del reporte.')


In [ ]:
# [G-1 v1.0] Catálogo de rasgos para elegir la ficha del retrato (SOLO LECTURA: no escribe nada)
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Insumo para fijar la ficha por componente (S14-01, S14-02)
# Junta en una sola tabla, por rasgo, lo que ya está calculado en salidas/fw5 (nada se recalcula):
#   confiabilidad y reloj (R5 / R6r-2: *_confiabilidad.csv) · regla C de implantación (*_implantacion.csv) ·
#   dueño en el mapa de dos ejes: cuadrante, firmeza, eje técnico, eje club y sus versiones sin Elo (C-1: c_mapa.csv;
#   para r9, C-1b: c1b_mapa.csv).
# Rasgos con confiabilidad < 0.5 no están en el mapa: se listan aparte (solo podrían mostrarse como "no se puede leer").
# Orden: celda 1 (Drive) → G-1. Tarda segundos. Pega la salida completa en el chat del reporte.
# =====================================================================
import os
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8', 'r9']
sal = lambda p: 'r5' if p == 'r4' else p
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 40)

# ---------- Confiabilidad, reloj y regla C ----------
CF, IM = [], []
for p in PREFIJOS:
    rc, ri = os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'), os.path.join(SAL5, f'{sal(p)}_implantacion.csv')
    if not os.path.exists(rc):
        print(f'  AVISO: falta {os.path.basename(rc)}'); continue
    c = pd.read_csv(rc); c['rasgo'] = p + ':' + c['metrica']; c['prefijo'] = p
    CF.append(c[['rasgo', 'prefijo', 'conf'] + [k for k in ['reloj'] if k in c.columns]])
    if os.path.exists(ri):
        im = pd.read_csv(ri)
        rC = im.groupby('metrica').apply(lambda g: bool(((g.p_valor < 0.05).sum() >= 2) and (g.cociente > 1).all()))
        IM.append(pd.DataFrame({'rasgo': p + ':' + rC.index, 'regla_C': rC.values}))
CF = pd.concat(CF, ignore_index=True)
IM = pd.concat(IM, ignore_index=True) if IM else pd.DataFrame(columns=['rasgo', 'regla_C'])

# ---------- Dueño (mapa de dos ejes) ----------
cols = ['cuadrante', 'firmeza', 'tec', 'club', 'tec_sin_elo', 'club_sin_elo']
M = pd.read_csv(os.path.join(SAL5, 'c_mapa.csv'), index_col=0)
rb = os.path.join(SAL5, 'c1b_mapa.csv')
if os.path.exists(rb):
    Mb = pd.read_csv(rb, index_col=0)
    M = pd.concat([M, Mb[Mb.index.str.startswith('r9:')]])
    print(f'Mapa: c_mapa.csv ({(~M.index.str.startswith("r9:")).sum()} rasgos) + r9 de c1b_mapa.csv ({M.index.str.startswith("r9:").sum()})')
else:
    print('AVISO: falta c1b_mapa.csv; r9 queda sin dueño en este catálogo')
M = M[[c for c in cols if c in M.columns]]

T = CF.merge(IM, on='rasgo', how='left').merge(M, left_on='rasgo', right_index=True, how='left')
T['regla_C'] = T['regla_C'].map({True: 'sí', False: 'no'}).fillna('—')
num = [c for c in ['conf', 'reloj', 'tec', 'club', 'tec_sin_elo', 'club_sin_elo'] if c in T.columns]
T[num] = T[num].astype(float).round(2)
if 'reloj' in T.columns: T['reloj'] = T['reloj'].round(0)

print(f'\nRasgos: {len(T)} · con confiabilidad ≥ 0.5: {(T.conf >= 0.5).sum()} · en el mapa: {T.cuadrante.notna().sum()} '
      f'(deben ser 65 de C-1 + 11 de r9 = 76)')
print('Chequeo: rasgos del mapa sin confiabilidad (debe ser 0):', len(set(M.index) - set(CF.rasgo)))

print('\n--- (1) Rasgos que se pueden leer (confiabilidad ≥ 0.5), por prefijo ---')
print('    reloj = partidos para leer el rasgo con confiabilidad 0.7 · regla_C = el técnico nuevo lo instala al llegar')
for p, g in T[T.conf >= 0.5].groupby('prefijo', sort=False):
    print(f'\n[{p}]')
    print(g.drop(columns='prefijo').sort_values('conf', ascending=False).to_string(index=False))

print('\n--- (2) Rasgos que NO se pueden leer (confiabilidad < 0.5) ---')
print(T[T.conf < 0.5][['rasgo', 'conf']].sort_values('conf', ascending=False).to_string(index=False))

print('\n--- (3) Resumen por dueño (rasgos legibles) ---')
print(pd.crosstab(T.loc[T.conf >= 0.5, 'cuadrante'].fillna('fuera del mapa'), T.loc[T.conf >= 0.5, 'firmeza'].fillna('—')).to_string())
print('\nG-1 terminó (no escribió nada).')

In [ ]:
# [G-2 v1.0] Sensibilidad del mapa de dos ejes: "sin Elo" con el NIVEL HEREDADO (no circular) · solo lectura
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Decisión S14-11 (Arturo aprueba 4a)
# Objeción que responde: el "sin Elo" de C-1 quita a cada perfil lo que explica el Elo MEDIO DE LA ETAPA, y ese Elo sale de los
#   resultados del propio técnico: podría estar quitando parte del efecto del técnico (sobrecontrol).
# Qué hace (copia literal de C-1b, secciones 0-2, sin compatibilidad ni escritura):
#   (A) REPRODUCCIÓN: recalcula el "sin Elo" con el Elo medio de la etapa y lo compara con c1b_mapa.csv (diferencia < 1e-6).
#   (B) H2 (principal): quita a cada perfil lo que explica el NIVEL HEREDADO = Elo medio del club en sus últimos 18 partidos
#       antes de que llegara el técnico (mínimo 10; misma ventana que S12-05). Etapas sin ventana → fuera de esta versión.
#   (C) H1 (sensibilidad): Elo del club en el primer partido de la etapa (cubre todas las etapas).
#   (D) Resumen por dueño (etiquetas de C-1/C-1b): eje técnico de lo que viaja y eje club de lo que depende de la plantilla,
#       crudo · sin Elo de la etapa · sin nivel heredado (H2, H1); y el detalle de los rasgos de la ficha, con IC para H2.
# Criterio (propuesto por Claude, aprobado por Arturo ANTES de correr; ver decisiones_entrega_s14.md):
#   "se sostiene" si con H2 (i) el eje técnico medio de lo que viaja queda ≥ 0.6 y (ii) el eje club medio de lo que depende de la
#   plantilla cae al menos un tercio frente al crudo. (i) sí y (ii) no → lo que viaja es robusto, pero que "plantilla = nivel" se
#   apoya en el Elo de la etapa y en L1, no en el nivel heredado (se redacta así).
# Orden: celda 1 → C-0 (todas las pruebas PASAN) → G-2. Tarda ~2-5 min. No escribe nada.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 (claude/codigo/sesion12/C_0_prueba.py) y confirma que todas sus pruebas PASAN'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); FW1 = os.path.join(DP, 'fw1'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8', 'r9']
ETAPA_MIN, VENTANA, VENT_MIN, CONF_MIN = 15, 18, 10, 0.5
B, PLAC = 500, 100
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 120)
t0 = time.time()
sal = lambda p: 'r5' if p == 'r4' else p

# ---------- 0. Perfiles y confiabilidad (copia de C-1b, sin residuos: no se calcula compatibilidad) ----------
CONT, CF = [], []
for p in PREFIJOS:
    c = pd.read_parquet(os.path.join(FW5, f'{sal(p)}_perfiles_contraidos.parquet'))
    f = pd.read_csv(os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'))
    for k, cols in [('contraidos', ['metrica', 'etapa_id', 'perfil_r4']), ('confiabilidad', ['metrica', 'conf', 'tau'])]:
        d = c if k == 'contraidos' else f
        assert all(x in d.columns for x in cols), f'Faltan columnas en {p} {k}: avisar a Claude'
    CONT.append(c.assign(rasgo=lambda d: p + ':' + d.metrica)); CF.append(f.assign(rasgo=lambda d: p + ':' + d.metrica, prefijo=p))
CONT = pd.concat(CONT, ignore_index=True); CF = pd.concat(CF, ignore_index=True).set_index('rasgo')
RASGOS = [r for r in CF.index if CF.loc[r, 'conf'] >= CONF_MIN and CF.loc[r, 'tau'] > 0]

with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max')).reset_index()
ETAPAS = ET.etapa_id.tolist(); IX = {e: i for i, e in enumerate(ETAPAS)}
tau = CF.loc[RASGOS, 'tau'].values.astype(float); conf = CF.loc[RASGOS, 'conf'].values.astype(float)
ZR = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_r4').reindex(index=ETAPAS, columns=RASGOS).astype(float) / tau).values
print(f'Rasgos en el mapa: {len(RASGOS)} (deben ser 76) · etapas: {len(ETAPAS)}')

# ---------- 1. Parejas (copia literal de C-1b) ----------
L = ET[ET.partidos >= ETAPA_MIN]
pt, ct = [], []
for tec, d in L.groupby('tecnico'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix)):
        for j in range(i + 1, len(ix)):
            if ET.team[ix[i]] != ET.team[ix[j]]: pt.append((ix[i], ix[j])); ct.append(tec)
pc, cc = [], []
for team, d in L.sort_values('ini').groupby('team'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix) - 1):
        if ET.tecnico[ix[i]] != ET.tecnico[ix[i + 1]]: pc.append((ix[i], ix[i + 1])); cc.append(team)
li = [IX[e] for e in L.etapa_id]
pool = [(a, b) for k, a in enumerate(li) for b in li[k + 1:] if ET.tecnico[a] != ET.tecnico[b] and ET.team[a] != ET.team[b]]
pt, ct, pc, cc, pool = np.array(pt), np.array(ct), np.array(pc), np.array(cc), np.array(pool)
print(f'Parejas: viajeros {len(pt)} de {len(set(ct))} técnicos (deben ser 29 de 20) · relevos {len(pc)} en {len(set(cc))} clubes (69 en 18)')

# ---------- 2. Tres medidas de nivel por etapa ----------
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'), columns=['match_id', 'team', 'elo_pre'])
EP = ptp.merge(elo, on=['match_id', 'team'], how='left')
nivel_etapa = EP.groupby('etapa_id')['elo_pre'].mean().reindex(ETAPAS).values                    # el de C-1 (circular)
h1 = EP.sort_values('fecha').groupby('etapa_id')['elo_pre'].first().reindex(ETAPAS).values       # Elo al llegar
h2 = np.full(len(ETAPAS), np.nan)                                                                  # nivel heredado (ventana de 18)
for i, r_ in enumerate(ET.itertuples(index=False)):
    v = EP[(EP.team == r_.team) & (EP.fecha < r_.ini)].sort_values('fecha').tail(VENTANA)
    if len(v) >= VENT_MIN: h2[i] = v.elo_pre.mean()
en15 = np.isin(np.arange(len(ETAPAS)), li)
print(f'Etapas de 15+ con nivel heredado (H2): {int(np.isfinite(h2[en15]).sum())} de {int(en15.sum())} · '
      f'correlación entre medidas (etapas 15+): etapa–H2 {np.corrcoef(nivel_etapa[en15 & np.isfinite(h2)], h2[en15 & np.isfinite(h2)])[0, 1]:.2f} · '
      f'etapa–H1 {np.corrcoef(nivel_etapa[en15], h1[en15])[0, 1]:.2f}')

def quitar(nivel, nan_fuera):
    """Copia del bloque 'sin Elo' de C-1: a cada rasgo le quita la recta contra el nivel. Si nan_fuera, las etapas sin nivel
    quedan fuera (NaN) en vez de conservar su perfil crudo."""
    Z = ZR.copy() if not nan_fuera else np.where(np.isfinite(nivel)[:, None], ZR, np.nan)
    for j in range(ZR.shape[1]):
        m = np.isfinite(ZR[:, j]) & np.isfinite(nivel)
        if m.sum() > 10:
            b = np.polyfit(nivel[m], ZR[m, j], 1); Z[m, j] = ZR[m, j] - np.polyval(b, nivel[m])
    return Z
cdiv = np.clip(conf, .05, None)
def ejes_pt(Z): return r_pares(Z, pt) / cdiv, r_pares(Z, pc) / cdiv

# ---------- (A) Reproducción del "sin Elo" de C-1b ----------
ref = pd.read_csv(os.path.join(SAL5, 'c1b_mapa.csv')).set_index('rasgo').reindex(RASGOS)
te, ce = ejes_pt(quitar(nivel_etapa, False))
dif = np.nanmax(np.abs(np.r_[te - ref.tec_sin_elo.values, ce - ref.club_sin_elo.values]))
tr, cr = ejes_pt(ZR)
dif0 = np.nanmax(np.abs(np.r_[tr - ref.tec.values, cr - ref.club.values]))
print(f'\n--- (A) Reproducción · ejes crudos frente a c1b_mapa: {dif0:.2e} · "sin Elo" frente a c1b_mapa: {dif:.2e} (ambos deben ser < 1e-6) ---')
assert dif0 < 1e-6 and dif < 1e-6, 'No reproduce C-1b: no se interpreta nada. Manda esta salida a Claude.'

# ---------- (B) y (C) Sin nivel heredado ----------
t2, c2 = ejes_pt(quitar(h2, True)); t1, c1 = ejes_pt(quitar(h1, False))
dueño = ref.cuadrante.map({'del técnico': 'viaja con el técnico', 'del club': 'depende de la plantilla',
                           'de nadie': 'depende de la mezcla', 'de los dos': 'de los dos'}).values
R = pd.DataFrame({'dueño': dueño, 'firmeza': ref.firmeza.values, 'conf': conf, 'tec_crudo': tr, 'tec_sin_elo_etapa': te,
                  'tec_sin_heredado': t2, 'tec_sin_elo_llegar': t1, 'club_crudo': cr, 'club_sin_elo_etapa': ce,
                  'club_sin_heredado': c2, 'club_sin_elo_llegar': c1}, index=pd.Index(RASGOS, name='rasgo'))

print('\n--- (D) Resumen por dueño (media de los rasgos) ---')
print('    tec = viaja con el técnico (parejas de viajeros) · club = se queda con el club (relevos) · ambos ÷ confiabilidad')
S = R.groupby('dueño')[[c for c in R.columns if c.startswith(('tec_', 'club_'))]].mean().round(2)
S.insert(0, 'rasgos', R.groupby('dueño').size())
print(S.T.to_string())
v, p_ = R[R.dueño == 'viaja con el técnico'], R[R.dueño == 'depende de la plantilla']
i_ok = v.tec_sin_heredado.mean() >= 0.6
ii_ok = p_.club_sin_heredado.mean() <= p_.club_crudo.mean() * 2 / 3
print(f'\nCriterio (H2): (i) eje técnico de lo que viaja {v.tec_sin_heredado.mean():.2f} (≥ 0.60) → {"CUMPLE" if i_ok else "NO CUMPLE"} · '
      f'(ii) eje club de lo de la plantilla {p_.club_crudo.mean():.2f} → {p_.club_sin_heredado.mean():.2f} '
      f'(debe quedar ≤ {p_.club_crudo.mean() * 2 / 3:.2f}) → {"CUMPLE" if ii_ok else "NO CUMPLE"}')
print('Lectura preescrita:', 'SE SOSTIENE: lo que viaja no depende del nivel, y lo de la plantilla es sobre todo nivel heredado.' if i_ok and ii_ok else
      'LO QUE VIAJA ES ROBUSTO; "plantilla = nivel" se apoya en el Elo de la etapa y en L1, no en el nivel heredado.' if i_ok else
      'NO SE SOSTIENE con el nivel heredado: revisar la redacción de la tesis con Claude antes de seguir.')

# ---------- IC con H2 para los rasgos de la ficha (remuestreo por técnico / por club, como C-1) ----------
ZH2 = quitar(h2, True)
TH = ejes(ZH2, conf, pt, ct, pc, cc, pool, B=B, P=PLAC).set_index(pd.Index(RASGOS, name='rasgo'))
FICHA = ['r4:posesion', 'r4:field_tilt', 'r4:presion_alta', 'r4:intensidad_ppda', 'r6:paso_ultimo', 'r6:toque_area', 'r6:seg_hasta_ultimo',
         'r6:pases_hasta_ultimo', 'r6c:obv_transicion_pct', 'r7u:inicio_linea_3_5', 'r7u:cambio_familia_90', 'r7c:altura_laterales',
         'r7c:libertad_posicional', 'r7c:portero_libero', 'r8:zonales', 'r8:pct_marca', 'r8:def_area_360', 'r8:pegados_portero',
         'r8:envio_segundo_palo', 'r9:espacio_conc', 'r9:rompe_lineas_conc', 'r9:dist_defensor', 'r9:adelante_contra', 'r9:altura_presion']
FICHA = [f for f in FICHA if f in TH.index]
D = R.loc[FICHA, ['dueño', 'firmeza', 'tec_crudo', 'tec_sin_elo_etapa', 'club_crudo', 'club_sin_elo_etapa']].join(
    TH.loc[FICHA, ['tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi']].add_suffix('_H2'))
print(f'\n--- Detalle: rasgos de la ficha (H2 con IC 95%, {B} remuestreos) ---')
print(D.round(2).to_string())
print(f'\nPlacebo con H2 (debe ser ≈ 0): media {TH.placebo_media.mean():+.3f} · ruido de un rasgo sin efecto ±{2 * TH.placebo_de.median():.2f}')
print(f'\nG-2 terminó en {time.time() - t0:.0f} s (no escribió nada).')

In [ ]:
# [G-3 v1.1] Exportación de AGREGADOS para el reporte HTML → salidas/fw5/reporte_datos.json (no toca nada más)
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Decisiones S14-01 a S14-16 · v1.1: agrega la tabla de arquetipos (R7c-9)
# Regla de datos del reto: solo agregados (perfiles de etapa, tablas de liga, resúmenes). Freno: cualquier tabla con columna
#   match_id, index o id de evento se descarta y se avisa. Nada por partido sale de Drive.
# Qué exporta:
#   (1) rasgos: confiabilidad, τ, σ por partido, reloj, regla C, dueño (C-1 / C-1b) con ejes, IC y sin Elo; media de la liga
#       en unidades naturales cuando el archivo de tramos lo permite.
#   (2) etapas: las 9 del selector (Jardine, Larcamón, Ortiz), América de Solari, y las 3 de referencia de E1 (anónimas en el
#       HTML): perfil contraído en unidades naturales, banda de 95% y partidos con dato, por rasgo.
#   (3) cadena de relevos: antecesor, ventana del club, nivel heredado / al llegar / final (Elo), puntos y diferencia de goles,
#       xG a favor y en contra por partido (contexto, no rasgo); perfil del club antes de llegar (zc), del técnico en su etapa
#       previa (zt) y de la etapa (zn), por rasgo (copia literal de perfil_club de C-1b).
#   (4) tablas ya calculadas (familias, validación, duración, L1, K1, R8h, M1, B7, Almada en vivo...): se copian tal cual.
# Orden: celda 1 → C-0 (todas las pruebas PASAN) → G-3. Tarda ~5-10 min (lee los residuos, como C-1b).
# Después: descargar salidas/fw5/reporte_datos.json de Drive y subirlo al chat del reporte junto con la salida de esta celda.
# =====================================================================
import os, json, glob, time
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 (claude/codigo/sesion12/C_0_prueba.py) y confirma que todas sus pruebas PASAN'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5, FW1 = os.path.join(DP, 'fw5'), os.path.join(DP, 'fw1')
SAL5, SAL6 = os.path.join(RAIZ, 'salidas', 'fw5'), os.path.join(RAIZ, 'salidas', 'fw6')
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8', 'r9']
ETAPA_MIN, VENTANA, VENT_MIN, CONF_MIN = 15, 18, 10, 0.5
SELECTOR = {'Jardine': 'Jardine', 'Larcamón': 'Larcam', 'Ortiz': 'Fernando Ortiz'}
E1_REF = {'A': 'Reynoso', 'B': 'Ferretti', 'C': 'Almada'}            # perfiles de E1: en el HTML NO se nombran (salvo C al final)
PROHIBIDAS = {'match_id', 'index', 'id', 'event_id', 'player_id'}
sal = lambda p: 'r5' if p == 'r4' else p
t0 = time.time(); avisos = []
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 120)

def limpio(v):
    if isinstance(v, (np.floating, float)): return None if not np.isfinite(v) else round(float(v), 4)
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.bool_,)): return bool(v)
    if isinstance(v, (pd.Timestamp,)): return str(v.date())
    return v
def registros(df):
    return [{k: limpio(v) for k, v in r.items()} for r in df.to_dict('records')]

# ---------- (1) Rasgos ----------
CONT, CF, TAUE, RES, S2, TRM = [], [], {}, {}, {}, {}
for p in PREFIJOS:
    c = pd.read_parquet(os.path.join(FW5, f'{sal(p)}_perfiles_contraidos.parquet'))
    f = pd.read_csv(os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'))
    pe = pd.read_parquet(os.path.join(FW5, f'{p}_perfiles_etapa.parquet'))
    rs = pd.read_parquet(os.path.join(FW5, f'{p}_residuos.parquet'), columns=['metrica', 'match_id', 'team', 'e', 'den', 'res'])
    CONT.append(c.assign(rasgo=lambda d: p + ':' + d.metrica)); CF.append(f.assign(rasgo=lambda d: p + ':' + d.metrica, prefijo=p))
    TAUE.update({p + ':' + m: v for m, v in pe.groupby('metrica')['tau_estado'].first().items()})
    S2.update({p + ':' + m: float((g.den * g.res ** 2).sum() / (len(g) - 19)) for m, g in rs.groupby('metrica')})   # copia de C-1b
    RES[p] = {t: g for t, g in rs.groupby('team')}
    # media de la liga en unidades naturales: tramos con num/den si existen (si no, queda vacía y se avisa)
    rt = os.path.join(FW5, f'{p}_tramos.parquet')
    if os.path.exists(rt):
        tr = pd.read_parquet(rt)
        if {'metrica', 'num', 'den'} <= set(tr.columns):
            g = tr.groupby('metrica')[['num', 'den']].sum(); TRM.update({p + ':' + m: float(v) for m, v in (g.num / g.den).items()})
        else: avisos.append(f'{p}_tramos sin num/den (columnas: {list(tr.columns)[:12]})')
    else: avisos.append(f'falta {p}_tramos.parquet en fw5')
CONT = pd.concat(CONT, ignore_index=True); CF = pd.concat(CF, ignore_index=True).set_index('rasgo')
for c_ in ['conf', 'tau', 'sigma_partido']: CF[c_] = CF[c_].astype(float)
RASGOS = [r for r in CF.index if CF.loc[r, 'conf'] >= CONF_MIN and CF.loc[r, 'tau'] > 0]
TODOS = CF.index.tolist()

im = []
for p in PREFIJOS:
    ri = os.path.join(SAL5, f'{sal(p)}_implantacion.csv')
    if os.path.exists(ri):
        d = pd.read_csv(ri)
        for m, g in d.groupby('metrica'): im.append((p + ':' + m, bool(((g.p_valor < 0.05).sum() >= 2) and (g.cociente > 1).all())))
IMP = dict(im)
M1 = pd.read_csv(os.path.join(SAL5, 'c_mapa.csv')).set_index('rasgo')
Mb = pd.read_csv(os.path.join(SAL5, 'c1b_mapa.csv')).set_index('rasgo')
MAPA = pd.concat([M1, Mb[Mb.index.str.startswith('r9:')]])
cols_mapa = [c for c in ['tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi', 'dif_lo', 'dif_hi', 'cuadrante', 'firmeza',
                         'tec_sin_elo', 'club_sin_elo', 'corr_elo', 'placebo_de'] if c in MAPA.columns]
rasgos_out = {}
for r in TODOS:
    d = {'prefijo': r.split(':')[0], 'metrica': r.split(':', 1)[1], 'conf': CF.loc[r, 'conf'], 'tau': CF.loc[r, 'tau'],
         'sigma_partido': CF.loc[r, 'sigma_partido'], 'reloj': CF.loc[r, 'reloj'] if 'reloj' in CF.columns else None,
         'regla_C': IMP.get(r), 'liga': TRM.get(r), 'en_mapa': r in MAPA.index}
    if r in MAPA.index: d.update({c: MAPA.loc[r, c] for c in cols_mapa})
    rasgos_out[r] = {k: limpio(v) for k, v in d.items()}
print(f'(1) Rasgos: {len(TODOS)} · legibles (≥ {CONF_MIN}) {len(RASGOS)} · en el mapa {int(sum(v["en_mapa"] for v in rasgos_out.values()))} (76) · '
      f'con media de liga {sum(v["liga"] is not None for v in rasgos_out.values())}')

# ---------- Etapas (copia literal de C-00 / C-1b) ----------
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max'), rol=('rol', lambda s: s.mode().iat[0])).reset_index()
ETI = ET.set_index('etapa_id')
etapa_de = ptp.set_index(['match_id', 'team'])['etapa_id']
tau_v = CF.loc[RASGOS, 'tau'].values

# selección de etapas
SEL = {}
for nombre, patron in SELECTOR.items():
    for e in ET[ET.tecnico.astype(str).str.contains(patron, case=False)].sort_values('ini').itertuples(index=False):
        SEL[e.etapa_id] = nombre
am = ET[ET.team.str.startswith('Am')].sort_values('ini')
solari = am[am.tecnico.str.contains('Solari', case=False)]
EXTRA = {e: 'América (contexto)' for e in solari.etapa_id}
rk = pd.read_csv(os.path.join(SAL5, 'c_ranking_america.csv'))
E1 = {}
for k, patron in E1_REF.items():
    h = rk[rk.tecnico.astype(str).str.contains(patron, case=False)]
    if len(h): E1[h.etapa.iloc[0]] = k
    else: avisos.append(f'E1: no se encontró {patron} en c_ranking_america')
print(f'(2) Etapas del selector: {len(SEL)} (deben ser 9) → {list(SEL)}')
print(f'    Contexto: {list(EXTRA)} · referencias de E1: {E1}')

# partidos con dato por rasgo y etapa (desde los residuos: mismo conteo que usa la contracción)
def n_por_etapa(etapas):
    out = {}
    for r in RASGOS:
        p, m = r.split(':', 1)
        for e in etapas:
            g = RES[p].get(ETI.loc[e, 'team'])
            if g is None: out[(r, e)] = 0; continue
            ids = set(ptp.loc[ptp.etapa_id == e, 'match_id'])
            out[(r, e)] = int(g.loc[(g.metrica == m) & g.match_id.isin(ids), 'match_id'].nunique())
    return out
ETAPAS_OUT = list(SEL) + list(EXTRA) + list(E1)
NN = n_por_etapa(ETAPAS_OUT)
CONTp = CONT.set_index(['rasgo', 'etapa_id'])
def perfil_etapa(e):
    filas = {}
    for r in RASGOS:
        if (r, e) not in CONTp.index: continue
        x = CONTp.loc[(r, e)]
        n = NN.get((r, e), 0); tau_, sg = CF.loc[r, 'tau'], CF.loc[r, 'sigma_partido']
        k = tau_ ** 2 / (tau_ ** 2 + sg ** 2 / n) if n > 0 else np.nan
        sd = np.sqrt(k) * sg / np.sqrt(n) if n > 0 else np.nan              # sd posterior del perfil contraído
        filas[r] = {k_: limpio(v) for k_, v in {'perfil': x['perfil_mostrado'], 'perfil_sin_contraer': x['perfil_r4'],
                                                'lo': x['perfil_mostrado'] - 1.96 * sd, 'hi': x['perfil_mostrado'] + 1.96 * sd,
                                                'z': x['perfil_mostrado'] / tau_, 'n': n}.items()}
    return filas

# ---------- (3) Cadena de relevos y contexto ----------
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'), columns=['match_id', 'team', 'elo_pre', 'elo_post'])
par = pd.read_parquet(os.path.join(FW1, 'elo_partido_e2.parquet'), columns=['match_id', 'local', 'visita', 'gl', 'gv'])
xg = pd.read_parquet(os.path.join(FW1, 'elo_xg_partido.parquet'), columns=['match_id', 'xg_local', 'xg_visita'])
pr = par.merge(xg, on='match_id', how='left')
res = pd.concat([pr.assign(team=pr.local, gf=pr.gl, gc=pr.gv, xgf=pr.xg_local, xgc=pr.xg_visita),
                 pr.assign(team=pr.visita, gf=pr.gv, gc=pr.gl, xgf=pr.xg_visita, xgc=pr.xg_local)])
res['pts'] = np.select([res.gf > res.gc, res.gf == res.gc], [3, 1], 0)
EP = ptp[['match_id', 'team', 'fecha', 'etapa_id', 'tec_p']].merge(elo, on=['match_id', 'team'], how='left') \
        .merge(res[['match_id', 'team', 'gf', 'gc', 'xgf', 'xgc', 'pts']], on=['match_id', 'team'], how='left')
print(f'(3) Cruce: equipo-partidos {len(EP)} · sin Elo {int(EP.elo_pre.isna().sum())} · sin marcador {int(EP.pts.isna().sum())} · '
      f'sin xG {int(EP.xgf.isna().sum())}')
ELO_ET = EP.groupby('etapa_id').elo_pre.mean(); ref15 = ELO_ET[ELO_ET.index.isin(ET.loc[ET.partidos >= ETAPA_MIN, 'etapa_id'])]
pct = lambda x: None if not np.isfinite(x) else round(100 * float((ref15 < x).mean()), 0)

# copia literal de C-1b (sección 3): perfil del club en una ventana de partidos
MIN_ = pd.read_parquet(os.path.join(FW5, 'r4_minutos_v21.parquet'))
W3 = MIN_.assign(e3=np.sign(MIN_.e5)).groupby('e3')['minutos'].sum(); W3v = (W3 / W3.sum()).reindex([-1, 0, 1]).fillna(0).values
def perfil_club(team, partidos, contraido=True):
    z = np.full(len(RASGOS), np.nan)
    for j, r in enumerate(RASGOS):
        p, m = r.split(':', 1); g = RES[p].get(team)
        if g is None: continue
        g = g[(g.metrica == m) & g.match_id.isin(partidos)]
        n = g.match_id.nunique()
        if n == 0: continue
        pf = perfil_ventana(g.e.values, g.den.values, g.res.values, S2[r], TAUE[r], W3v)
        z[j] = (contraer(pf, n, CF.loc[r, 'tau'], CF.loc[r, 'sigma_partido']) if contraido else pf) / CF.loc[r, 'tau']
    return z
ZM = CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_mostrado').reindex(columns=RASGOS).astype(float) / tau_v
# coherencia (buena práctica 13): ventana = etapa completa sin contraer reproduce perfil_r4
ZR = CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_r4').reindex(columns=RASGOS).astype(float) / tau_v
e_ = list(SEL)[0]
dmax = np.nanmax(np.abs(perfil_club(ETI.loc[e_, 'team'], set(ptp.loc[ptp.etapa_id == e_, 'match_id']), contraido=False) - ZR.loc[e_].values))
print(f'    Coherencia de perfil_club con C-1b (etapa completa {e_}): diferencia máxima {dmax:.2e} z (≈ 0; < 0.01 aceptable)')
if dmax > 0.01: avisos.append(f'perfil_club no reproduce perfil_r4 ({dmax:.3f} z): revisar antes de usar zc')

cadena = {}
for e in list(SEL) + list(EXTRA):
    r_ = ETI.loc[e]; d = EP[EP.etapa_id == e].sort_values('fecha')
    ven = EP[(EP.team == r_.team) & (EP.fecha < r_.ini)].sort_values('fecha').tail(VENTANA)
    prev = ET[(ET.team == r_.team) & (ET.fin < r_.ini)].sort_values('ini')
    ant = prev.iloc[-1] if len(prev) else None
    previa = ET[(ET.tecnico == r_.tecnico) & (ET.team != r_.team) & (ET.fin < r_.ini) & (ET.partidos >= ETAPA_MIN)].sort_values('fin')
    siguiente = ET[(ET.tecnico == r_.tecnico) & (ET.team != r_.team) & (ET.ini > r_.fin)].sort_values('ini')
    ok_v = len(ven) >= VENT_MIN
    zc = perfil_club(r_.team, set(ven.match_id)) if ok_v else np.full(len(RASGOS), np.nan)
    zt = ZM.loc[previa.etapa_id.iloc[-1]].values if len(previa) else np.full(len(RASGOS), np.nan)
    zn = ZM.loc[e].values
    cadena[e] = {k: limpio(v) for k, v in {
        'tecnico_grupo': SEL.get(e, EXTRA.get(e)), 'tecnico': r_.tecnico, 'club': r_.team, 'partidos': r_.partidos, 'rol': r_.rol,
        'del': r_.ini, 'al': r_.fin, 'antecesor': None if ant is None else ant.tecnico, 'antecesor_partidos': None if ant is None else ant.partidos,
        'antecesor_rol': None if ant is None else ant.rol, 'ventana_partidos': len(ven), 'ventana_valida': ok_v,
        'ventana_tecnicos': ', '.join(f'{t} ({n})' for t, n in ven.tec_p.value_counts().items()),
        'etapa_previa': previa.etapa_id.iloc[-1] if len(previa) else None, 'etapa_siguiente': siguiente.etapa_id.iloc[0] if len(siguiente) else None,
        'elo_heredado': ven.elo_pre.mean() if ok_v else np.nan, 'elo_al_llegar': d.elo_pre.iloc[0], 'elo_final': d.elo_post.iloc[-1],
        'elo_medio_etapa': d.elo_pre.mean(), 'pct_heredado': pct(ven.elo_pre.mean()) if ok_v else None, 'pct_final': pct(d.elo_post.iloc[-1]),
        'pts_por_partido': d.pts.mean(), 'dg_por_partido': (d.gf - d.gc).mean(), 'xg_favor_pp': d.xgf.mean(), 'xg_contra_pp': d.xgc.mean(),
        'pts_ventana': ven.pts.mean() if ok_v else np.nan}.items()}
    cadena[e]['rasgos'] = {r: {'zc': limpio(zc[j]), 'zt': limpio(zt[j]), 'zn': limpio(zn[j])} for j, r in enumerate(RASGOS)}
ref_liga = {'elo_mediana_15': limpio(ref15.median()), 'elo_p10_15': limpio(ref15.quantile(.1)), 'elo_p90_15': limpio(ref15.quantile(.9)),
            'xg_favor_pp_liga': limpio(EP.xgf.mean()), 'pts_pp_liga': limpio(EP.pts.mean()), 'etapas_15': int(len(ref15))}

etapas_out = {}
for e in ETAPAS_OUT:
    etapas_out[e] = {'grupo': SEL.get(e) or EXTRA.get(e) or f'E1-{E1[e]}', 'partidos': int(ETI.loc[e, 'partidos']),
                     'del': str(ETI.loc[e, 'ini'].date()), 'al': str(ETI.loc[e, 'fin'].date()), 'rasgos': perfil_etapa(e)}
    if e in E1: etapas_out[e].update({'club': None, 'tecnico': None})          # anónimas en el archivo (E1 a ciegas)
    else: etapas_out[e].update({'club': ETI.loc[e, 'team'], 'tecnico': ETI.loc[e, 'tecnico']})
E1_out = {E1[e]: {'etapa_key': e} for e in E1}
for k_ in E1_out:
    h = rk[rk.etapa == E1_out[k_]['etapa_key']]
    E1_out[k_].update({c: limpio(h[c].iloc[0]) for c in ['compat', 'distancia', 'choques'] if c in h.columns})

# ---------- (4) Tablas ya calculadas (copia tal cual, con freno de datos) ----------
TABLAS = {'familias': 'f_familias*', 'mapa_familias': 'f_mapa_familias*', 'familias_sin_elo': 'a2_mapa_familias_sin_elo*',
          'llegadas': 'c_llegadas*', 'ranking_america': 'c_ranking_america*', 'ensayo': 'c_ensayo_*', 'preregistro': 'c_preregistro_almada*',
          'duracion': 'd1_duracion_etapas*', 'duracion_nac': 'd1b_duracion_nacionalidad*', 'l1_alineacion': 'l1_alineacion*',
          'l1_relevos': 'l1_relevos*', 'concordancia': 'k_concordancia*', 'concordancia_tasas': 'k1b_concordancia_tasas*',
          'aereo': 'r8h_resultados*', 'disputa_liga': 'm1_mapa_liga*', 'disputa_etapas': 'm1_etapas*', 'disputa_america': 'm1_america*',
          'jardine_salida': 'b7_jardine_bloques*', 'r9_coeficientes': 'r9_coeficientes*', 'r9_implantacion': 'r9_implantacion*',
          'r5_implantacion': 'r5_implantacion*', 'arquetipos': 'r7c_arquetipos*'}
tablas_out = {}
def cargar(ruta):
    return pd.read_csv(ruta) if ruta.endswith('.csv') else pd.read_parquet(ruta)
for k, patron in TABLAS.items():
    for ruta in sorted(glob.glob(os.path.join(SAL5, patron))):
        if not ruta.endswith(('.csv', '.parquet')): continue
        df = cargar(ruta)
        clave = k if len([x for x in glob.glob(os.path.join(SAL5, patron)) if x.endswith(('.csv', '.parquet'))]) == 1 else os.path.splitext(os.path.basename(ruta))[0]
        if PROHIBIDAS & {c.lower() for c in df.columns}: avisos.append(f'DESCARTADA por tener columnas por partido/evento: {os.path.basename(ruta)}'); continue
        if len(df) > 5000: avisos.append(f'DESCARTADA por tamaño ({len(df)} filas): {os.path.basename(ruta)}'); continue
        tablas_out[clave] = registros(df)
for ruta in sorted(glob.glob(os.path.join(SAL6, 'v1_*'))):                  # Almada en vivo (V-1)
    if not ruta.endswith(('.csv', '.parquet')): continue
    df = cargar(ruta)
    if PROHIBIDAS & {c.lower() for c in df.columns}: avisos.append(f'DESCARTADA (fw6): {os.path.basename(ruta)}'); continue
    if len(df) > 5000: avisos.append(f'DESCARTADA por tamaño (fw6): {os.path.basename(ruta)}'); continue
    tablas_out['almada_' + os.path.splitext(os.path.basename(ruta))[0]] = registros(df)
for k, patron in TABLAS.items():
    if not glob.glob(os.path.join(SAL5, patron)): avisos.append(f'no se encontró {patron} en salidas/fw5')

# ---------- Guardar ----------
OUT = {'version': 'G-3 v1.1', 'generado': time.strftime('%Y-%m-%d %H:%M'), 'temporadas': TEMPS, 'rasgos': rasgos_out,
       'etapas': etapas_out, 'cadena': cadena, 'referencia_liga': ref_liga, 'e1': E1_out, 'tablas': tablas_out}
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
ruta_out = os.path.join(SAL5, 'reporte_datos.json')
with open(ruta_out, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False)

# ---------- Chequeos para pegar en el chat ----------
print(f'\n--- Chequeos ---')
print(f'Archivo: {ruta_out} · {os.path.getsize(ruta_out) / 1e6:.2f} MB · tablas copiadas: {len(tablas_out)} → {sorted(tablas_out)}')
print('\nEscala (para fijar unidades): media de liga y dispersión entre técnicos (τ) de algunos rasgos')
muestra = ['r4:posesion', 'r4:field_tilt', 'r4:intensidad_ppda', 'r6:paso_ultimo', 'r6:seg_hasta_ultimo', 'r6:pases_hasta_ultimo',
           'r7c:libertad_posicional', 'r7u:cambio_familia_90', 'r8:zonales', 'r8:pct_marca', 'r8:def_area_360', 'r9:espacio_conc', 'r9:altura']
print(pd.DataFrame({r: {'liga': rasgos_out[r]['liga'], 'tau': rasgos_out[r]['tau'], 'sigma': rasgos_out[r]['sigma_partido'],
                         'reloj': rasgos_out[r]['reloj']} for r in muestra if r in rasgos_out}).T.round(3).to_string())
ej = next(e for e in SEL if 'Jardine' in str(ETI.loc[e, 'tecnico']) and ETI.loc[e, 'team'].startswith('Am'))
print(f'\nEjemplo · {ej}: perfil contraído [banda 95%] y partidos con dato')
print(pd.DataFrame({r: etapas_out[ej]['rasgos'].get(r, {}) for r in muestra}).T[['perfil', 'lo', 'hi', 'z', 'n']].round(2).to_string())
print('\nCadena (contexto):')
print(pd.DataFrame({e: {k: cadena[e][k] for k in ['antecesor', 'etapa_previa', 'elo_heredado', 'pct_heredado', 'elo_final', 'pct_final',
                                                  'pts_por_partido', 'xg_favor_pp', 'xg_contra_pp']} for e in cadena}).T.to_string())
cub = {e: int(sum(v['zc'] is not None and v['zt'] is not None for v in cadena[e]['rasgos'].values())) for e in cadena}
print('\nRasgos con zc y zt (cobertura calculable) por etapa:', cub)
print('\nAVISOS:' if avisos else '\nSin avisos.'); [print('  ·', a) for a in avisos]
print(f'\nG-3 terminó en {time.time() - t0:.0f} s. Sube reporte_datos.json y esta salida al chat del reporte.')

In [ ]:
# R7d v1.0 · Huella DEFENSIVA por rol: dónde defiende cada puesto (eventos defensivos con jugador y ubicación)
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Decisión S14-21 (Arturo aprueba). Gemela de R7c (huella con balón).
# Por qué eventos y no 360: las fotos 360 no identifican jugadores; los eventos defensivos sí (quién, dónde y en qué puesto).
# Límite declarado: solo se ve dónde ACTÚA cada rol (presiona, entra, intercepta, recupera, bloquea, despeja), no dónde se para
#   cuando no interviene.
# Acciones (juego abierto, sin balón parado ni reanudaciones, periodos 1-2, durante posesión del RIVAL, como R4):
#   Pressure · Duel (solo Tackle si existe la columna) · Interception · Ball Recovery · Block · Clearance.
#   Coordenadas en el marco del que defiende (x = 0 su portería, 120 la del rival), como todos los eventos.
# Rasgos por equipo-partido (formato R4, una fila por partido; pasan por R6r con PREFIJO 'r7d'):
#   def_altura_delanteros · def_altura_interiores_mp · def_altura_extremos · def_altura_contenciones · def_altura_laterales ·
#   def_altura_centrales (x media de sus acciones defensivas) · def_amplitud_centrales (|y − 40| media: centrales que salen a banda)
#   · def_reparto_adelante (% de las acciones defensivas del equipo hechas por delanteros, extremos, interiores y mediapuntas)
#   · def_medios_ultima_linea (% de las acciones defensivas de contenciones e interiores a < 30 yd de su portería y en el carril
#     central: medios que bajan a defender la última línea; idea de Arturo: el interior que se incrusta entre centrales).
# Orden: celda 1 → R7d-0 → R7d-r0 → R6r-1 v1.1 y R6r-2 v1.1 SIN CAMBIOS (claude/codigo/sesion8/R6r_receta_confiabilidad.py)
#        → R7d-r9 → C-0 → R7d-9 (mapa de dos ejes y exportación). Tarda ~10-25 min en total.
# =====================================================================

#[R7d-0 v1.0] Acciones defensivas con puesto y rasgos r7d ------------------------------------------------------
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); DIR_EV = os.path.join(DP, 'liga', 'eventos')
REHACER = False
CARRIL = (80 / 3, 160 / 3); X_ULT = 30
TIPOS = ['Pressure', 'Duel', 'Interception', 'Ball Recovery', 'Block', 'Clearance']
pd.set_option('display.width', 250)
t0 = time.time()
GRUPO = {1: 'POR', 3: 'CEN', 4: 'CEN', 5: 'CEN', 2: 'LAT', 6: 'LAT', 7: 'LAT', 8: 'LAT', 9: 'MCD', 10: 'MCD', 11: 'MCD',
         13: 'INT', 14: 'INT', 15: 'INT', 18: 'MP', 19: 'MP', 20: 'MP', 12: 'EXT', 16: 'EXT', 17: 'EXT', 21: 'EXT',
         22: 'DEL', 23: 'DEL', 24: 'DEL', 25: 'DEL'}                                                   # copia de R7c
K = lambda p, c: p * 1000.0 + c
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); MIDS = set(Mm.match_id); TEMP_M = Mm.set_index('match_id')['temporada']

# ---------- 1. Acciones defensivas con jugador y ubicación ----------
ruta_a = os.path.join(FW5, 'r7d_acciones.parquet')
if os.path.exists(ruta_a) and not REHACER:
    AC = pd.read_parquet(ruta_a)
else:
    piezas = []
    for temp in TEMPS:
        t_ = temp.replace('/', '-'); r1 = os.path.join(FW1, f'eventos_{t_}.parquet'); disp = set(pq.read_schema(r1).names)
        cols = [c for c in ['match_id', 'index', 'period', 'minute', 'second', 'type.name', 'team.name', 'possession_team.name', 'loc_x', 'loc_y',
                            'fase', 'es_reanudacion', 'duel.type.name', 'player.id'] if c in disp]
        E = pd.read_parquet(r1, columns=cols); E = E[E.match_id.isin(MIDS) & E.period.isin([1, 2])]
        if 'second' not in E: E['second'] = 30
        tn = E['type.name']
        m = (E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool) & E.loc_x.notna() & E.loc_y.notna() & tn.isin(TIPOS)
        if 'possession_team.name' in E: m &= E['possession_team.name'] != E['team.name']
        if 'duel.type.name' in E: m &= ~((tn == 'Duel') & (E['duel.type.name'] != 'Tackle'))
        E = E[m]
        fuente = 'R1'
        if 'player.id' not in E:                         # los eventos defensivos no están en la caché de R6b: se leen los crudos
            xs = [pd.read_parquet(os.path.join(DIR_EV, f'{mm}.parquet'), columns=['index', 'player.id']).assign(match_id=mm)
                  for mm in E.match_id.unique() if os.path.exists(os.path.join(DIR_EV, f'{mm}.parquet'))]
            E = E.merge(pd.concat(xs), on=['match_id', 'index'], how='inner'); fuente = 'eventos crudos'
        piezas.append(pd.DataFrame({'match_id': E.match_id.values, 'team': E['team.name'].values, 'player_id': E['player.id'].values,
                                    'tipo': E['type.name'].values, 'k': K(E.period, E.minute + E.second / 60).values,
                                    'x': E.loc_x.values, 'y': E.loc_y.values}))
        print(f'  {temp}: {E.match_id.nunique()} partidos · {len(E):,} acciones defensivas · jugador desde {fuente} ({time.time() - t0:.0f} s)')
    AC = pd.concat(piezas, ignore_index=True).dropna(subset=['player_id']); AC['player_id'] = AC.player_id.astype('int64')
    TRA = pd.read_parquet(os.path.join(FW5, 'r7_tramos.parquet'), columns=['match_id', 'player_id', 'position_id', 'from_clk', 'from_period', 'to_clk', 'to_period'])
    TRA = TRA[TRA.from_period.isin([1, 2])].copy()                                                 # copia de R7c
    TRA['k0'] = K(TRA['from_period'], TRA['from_clk']); TRA['k1'] = np.where(TRA['to_clk'].notna(), K(TRA['to_period'].fillna(2), TRA['to_clk']), np.inf)
    AC = AC.merge(TRA[['match_id', 'player_id', 'position_id', 'k0', 'k1']], on=['match_id', 'player_id'], how='inner')
    AC = AC[(AC.k >= AC.k0 - 0.02) & (AC.k < AC.k1 + 0.02)].drop_duplicates(['match_id', 'player_id', 'k', 'x', 'y', 'tipo']).drop(columns=['k0', 'k1'])
    AC.to_parquet(ruta_a, index=False)
AC['grupo'] = AC.position_id.map(GRUPO); AC['temporada'] = AC.match_id.map(TEMP_M)
print(f'--- (1) Acciones defensivas con puesto: {len(AC):,} · partidos {AC.match_id.nunique()} · sin puesto {int(AC.grupo.isna().sum())}')
print('Por tipo (%):', (100 * AC.tipo.value_counts(normalize=True)).round(1).to_dict())
print('Por grupo (%):', (100 * AC.grupo.value_counts(normalize=True)).round(1).to_dict())
print('Altura media (x, desde su portería) por grupo, liga:', AC.groupby('grupo').x.mean().round(1).to_dict())

# ---------- 2. Rasgos por equipo-partido ----------
k = ['match_id', 'team']
def media(g, col='x'): return AC[AC.grupo.isin(g)].groupby(k)[col].mean()
AC['amp'] = (AC.y - 40).abs()
tot = AC.groupby(k).size()
adel = AC[AC.grupo.isin(['DEL', 'EXT', 'INT', 'MP'])].groupby(k).size()
med = AC[AC.grupo.isin(['MCD', 'INT'])]
med_ult = med[(med.x < X_ULT) & (med.y >= CARRIL[0]) & (med.y <= CARRIL[1])].groupby(k).size()
R = pd.DataFrame({
    'def_altura_delanteros': media(['DEL']), 'def_altura_interiores_mp': media(['INT', 'MP']), 'def_altura_extremos': media(['EXT']),
    'def_altura_contenciones': media(['MCD']), 'def_altura_laterales': media(['LAT']), 'def_altura_centrales': media(['CEN']),
    'def_amplitud_centrales': media(['CEN'], 'amp'),
    'def_reparto_adelante': 100 * adel.reindex(tot.index).fillna(0) / tot,
    'def_medios_ultima_linea': 100 * med_ult.reindex(med.groupby(k).size().index).fillna(0) / med.groupby(k).size()}).reset_index()
RASGOS = [c for c in R.columns if c.startswith('def_')]
print('\n--- (2) Rasgos por equipo-partido (media, p10, p90 y nulos) ---')
print(R[RASGOS].describe(percentiles=[.1, .9]).T[['count', 'mean', '10%', '90%']].round(2).assign(nulos=R[RASGOS].isna().sum().values).to_string())
print('Correlaciones entre rasgos (|r| alto = miden casi lo mismo):'); print(R[RASGOS].corr().round(2).to_string())
r7c = os.path.join(FW5, 'r7c_equipo_partido.parquet')
if os.path.exists(r7c):
    C = R.merge(pd.read_parquet(r7c), on=k)
    pares = [('def_altura_laterales', 'altura_laterales'), ('def_altura_contenciones', 'altura_contenciones'),
             ('def_altura_interiores_mp', 'altura_interiores_mp'), ('def_altura_delanteros', 'altura_delanteros')]
    print('Mismo rol con y sin balón (correlación por equipo-partido; si es ~1, la huella defensiva no agrega nada):',
          {a: round(C[a].corr(C[b]), 2) for a, b in pares})
filas = [pd.DataFrame({'metrica': m, 'match_id': R.match_id, 'team': R.team, 'e5': 0, 'e5b': 0, 'cuarto': 1, 'num': R[m], 'den': 1.0}).dropna(subset=['num']) for m in RASGOS]
pd.concat(filas, ignore_index=True).to_parquet(os.path.join(FW5, 'r7d_tramos.parquet'), index=False)
R.to_parquet(os.path.join(FW5, 'r7d_equipo_partido.parquet'), index=False)
print(f'\nR7d-0 terminó en {time.time() - t0:.0f} s. Guardado: fw5/r7d_tramos, r7d_equipo_partido, r7d_acciones.')


In [ ]:
# [R7d-r0 v1.0] Carpeta espejo para R6r-1 y R6r-2 con PREFIJO = 'r7d' (copia de R7c-r0 con otro prefijo)
import os, shutil
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7D', '/content/isac_r7d/')
D_DP, D_FW1, D_FW5 = os.path.join(DRIVE, 'datos_parquet'), os.path.join(DRIVE, 'datos_parquet', 'fw1'), os.path.join(DRIVE, 'datos_parquet', 'fw5')
L_DP, L_FW = os.path.join(LOCAL, 'datos_parquet'), os.path.join(LOCAL, 'datos_parquet', 'fw1')
shutil.rmtree(LOCAL, ignore_errors=True); os.makedirs(L_FW, exist_ok=True); os.makedirs(os.path.join(LOCAL, 'salidas'), exist_ok=True)
def enlazar(src, dst):
    if os.path.exists(src): os.symlink(src, dst); return True
    return False
enl = {'partido_tecnico_periodo.parquet': enlazar(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'), os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))}
for f in ['r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet', 'r6_perfiles_etapa.parquet',
          'r6b_perfiles_etapa.parquet', 'r6c_perfiles_etapa.parquet', 'r7c_perfiles_etapa.parquet', 'r7d_tramos.parquet']:
    enl[f] = enlazar(os.path.join(D_FW5, f), os.path.join(L_FW, f))
enl['elo (fw1)'] = enlazar(os.path.join(D_FW1, 'elo_equipo_partido_e2.parquet'), os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'))
print('--- (R7d-r0) Carpeta espejo', LOCAL); print({k_: ('ok' if v else 'FALTA') for k_, v in enl.items()})
assert all(enl[k_] for k_ in ['partido_tecnico_periodo.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json',
                              'r4_perfiles_etapa.parquet', 'r7d_tramos.parquet', 'elo (fw1)']), 'Falta un insumo obligatorio: avisar a Claude'
BASE = LOCAL; PREFIJO = 'r7d'
print("Listo → corre R6r-1 v1.1 y R6r-2 v1.1 sin cambios; luego R7d-r9.")




In [ ]:
# [R7d-r9 v1.0] Copiar resultados r7d_* a Drive y regresar BASE a Drive (copia de R7c-r9)
import os, shutil
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7D', '/content/isac_r7d/')
L_FW = os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_FW5, D_SAL5 = os.path.join(DRIVE, 'datos_parquet', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw5'); os.makedirs(D_SAL5, exist_ok=True)
n = []
for src_dir, dst_dir in [(L_FW, D_FW5), (os.path.join(LOCAL, 'salidas'), D_SAL5)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if f.startswith('r7d_') and os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print('--- (R7d-r9) Copiados a Drive:', sorted(n)); BASE = DRIVE; print('BASE =', BASE)
print('Siguiente: corre C-0 (claude/codigo/sesion12/C_0_prueba.py) y después R7d-9.')

In [ ]:
# [R7d-9 v1.0] Huella defensiva por rol: mapa de dos ejes (dueño) y exportación de agregados → salidas/fw5/reporte_r7d.json
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Después de R7d-0 → R7d-r0 → R6r-1 → R6r-2 → R7d-r9 → C-0 (todas las pruebas PASAN).
# (1) Confiabilidad y reloj de los rasgos r7d (de R6r-2). (2) Mapa de dos ejes con las MISMAS parejas que C-1b (29 viajeros,
#     69 relevos) y las mismas reglas (÷ confiabilidad, corte 0.5, firmeza por IC de la diferencia), más sin Elo de la etapa y sin
#     nivel heredado (G-2). (3) Exporta, para las etapas del selector, perfil contraído, banda de 95%, partidos con dato, perfil
#     del club antes de llegar (zc), del técnico en su etapa previa (zt) y de la etapa (zn): copia de G-3. Solo agregados.
# Tarda ~2-4 min. Escribe solo salidas/fw5/r7d_mapa.csv y salidas/fw5/reporte_r7d.json.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 (claude/codigo/sesion12/C_0_prueba.py) y confirma que todas sus pruebas PASAN'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5, FW1 = os.path.join(DP, 'fw5'), os.path.join(DP, 'fw1'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
ETAPA_MIN, VENTANA, VENT_MIN, CONF_MIN, B, PLAC = 15, 18, 10, 0.5, 1000, 200
SELECTOR = {'Jardine': 'Jardine', 'Larcamón': 'Larcam', 'Fernando Ortiz': 'Fernando Ortiz'}
t0 = time.time(); pd.set_option('display.width', 250)

c = pd.read_parquet(os.path.join(FW5, 'r7d_perfiles_contraidos.parquet')); f = pd.read_csv(os.path.join(SAL5, 'r7d_confiabilidad.csv'))
pe = pd.read_parquet(os.path.join(FW5, 'r7d_perfiles_etapa.parquet'))
rs = pd.read_parquet(os.path.join(FW5, 'r7d_residuos.parquet'), columns=['metrica', 'match_id', 'team', 'e', 'den', 'res'])
tr = pd.read_parquet(os.path.join(FW5, 'r7d_tramos.parquet'))
for col_ in ['conf', 'tau', 'sigma_partido']: f[col_] = f[col_].astype(float)
f = f.set_index('metrica')
print('--- (1) Confiabilidad y reloj (partidos para leer el rasgo con confiabilidad 0.7) ---')
print(f[[x for x in ['conf', 'reloj', 'tau', 'sigma_partido'] if x in f.columns]].sort_values('conf', ascending=False).round(2).to_string())
RASGOS = [m for m in f.index if f.loc[m, 'conf'] >= CONF_MIN and f.loc[m, 'tau'] > 0]
print('Entran al mapa (confiabilidad ≥ 0.5):', RASGOS)

# ---------- Etapas y parejas (copia literal de C-1b / G-2) ----------
with open(os.path.join(FW5, 'parametros_fw1.json')) as fh:
    TEMPS = sorted(json.load(fh)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max')).reset_index()
ETAPAS = ET.etapa_id.tolist(); IX = {e: i for i, e in enumerate(ETAPAS)}; ETI = ET.set_index('etapa_id')
tau = f.loc[RASGOS, 'tau'].values; conf = f.loc[RASGOS, 'conf'].values
ZR = (c.pivot_table(index='etapa_id', columns='metrica', values='perfil_r4').reindex(index=ETAPAS, columns=RASGOS).astype(float) / tau).values
ZM = (c.pivot_table(index='etapa_id', columns='metrica', values='perfil_mostrado').reindex(index=ETAPAS, columns=RASGOS).astype(float) / tau).values
L = ET[ET.partidos >= ETAPA_MIN]
pt, ct = [], []
for tec, d in L.groupby('tecnico'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix)):
        for j in range(i + 1, len(ix)):
            if ET.team[ix[i]] != ET.team[ix[j]]: pt.append((ix[i], ix[j])); ct.append(tec)
pc, cc = [], []
for team, d in L.sort_values('ini').groupby('team'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix) - 1):
        if ET.tecnico[ix[i]] != ET.tecnico[ix[i + 1]]: pc.append((ix[i], ix[i + 1])); cc.append(team)
li = [IX[e] for e in L.etapa_id]
pool = [(a, b) for k_, a in enumerate(li) for b in li[k_ + 1:] if ET.tecnico[a] != ET.tecnico[b] and ET.team[a] != ET.team[b]]
pt, ct, pc, cc, pool = np.array(pt), np.array(ct), np.array(pc), np.array(cc), np.array(pool)
print(f'\nParejas: viajeros {len(pt)} de {len(set(ct))} técnicos (deben ser 29 de 20) · relevos {len(pc)} en {len(set(cc))} clubes (69 en 18)')

# ---------- (2) Mapa ----------
T = ejes(ZR, conf, pt, ct, pc, cc, pool, B=B, P=PLAC).set_index(pd.Index(RASGOS, name='rasgo'))
T['cuadrante'], T['firmeza'] = clasificar(T)
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'), columns=['match_id', 'team', 'elo_pre'])
EP = ptp.merge(elo, on=['match_id', 'team'], how='left')
nivel = EP.groupby('etapa_id')['elo_pre'].mean().reindex(ETAPAS).values
h2 = np.full(len(ETAPAS), np.nan)
for i, r_ in enumerate(ET.itertuples(index=False)):
    v = EP[(EP.team == r_.team) & (EP.fecha < r_.ini)].sort_values('fecha').tail(VENTANA)
    if len(v) >= VENT_MIN: h2[i] = v.elo_pre.mean()
def quitar(niv, nan_fuera):                                                      # copia de G-2
    Z = ZR.copy() if not nan_fuera else np.where(np.isfinite(niv)[:, None], ZR, np.nan)
    for j in range(ZR.shape[1]):
        m = np.isfinite(ZR[:, j]) & np.isfinite(niv)
        if m.sum() > 10:
            b = np.polyfit(niv[m], ZR[m, j], 1); Z[m, j] = ZR[m, j] - np.polyval(b, niv[m])
    return Z
cd = np.clip(conf, .05, None)
for nom, Z in [('sin_elo', quitar(nivel, False)), ('sin_heredado', quitar(h2, True))]:
    T[f'tec_{nom}'] = r_pares(Z, pt) / cd; T[f'club_{nom}'] = r_pares(Z, pc) / cd
T['conf'] = conf; T['reloj'] = f.loc[RASGOS, 'reloj'].values if 'reloj' in f.columns else np.nan
V = ['conf', 'reloj', 'tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi', 'cuadrante', 'firmeza', 'tec_sin_elo', 'club_sin_elo', 'tec_sin_heredado', 'club_sin_heredado']
print(f'\n--- (2) Mapa de dos ejes · placebo (≈ 0): {T.placebo_media.mean():+.3f} · ruido de un rasgo sin efecto ±{2 * T.placebo_de.median():.2f} ---')
print(T[V].round(2).to_string())
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
T.to_csv(os.path.join(SAL5, 'r7d_mapa.csv'))

# ---------- (3) Exportación (copia de G-3) ----------
S2 = {m: float((g.den * g.res ** 2).sum() / (len(g) - 19)) for m, g in rs.groupby('metrica')}
TAUE = pe.groupby('metrica')['tau_estado'].first().to_dict(); RES = {t: g for t, g in rs.groupby('team')}
MIN_ = pd.read_parquet(os.path.join(FW5, 'r4_minutos_v21.parquet'))
W3 = MIN_.assign(e3=np.sign(MIN_.e5)).groupby('e3')['minutos'].sum(); W3v = (W3 / W3.sum()).reindex([-1, 0, 1]).fillna(0).values
def perfil_club(team, partidos, contraido=True):
    z = np.full(len(RASGOS), np.nan)
    for j, m_ in enumerate(RASGOS):
        g = RES.get(team)
        if g is None: continue
        g = g[(g.metrica == m_) & g.match_id.isin(partidos)]; n = g.match_id.nunique()
        if n == 0: continue
        pf = perfil_ventana(g.e.values, g.den.values, g.res.values, S2[m_], TAUE[m_], W3v)
        z[j] = (contraer(pf, n, f.loc[m_, 'tau'], f.loc[m_, 'sigma_partido']) if contraido else pf) / f.loc[m_, 'tau']
    return z
e0 = L.etapa_id.iloc[0]
dmax = np.nanmax(np.abs(perfil_club(ETI.loc[e0, 'team'], set(ptp.loc[ptp.etapa_id == e0, 'match_id']), contraido=False) - ZR[IX[e0]]))
print(f'\nCoherencia de perfil_club (etapa completa {e0}): diferencia máxima {dmax:.2e} z (≈ 0)')
SEL = [e for p in SELECTOR.values() for e in ET[ET.tecnico.astype(str).str.contains(p, case=False)].etapa_id]
liga = tr.groupby('metrica').apply(lambda g: (g.num * g.den).sum() / g.den.sum()).to_dict()
cp = c.set_index(['metrica', 'etapa_id'])
lim = lambda v: None if v is None or not np.isfinite(float(v)) else round(float(v), 4)
out = {'rasgos': {}, 'etapas': {}}
for j, m_ in enumerate(RASGOS):
    out['rasgos'][m_] = {k_: (lim(T.loc[m_, k_]) if isinstance(T.loc[m_, k_], (int, float, np.integer, np.floating)) else str(T.loc[m_, k_])) for k_ in V}
    out['rasgos'][m_].update({'tau': lim(f.loc[m_, 'tau']), 'sigma_partido': lim(f.loc[m_, 'sigma_partido']), 'liga': lim(liga.get(m_))})
for e in SEL:
    r_ = ETI.loc[e]; ids = set(ptp.loc[ptp.etapa_id == e, 'match_id'])
    ven = ptp[(ptp.team == r_.team) & (ptp.fecha < r_.ini)].sort_values('fecha').tail(VENTANA)
    previa = ET[(ET.tecnico == r_.tecnico) & (ET.team != r_.team) & (ET.fin < r_.ini) & (ET.partidos >= ETAPA_MIN)].sort_values('fin')
    zc = perfil_club(r_.team, set(ven.match_id)) if len(ven) >= VENT_MIN else np.full(len(RASGOS), np.nan)
    zt = ZM[IX[previa.etapa_id.iloc[-1]]] if len(previa) else np.full(len(RASGOS), np.nan)
    fil = {}
    for j, m_ in enumerate(RASGOS):
        if (m_, e) not in cp.index: continue
        x = cp.loc[(m_, e)]; g = RES.get(r_.team); n = int(g.loc[(g.metrica == m_) & g.match_id.isin(ids), 'match_id'].nunique()) if g is not None else 0
        ta, sg = f.loc[m_, 'tau'], f.loc[m_, 'sigma_partido']; kk = ta ** 2 / (ta ** 2 + sg ** 2 / n) if n else np.nan
        sd = np.sqrt(kk) * sg / np.sqrt(n) if n else np.nan
        fil[m_] = {'perfil': lim(x['perfil_mostrado']), 'lo': lim(x['perfil_mostrado'] - 1.96 * sd), 'hi': lim(x['perfil_mostrado'] + 1.96 * sd),
                   'z': lim(x['perfil_mostrado'] / ta), 'n': n, 'zc': lim(zc[j]), 'zt': lim(zt[j]), 'zn': lim(ZM[IX[e], j])}
    out['etapas'][e] = fil
ruta = os.path.join(SAL5, 'reporte_r7d.json')
with open(ruta, 'w', encoding='utf-8') as fh: json.dump(out, fh, ensure_ascii=False)
print(f'Etapas exportadas ({len(SEL)}): {SEL}')
ej = next((e for e in SEL if 'Jardine' in str(ETI.loc[e, 'tecnico']) and e.startswith('Am')), SEL[0])
print(f'\nEjemplo · {ej} (valor = liga + perfil):')
print(pd.DataFrame({m_: {'liga': out['rasgos'][m_]['liga'], 'perfil': out['etapas'][ej].get(m_, {}).get('perfil'), 'n': out['etapas'][ej].get(m_, {}).get('n')}
                    for m_ in RASGOS}).T.round(2).to_string())
print(f'\nR7d-9 terminó en {time.time() - t0:.0f} s. Guardado: salidas/fw5/r7d_mapa.csv y reporte_r7d.json. Sube el JSON y esta salida al chat.')

In [ ]:
# [G-4 v1.0] Los 11 en la cancha (con y sin balón) y percentiles entre técnicos → salidas/fw5/reporte_posiciones.json
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Pedido de Arturo: ver a los 11 en la cancha y leer cada rasgo más rápido.
# (1) Posición media por puesto exacto (los 25 puestos de StatsBomb: lateral derecho, central izquierdo, mediapunta...) para las
#     etapas del selector y la media de la liga por puesto. Con balón: acciones de R7c (r7c_acciones). Sin balón: acciones
#     defensivas de R7d (r7d_acciones; correr R7d-0 antes). Por etapa se toman los 11 puestos con más acciones (1 portero) y solo
#     puestos con 30 acciones o más. Es DESCRIPTIVO (sin ajuste por contexto ni contracción): los rasgos medidos de la ficha
#     siguen siendo los de R7c/R7d con su confiabilidad.
# (2) Percentil de cada etapa del selector entre las etapas de 15+ partidos, en cada rasgo (perfil contraído): "más que el X% de
#     los técnicos". Una cifra fácil de leer para la ficha.
# Orden: celda 1 → (R7d-0 ya corrido) → G-4. Tarda ~1-3 min. Solo agregados; no escribe nada más.
# =====================================================================
import os, json, glob, time
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SELECTOR = ['Jardine', 'Larcam', 'Fernando Ortiz']; MIN_ACC, ETAPA_MIN = 30, 15
NOMBRE_PUESTO = {1: 'Portero', 2: 'Lateral derecho', 3: 'Central derecho', 4: 'Central', 5: 'Central izquierdo', 6: 'Lateral izquierdo',
                 7: 'Carrilero derecho', 8: 'Carrilero izquierdo', 9: 'Contención derecha', 10: 'Contención', 11: 'Contención izquierda',
                 12: 'Volante derecho', 13: 'Interior derecho', 14: 'Medio centro', 15: 'Interior izquierdo', 16: 'Volante izquierdo',
                 17: 'Extremo derecho', 18: 'Mediapunta derecho', 19: 'Mediapunta', 20: 'Mediapunta izquierdo', 21: 'Extremo izquierdo',
                 22: 'Delantero derecho', 23: 'Delantero centro', 24: 'Delantero izquierdo', 25: 'Segundo delantero'}
t0 = time.time(); pd.set_option('display.width', 250)
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique')).reset_index()
SEL = [e for p in SELECTOR for e in ET[ET.tecnico.astype(str).str.contains(p, case=False)].etapa_id]
print(f'Etapas del selector: {len(SEL)} (deben ser 9) → {SEL}')

def once(ruta, nombre):
    if not os.path.exists(ruta): print(f'  AVISO: falta {os.path.basename(ruta)} ({nombre}); se omite'); return None, None
    A = pd.read_parquet(ruta, columns=['match_id', 'team', 'position_id', 'x', 'y']).merge(ptp[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'])
    A['position_id'] = A.position_id.astype(int)
    liga = A.groupby('position_id').agg(x=('x', 'mean'), y=('y', 'mean'), n=('x', 'size'))
    out = {}
    for e in SEL:
        g = A[A.etapa_id == e].groupby('position_id').agg(x=('x', 'mean'), y=('y', 'mean'), n=('x', 'size'))
        g = g[g.n >= MIN_ACC].sort_values('n', ascending=False)
        por = g[g.index == 1].head(1); campo = g[g.index != 1].head(10)
        sel = pd.concat([por, campo])
        out[e] = [{'puesto': int(p), 'nombre': NOMBRE_PUESTO.get(int(p), str(p)), 'x': round(float(r.x), 2), 'y': round(float(r.y), 2),
                   'n': int(r.n), 'lx': round(float(liga.loc[p, 'x']), 2), 'ly': round(float(liga.loc[p, 'y']), 2),
                   'pct_acciones': round(100 * float(r.n) / float(g.n.sum()), 1)} for p, r in sel.iterrows()]
    print(f'  {nombre}: {len(A):,} acciones · puestos en la liga {len(liga)}')
    return out, {int(p): {'nombre': NOMBRE_PUESTO.get(int(p), str(p)), 'x': round(float(r.x), 2), 'y': round(float(r.y), 2), 'n': int(r.n)} for p, r in liga.iterrows()}
con, liga_con = once(os.path.join(FW5, 'r7c_acciones.parquet'), 'con balón (R7c)')
sin, liga_sin = once(os.path.join(FW5, 'r7d_acciones.parquet'), 'sin balón (R7d)')

# ---------- (2) Percentiles entre etapas de 15+ ----------
L15 = set(ET.loc[ET.partidos >= ETAPA_MIN, 'etapa_id'])
pct = {e: {} for e in SEL}
for ruta in sorted(glob.glob(os.path.join(FW5, '*_perfiles_contraidos.parquet'))):
    pre = os.path.basename(ruta).split('_')[0]; pre = 'r4' if pre == 'r5' else pre
    c = pd.read_parquet(ruta, columns=['metrica', 'etapa_id', 'perfil_mostrado'])
    for m, g in c.groupby('metrica'):
        ref = g[g.etapa_id.isin(L15)].perfil_mostrado.dropna().values
        if len(ref) < 20: continue
        for e in SEL:
            v = g.loc[g.etapa_id == e, 'perfil_mostrado']
            if len(v) and np.isfinite(v.iloc[0]):
                pct[e][f'{pre}:{m}'] = round(100 * ((ref < v.iloc[0]).mean() + 0.5 * (ref == v.iloc[0]).mean()), 0)
print(f'Percentiles: {len(next(iter(pct.values())))} rasgos por etapa (referencia: {len(L15)} etapas de 15+)')

OUT = {'version': 'G-4 v1.0', 'con_balon': con, 'sin_balon': sin, 'liga_con': liga_con, 'liga_sin': liga_sin, 'percentiles': pct}
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
ruta = os.path.join(SAL5, 'reporte_posiciones.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False)
ej = next((e for e in SEL if e.startswith('Am') and 'Jardine' in ET.set_index('etapa_id').loc[e, 'tecnico']), SEL[0])
for nom, d in [('con balón', con), ('sin balón', sin)]:
    if d: print(f'\n{ej} · {nom}:'); print(pd.DataFrame(d[ej])[['nombre', 'x', 'lx', 'y', 'ly', 'n', 'pct_acciones']].to_string(index=False))
print(f'\nPercentiles de ejemplo ({ej}):', {k: v for k, v in list(pct[ej].items())[:8]})
print(f'\nG-4 terminó en {time.time() - t0:.0f} s · {os.path.getsize(ruta) / 1e3:.0f} KB. Sube reporte_posiciones.json y esta salida al chat.')

In [ ]:
# [G-5 v1.0] Dónde juega cada etapa (por tercios) y medias de liga de los rasgos de R4 → salidas/fw5/reporte_territorio.json
# =====================================================================
# Sesión 14 · 2026-10-21 · Chat del reporte · Pedido de Arturo: la capa de espacio (posiciones medias) no refleja el territorio;
#   un equipo con field tilt alto juega mucho en el tercio rival, pero sus posiciones medias mezclan todas sus acciones.
# (1) Territorio real por tercios: % de las acciones con balón (R7c) y de las acciones defensivas (R7d) de cada etapa que ocurren
#     en el tercio propio, el medio y el rival; y la media de la liga. Descriptivo, solo agregados.
# (2) Medias de liga en unidades naturales de los rasgos de R4 (intensidad de presión, presión alta, saque de meta corto, etc.),
#     que faltaban en G-3 porque no hay r4_tramos en fw5: se buscan en fw1 y fw5 (tramos con num/den).
# Orden: celda 1 → G-5. Tarda ~1 min. No escribe nada más.
# =====================================================================
import os, json
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SELECTOR = ['Jardine', 'Larcam', 'Fernando Ortiz']
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(tecnico=('tec_p', 'first')).reset_index()
SEL = [e for p in SELECTOR for e in ET[ET.tecnico.astype(str).str.contains(p, case=False)].etapa_id]
print('Etapas:', len(SEL), SEL)

def tercios(ruta, nombre):
    if not os.path.exists(ruta): print('  AVISO: falta', os.path.basename(ruta)); return None
    A = pd.read_parquet(ruta, columns=['match_id', 'team', 'x']).merge(ptp[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'])
    A['t'] = pd.cut(A.x, [-1, 40, 80, 121], labels=['propio', 'medio', 'rival'])
    liga = (100 * A.t.value_counts(normalize=True)).round(1).to_dict()
    out = {'liga': liga}
    for e in SEL:
        out[e] = (100 * A.loc[A.etapa_id == e, 't'].value_counts(normalize=True)).round(1).to_dict()
    print(f'  {nombre}: liga {liga} · ejemplo {SEL[0]} {out[SEL[0]]}')
    return out
TER = {'con_balon': tercios(os.path.join(FW5, 'r7c_acciones.parquet'), 'con balón'),
       'sin_balon': tercios(os.path.join(FW5, 'r7d_acciones.parquet'), 'sin balón')}

LIGA = {}
for d in [FW5, FW1]:
    r = os.path.join(d, 'r4_tramos.parquet')
    if os.path.exists(r) and not LIGA:
        t = pd.read_parquet(r)
        print('  r4_tramos en', os.path.basename(d), '· columnas:', list(t.columns)[:12])
        if {'metrica', 'num', 'den'} <= set(t.columns):
            g = t.groupby('metrica')[['num', 'den']].sum(); LIGA = {m: round(float(v), 4) for m, v in (g.num / g.den).items()}
if not LIGA: print('  AVISO: no se encontró r4_tramos con num/den; las medias de liga de R4 quedan vacías')
print('Medias de liga de R4:', LIGA)

OUT = {'version': 'G-5 v1.0', 'tercios': TER, 'liga_r4': LIGA}
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
with open(os.path.join(SAL5, 'reporte_territorio.json'), 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False)
print('\nG-5 terminó. Sube salidas/fw5/reporte_territorio.json y esta salida al chat.')

In [ ]:
# [G-6 v1.0] Lugar en la tabla de Elo al llegar y al irse (SOLO LECTURA: no escribe nada)
# =====================================================================
# Sesión 15 · 2026-10-21 · Chat del reporte · Corrección 8 de Arturo: el nivel se muestra como lugar entre los 18 clubes, no como percentil.
# Definición (se propone a Arturo; ver el chat):
#   - "Al llegar": orden de los clubes de la temporada por su Elo vigente el día ANTES del primer partido del técnico
#     (Elo vigente = Elo después del último partido jugado antes de esa fecha; si el club aún no jugaba, su Elo de arranque).
#   - "Al irse": el mismo orden el día DESPUÉS de su último partido.
#   - Clubes que cuentan: los que juegan la temporada de esa fecha (18). Lugar 1 = Elo más alto.
#   El Elo es el mismo de siempre (E-2, fw1): K 20 · H 75 · regreso de 25% hacia 1,500 en verano. Ese regreso no cambia el orden.
# Orden: celda 1 (Drive) → G-6. Tarda segundos. Pega TODA la salida en el chat del reporte.
# =====================================================================
import os, json
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1')
ETAPAS = ['Atlético San Luis · 2', 'América · 3', 'Puebla · 1', 'León · 4', 'Necaxa · 9', 'Cruz Azul · 9',
          'América · 2', 'Monterrey · 3', 'Santos Laguna · 6', 'América · 1']

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['fecha'] = pd.to_datetime(ptp['fecha']); ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))
elo['fecha'] = pd.to_datetime(elo['fecha'])

def tabla(fecha, temporada):
    """Elo vigente de cada club de la temporada en `fecha` (antes de jugar ese día) y su lugar."""
    clubes = elo.loc[elo.temporada == temporada, 'team'].unique()
    vals = {}
    for c in clubes:
        e = elo[elo.team == c].sort_values('fecha')
        antes = e[e.fecha < fecha]
        vals[c] = antes.elo_post.iloc[-1] if len(antes) else e.elo_pre.iloc[0]
    s = pd.Series(vals).sort_values(ascending=False)
    return s, {c: i + 1 for i, c in enumerate(s.index)}

out, filas = {}, []
for et in ETAPAS:
    d = ptp[ptp.etapa_id == et]
    if not len(d): print('AVISO: no encontré', et); continue
    club, ini, fin = d.team.iloc[0], d.fecha.min(), d.fecha.max()
    t_ini = elo.loc[(elo.team == club) & (elo.fecha >= ini), 'temporada'].iloc[0]
    t_fin = elo.loc[(elo.team == club) & (elo.fecha <= fin), 'temporada'].iloc[-1]
    s0, r0 = tabla(ini, t_ini)
    s1, r1 = tabla(fin + pd.Timedelta(days=1), t_fin)
    out[et] = {'lugar_llega': r0[club], 'n_llega': len(s0), 'lugar_va': r1[club], 'n_va': len(s1),
               'elo_llega': round(float(s0[club]), 1), 'elo_va': round(float(s1[club]), 1),
               'lider_llega': s0.index[0], 'lider_va': s1.index[0]}
    filas.append({'etapa': et, 'llega': ini.date(), 'se va': fin.date(), 'temporada (llega/va)': f'{t_ini} / {t_fin}',
                  'lugar al llegar': f"{r0[club]} de {len(s0)}", 'lugar al irse': f"{r1[club]} de {len(s1)}",
                  'Elo llegar/irse': f"{s0[club]:.0f} / {s1[club]:.0f}"})
print('--- G-6 · Lugar en la tabla de Elo (contexto; no mide el efecto del técnico) ---')
print(pd.DataFrame(filas).to_string(index=False))
print('\nChequeo: todas las tablas deben tener 18 clubes →', sorted({v['n_llega'] for v in out.values()} | {v['n_va'] for v in out.values()}))
print('\nPEGA ESTO (va a datos_retrato.json):')
print(json.dumps(out, ensure_ascii=False))

In [ ]:
# [G-7 v1.0] Cómo cambia el estilo según el marcador (SOLO LECTURA de fw5/fw1; escribe un JSON chico en salidas/fw5)
# =====================================================================
# Sesión 15 · 2026-10-21 · Chat del reporte · Mejora 3 de la revisión: "¿cómo cambian los patrones según el contexto?" (componente
#   Consistencia y variabilidad del reto). Hoy el marcador solo se usa para AJUSTAR; aquí se MUESTRA.
# Qué hace:
#   (1) Busca los archivos de tramos (formato R4: metrica, match_id, team, e5, num, den) en fw5 y, si falta alguno, en fw1.
#       Imprime cuáles encontró y sus columnas (si algo no cuadra, Claude lo revisa con esta salida).
#   (2) Para cada rasgo y cada estado del marcador (perdiendo, empatando, ganando = signo de e5, marcador efectivo de R4):
#       valor de la liga = Σnum / Σden en ese estado; valor de cada etapa del selector igual, más partidos y peso (den) con ese estado.
#       Es el valor CRUDO (sin ajuste por rival ni contracción): se compara contra la liga en el MISMO estado.
#   (3) Imprime una tabla de control (América · 3 y Jardine en San Luis) y escribe salidas/fw5/reporte_contexto.json.
# Orden: celda 1 (Drive) → G-7. Tarda ~1-2 min. Pega TODA la salida en el chat del reporte y sube el JSON (o pégalo si es corto).
# =====================================================================
import os, re, json, glob
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
ETAPAS = ['Atlético San Luis · 2', 'América · 3', 'Puebla · 1', 'León · 4', 'Necaxa · 9', 'Cruz Azul · 9',
          'América · 2', 'Monterrey · 3', 'Santos Laguna · 6', 'América · 1']
EST = {-1: 'perdiendo', 0: 'empatando', 1: 'ganando'}

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
MAP = ptp[['match_id', 'team', 'etapa_id']].drop_duplicates(['match_id', 'team'])

# ---------- (1) Archivos de tramos ----------
cand = {}
for d in [FW5, FW1]:                                              # fw5 tiene prioridad (cinco temporadas)
    for f in sorted(glob.glob(os.path.join(d, '*tramos*.parquet'))):
        pre = os.path.basename(f).split('_')[0]
        if not re.fullmatch(r'r\d+[a-z]*', pre): continue
        if pre in cand and os.path.dirname(cand[pre]) == FW5: continue
        if pre in cand and 'v21' not in os.path.basename(f) and 'v21' in os.path.basename(cand[pre]): pass
        cand.setdefault(pre, f)
print('--- (1) Archivos de tramos encontrados ---')
TR = []
for pre, f in sorted(cand.items()):
    t = pd.read_parquet(f)
    ok = {'metrica', 'match_id', 'team', 'e5', 'num', 'den'} <= set(t.columns)
    print(f'  {pre:5s} {os.path.relpath(f, DP)} · filas {len(t):,} · {"OK" if ok else "FALTAN COLUMNAS: " + str(list(t.columns)[:10])}')
    if ok:
        t = t[['metrica', 'match_id', 'team', 'e5', 'num', 'den']].copy(); t['id'] = pre + ':' + t['metrica'].astype(str); TR.append(t)
assert TR, 'No encontré tramos con columnas metrica/match_id/team/e5/num/den: pega esta salida a Claude'
T = pd.concat(TR, ignore_index=True)
T = T[T.den > 0].copy(); T['s'] = np.sign(T['e5']).astype(int)
T = T.merge(MAP, on=['match_id', 'team'], how='left')
print(f'Filas con etapa: {T.etapa_id.notna().mean():.1%} (debe ser ~100%) · rasgos: {T.id.nunique()}')

# ---------- (2) Liga y etapas por estado ----------
g = T.groupby(['id', 's'])[['num', 'den']].sum(); LIGA = (g.num / g.den).unstack('s')
S = T[T.etapa_id.isin(ETAPAS)]
ge = S.groupby(['etapa_id', 'id', 's']).agg(num=('num', 'sum'), den=('den', 'sum'), partidos=('match_id', 'nunique'))
ge['v'] = ge.num / ge.den
OUT = {'version': 'G-7 v1.0', 'liga': {i: {str(s): round(float(v), 4) for s, v in r.dropna().items()} for i, r in LIGA.iterrows()}, 'etapas': {}}
for (e, i), d in ge.groupby(level=[0, 1]):
    OUT['etapas'].setdefault(e, {})[i] = {str(s): [round(float(r.v), 4), int(r.partidos), round(float(r.den), 1)] for (_, _, s), r in d.iterrows()}

# ---------- (3) Control ----------
CTRL = ['r4:posesion', 'r4:intensidad_ppda', 'r4:presion_alta', 'r4:pase_largo', 'r6:seg_hasta_ultimo', 'r7u:cambio_familia_90', 'r9:altura']
print('\n--- (2) Liga por estado del marcador (valor crudo) ---')
print(LIGA.reindex([c for c in CTRL if c in LIGA.index]).rename(columns=EST).round(2).to_string())
for e in ['América · 3', 'Atlético San Luis · 2']:
    if e not in OUT['etapas']: print('AVISO: sin datos para', e); continue
    print(f'\n{e}: valor (partidos con ese estado)')
    for c in CTRL:
        if c in OUT['etapas'][e]:
            print(f'  {c:24s}', ' · '.join(f"{EST[int(s)]} {v[0]:.2f} ({v[1]})" for s, v in sorted(OUT['etapas'][e][c].items(), key=lambda x: int(x[0]))))
print('\nEtapas con datos:', sorted(OUT['etapas']), '· faltan:', [e for e in ETAPAS if e not in OUT['etapas']])
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
ruta = os.path.join(SAL5, 'reporte_contexto.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False, separators=(',', ':'))
print(f'\nG-7 terminó. Escribió {ruta} ({os.path.getsize(ruta) / 1024:.0f} KB). Súbelo al chat (o pégalo si pesa menos de ~60 KB).')

In [ ]:
# [G-8 v1.0] Emparejamiento aéreo por HOPS frente a por estatura, en partidos reales (SOLO LECTURA de fw5 y liga/lineups; escribe un JSON chico)
# =====================================================================
# Sesión 15 · 2026-10-21 · Chat del reporte · Punto 12 de Arturo: E-3 con datos reales en vez de inventados.
# Qué hace (unidad = un lado de un partido: el equipo que defiende el córner contra los cabeceadores del rival):
#   (0) Reconstruye la capacidad aérea igual que R8h-1 (HOPS jugador-temporada con ≥ 10 duelos + estatura de lineups) y chequea
#       la correlación HOPS-estatura por equipo-partido contra R8m (0.35).
#   (1) En cada partido, los 5 jugadores MÁS ALTOS de cada equipo que jugaron (con HOPS y estatura) son "los que van al área".
#       Defensa: marca por estatura (el más alto al más alto) contra marca por HOPS (método húngaro que minimiza la suma de
#       desventajas al cuadrado). Desventaja de un duelo = max(0, HOPS del atacante − HOPS de su marca).
#       Ataque: si el rival marca por estatura, ¿cuál es nuestro duelo más favorable? ¿Es el más alto, el de mejor HOPS u otro?
#   (2) Resumen de la liga con IC 95% (remuestreo de partidos, 500, semilla 2026) y 3 ejemplos al azar, anónimos.
# Límite: no sabemos quién entró al área ni quién marcó a quién (el 360 no identifica jugadores). Es un ejercicio de asignación
#   sobre jugadores reales, no una prueba de que marcar por HOPS evita goles.
# Orden: celda 1 (Drive) → G-8. Tarda ~2-4 min. Pega TODA la salida y sube salidas/fw5/reporte_hops.json (o pégalo, es chico).
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5, LIGA = os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); SEMILLA = 2026; t0 = time.time()

# ---------- (0) Capacidad aérea: igual que R8h-1 ----------
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
HJ = pd.read_parquet(os.path.join(FW5, 'r8m_hops_jugador.parquet'))
JP = pd.read_parquet(os.path.join(FW5, 'r7_jugador_partido.parquet'))
pcol = next(c for c in ['player_id', 'player.id', 'jugador_id', 'jugador'] if c in JP.columns)
mcol = next(c for c in ['minutos', 'min', 'minutes'] if c in JP.columns)
JP = JP[JP[mcol] > 0].rename(columns={pcol: 'player_id'})
JP['temporada'] = JP.match_id.map(loc['temporada'])
JP = JP.merge(HJ[HJ.duelos >= 10][['player_id', 'temporada', 'hops']], on=['player_id', 'temporada'], how='left')
alt = []
for mid in Mm.match_id:
    r = os.path.join(LIGA, 'lineups', f'{mid}.parquet')
    if os.path.exists(r): alt.append(pd.read_parquet(r, columns=['player_id', 'player_height']))
ALT = pd.concat(alt).dropna().query('player_height > 0').drop_duplicates('player_id')
JP = JP.merge(ALT, on='player_id', how='left')
top5 = lambda s: s.dropna().nlargest(5).mean() if s.notna().sum() >= 5 else np.nan
TM = JP.groupby(['match_id', 'team']).agg(h=('hops', top5), a=('player_height', top5))
r_ha = TM[['h', 'a']].corr().iloc[0, 1]
print(f'--- (0) Correlación HOPS-estatura por equipo-partido {r_ha:.2f} (R8m: 0.35) → {"REPRODUCE" if abs(r_ha - .35) <= .01 else "NO REPRODUCE: pega esta línea"}')
J = JP.dropna(subset=['hops', 'player_height'])
r_j = J.drop_duplicates(['player_id', 'temporada'])[['hops', 'player_height']].corr().iloc[0, 1]
print(f'    Correlación HOPS-estatura entre jugadores (jugador-temporada): {r_j:.2f} · rango de HOPS p5 {J.hops.quantile(.05):.1f} · p50 {J.hops.median():.1f} · p95 {J.hops.quantile(.95):.1f}')

# ---------- (1) Asignación por partido ----------
G = {k: g.nlargest(5, 'player_height')[['player_height', 'hops']].to_numpy() for k, g in J.groupby(['match_id', 'team']) if len(g) >= 5}
filas = []
for mid in Mm.match_id:
    eq = [t for (m, t) in [(mid, loc.at[mid, 'local']), (mid, loc.at[mid, 'visita'])] if (m, t) in G]
    if len(eq) < 2: continue
    for de, at in [(eq[0], eq[1]), (eq[1], eq[0])]:
        D, A = G[(mid, de)], G[(mid, at)]                          # ya vienen del más alto al más bajo
        Dh, Ah = D[:, 1], A[:, 1]
        dz_alt = np.maximum(0, Ah - Dh)                             # marca por estatura: i-ésimo más alto con i-ésimo más alto
        C = np.maximum(0, Ah[None, :] - Dh[:, None]) ** 2
        fi, co = linear_sum_assignment(C); dz_h = np.maximum(0, Ah[co] - Dh[fi])
        adv = Ah - Dh                                               # ataque: atacante j contra su marca por estatura (mismo rango)
        b = int(np.argmax(adv))
        filas.append(dict(match_id=mid, tot_alt=dz_alt.sum(), tot_h=dz_h.sum(), peor_alt=dz_alt.max(), peor_h=dz_h.max(),
                          cambia=bool((co != np.arange(5)).any()), mejor_adv=adv[b], adv_alto=adv[0], adv_mejor_hops=adv[int(np.argmax(Ah))],
                          caza_no_alto=b != 0, caza_no_mejor=b != int(np.argmax(Ah)), caza_ninguno=(b != 0) and (b != int(np.argmax(Ah)))))
F = pd.DataFrame(filas)
print(f'\n--- (1) Lados de partido analizados: {len(F):,} ({F.match_id.nunique():,} partidos)')

def resumen(d):
    return {'reduccion_total_pct': 100 * (1 - d.tot_h.sum() / d.tot_alt.sum()), 'reduccion_peor_pct': 100 * (1 - d.peor_h.sum() / d.peor_alt.sum()),
            'tot_alt': d.tot_alt.mean(), 'tot_h': d.tot_h.mean(), 'peor_alt': d.peor_alt.mean(), 'peor_h': d.peor_h.mean(),
            'pct_cambia': 100 * d.cambia.mean(), 'pct_caza_no_alto': 100 * d.caza_no_alto.mean(), 'pct_caza_no_mejor': 100 * d.caza_no_mejor.mean(),
            'pct_caza_ninguno': 100 * d.caza_ninguno.mean(), 'mejor_adv': d.mejor_adv.mean(), 'adv_alto': d.adv_alto.mean(), 'adv_mejor_hops': d.adv_mejor_hops.mean()}
rng = np.random.default_rng(SEMILLA); idx = F.groupby('match_id').indices; mids = np.array(list(idx))
est = pd.Series(resumen(F)); bs = pd.DataFrame([resumen(F.iloc[np.concatenate([idx[m] for m in rng.choice(mids, len(mids))])]) for _ in range(500)])
RES = pd.DataFrame({'valor': est, 'ic_lo': bs.quantile(.025), 'ic_hi': bs.quantile(.975)})
print(RES.round(2).to_string())
print('Lectura: reduccion_total_pct = cuánto baja la desventaja aérea total al marcar por HOPS en vez de por estatura.'
      '\n         pct_caza_ninguno = partidos en que el mejor duelo a favor no es ni el más alto ni el de mejor HOPS.')

# ---------- (2) Ejemplos al azar (anónimos) y JSON ----------
ej = []
for mid in rng.choice(F.match_id.unique(), 3, replace=False):
    eq = [loc.at[mid, 'local'], loc.at[mid, 'visita']]
    D, A = G[(mid, eq[0])], G[(mid, eq[1])]
    ej.append({'def': [[int(round(a)), round(float(h), 1)] for a, h in D], 'ata': [[int(round(a)), round(float(h), 1)] for a, h in A]})
OUT = {'version': 'G-8 v1.0', 'n_lados': int(len(F)), 'n_partidos': int(F.match_id.nunique()), 'r_jugador': round(float(r_j), 2), 'r_equipo': round(float(r_ha), 2),
       'res': {k: [round(float(v), 2) for v in RES.loc[k]] for k in RES.index}, 'ejemplos': ej}
ruta = os.path.join(SAL5, 'reporte_hops.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False, separators=(',', ':'))
print('\nEjemplo 1 (defensores / atacantes, [estatura cm, HOPS]):', ej[0])
print(f'G-8 terminó en {time.time() - t0:.0f} s. Escribió {ruta} ({os.path.getsize(ruta)} bytes). Súbelo o pégalo en el chat.')

In [ ]:
# [G-4b v1.0] Los 11 en la cancha, versión corregida: más puestos para que el reporte junte carriles ANTES de elegir a los 10 de campo
# =====================================================================
# Sesión 15 · 2026-10-21 · Punto 4 de Arturo (León de Larcamón: el extremo derecho "aparece" con balón y desaparece sin balón).
# Causa: G-4 elegía los 10 puestos de campo con más acciones ANTES de juntar lateral con carrilero (y volante con extremo). Un técnico
#   que alterna línea de 4 y de 3/5 gasta 4 de esos 10 lugares en dos carriles, y se queda sin un jugador real. No era el portero.
# Arreglo: se exportan hasta 16 puestos de campo (con 30 acciones o más); el reporte junta carriles y luego toma los 10 con más acciones.
# Solo cambia la selección; posiciones, liga y percentiles quedan iguales (no se recalculan). Orden: celda 1 → G-4b. ~1-2 min.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SELECTOR = ['Jardine', 'Larcam', 'Fernando Ortiz']; MIN_ACC, N_CAMPO = 30, 16
NOMBRE_PUESTO = {1: 'Portero', 2: 'Lateral derecho', 3: 'Central derecho', 4: 'Central', 5: 'Central izquierdo', 6: 'Lateral izquierdo',
                 7: 'Carrilero derecho', 8: 'Carrilero izquierdo', 9: 'Contención derecha', 10: 'Contención', 11: 'Contención izquierda',
                 12: 'Volante derecho', 13: 'Interior derecho', 14: 'Medio centro', 15: 'Interior izquierdo', 16: 'Volante izquierdo',
                 17: 'Extremo derecho', 18: 'Mediapunta derecho', 19: 'Mediapunta', 20: 'Mediapunta izquierdo', 21: 'Extremo izquierdo',
                 22: 'Delantero derecho', 23: 'Delantero centro', 24: 'Delantero izquierdo', 25: 'Segundo delantero'}
t0 = time.time()
with open(os.path.join(FW5, 'parametros_fw1.json')) as f: TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(tecnico=('tec_p', 'first')).reset_index()
SEL = [e for p in SELECTOR for e in ET[ET.tecnico.astype(str).str.contains(p, case=False)].etapa_id]
print(f'Etapas del selector: {len(SEL)} (deben ser 9)')
def once(ruta):
    A = pd.read_parquet(ruta, columns=['match_id', 'team', 'position_id', 'x', 'y']).merge(ptp[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'])
    A['position_id'] = A.position_id.astype(int)
    liga = A.groupby('position_id').agg(x=('x', 'mean'), y=('y', 'mean'))
    out = {}
    for e in SEL:
        g = A[A.etapa_id == e].groupby('position_id').agg(x=('x', 'mean'), y=('y', 'mean'), n=('x', 'size'))
        g = g[g.n >= MIN_ACC].sort_values('n', ascending=False)
        sel = pd.concat([g[g.index == 1].head(1), g[g.index != 1].head(N_CAMPO)])
        out[e] = [{'puesto': int(p), 'nombre': NOMBRE_PUESTO.get(int(p), str(p)), 'x': round(float(r.x), 2), 'y': round(float(r.y), 2), 'n': int(r.n),
                   'lx': round(float(liga.loc[p, 'x']), 2), 'ly': round(float(liga.loc[p, 'y']), 2)} for p, r in sel.iterrows()]
    return out
OUT = {'version': 'G-4b v1.0', 'con_balon': once(os.path.join(FW5, 'r7c_acciones.parquet')), 'sin_balon': once(os.path.join(FW5, 'r7d_acciones.parquet'))}
for f in ['con_balon', 'sin_balon']:
    print(f'León · 4 · {f}:', [(p['nombre'], p['n']) for p in OUT[f].get('León · 4', [])])
ruta = os.path.join(SAL5, 'reporte_posiciones_v2.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False, separators=(',', ':'))
print(f'G-4b terminó en {time.time() - t0:.0f} s · {os.path.getsize(ruta) / 1e3:.0f} KB. Sube reporte_posiciones_v2.json al chat.')

In [2]:
# [G-9 v1.0] Córners con ocupación 360 REAL por etapa (SOLO LECTURA de fw5; escribe un JSON chico en salidas/fw5)
# =====================================================================
# Sesión 16 · 2026-10-21 · Chat del reporte · Escenas del córner (a favor y en contra) con datos reales en vez de posiciones ilustrativas.
# Insumos (ya existen, no se recalculan): fw5/r8p_corners (R8p: una fila por córner directo con foto 360 del saque; 20 celdas de
#   atacantes en el saque sa0-19, defensores en el saque sd0-19 y atacantes en el primer contacto ca0-19) y fw5/r8_bp (R8a: dónde fue
#   el primer contacto y de qué clase). Grilla de 20 celdas de R8p (BP-7): x 102-108 | 108-114 | 114-117 | 117-120 × y (desde la banda
#   del saque) 18-30 | 30-36 | 36-44 | 44-50 | 50-62. Todo en el marco del equipo que COBRA el córner.
# Qué hace:
#   (0) Chequeos: suma de las 20 celdas = conteos de R8a en el área (deben coincidir); media de la liga contra la del reporte (8.49).
#   (1) Asigna cada córner a la etapa del que ataca (córner a favor) y a la del que defiende (córner en contra), como G-7.
#   (2) Por etapa y para la liga: ocupación media por celda (atacantes y defensores en el saque; atacantes en el contacto, en los mismos
#       córners) y dónde remata el primer contacto (celda; 'fuera' = fuera del área) con su xG. Remates: TODOS los córners directos.
#   (3) Estabilidad: ¿la DIFERENCIA de la etapa contra la liga se repite entre dos mitades de sus partidos? (200 particiones al azar,
#       semilla 2026; corrección de Spearman-Brown, semáforo 0.7 / 0.5 de S15-09). Se usa la diferencia y no la ocupación cruda porque
#       todos los equipos llenan el área chica: la cruda se parecería siempre y daría una estabilidad inflada.
# Orden: celda 1 (Drive) → G-9. Tarda ~1-2 min. Pega TODA la salida y sube salidas/fw5/reporte_corners360.json (o pégalo).
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SEMILLA, N_PART, MIN_CORNERS = 2026, 200, 30
ETAPAS = ['Atlético San Luis · 2', 'América · 3', 'Puebla · 1', 'León · 4', 'Necaxa · 9', 'Cruz Azul · 9',
          'América · 2', 'Monterrey · 3', 'Santos Laguna · 6', 'América · 1']
t0 = time.time()

# Grilla de R8p (copia literal)
XB, YB = [102, 108, 114, 117, 120.0001], [18, 30, 36, 44, 50, 62.0001]
ZONA_CELDA = np.array(['area_chica' if xi >= 2 and 1 <= yi <= 3 else 'primer_palo' if yi <= 1 else 'punto_penal' if yi == 2 else 'segundo_palo'
                       for xi in range(4) for yi in range(5)])
def celda(px, yr):
    xi = np.digitize(px, XB) - 1; yi = np.digitize(yr, YB) - 1
    ok = (xi >= 0) & (xi <= 3) & (yi >= 0) & (yi <= 4)
    return np.where(ok, xi * 5 + yi, -1)
SA, SD, CA = [f'sa{i}' for i in range(20)], [f'sd{i}' for i in range(20)], [f'ca{i}' for i in range(20)]
ZONAS = ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']

# ---------- (0) Cargar y chequear ----------
H = pd.read_parquet(os.path.join(FW5, 'r8p_corners.parquet'))
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
BP = BP[BP.kind == 'corner'].copy()
cols_bp = [c for c in ['match_id', 'index', 'ruta', 'lado_bajo', 'contacto_x', 'contacto_y', 'contacto_clase', 's_def_area', 's_atac_area'] if c in BP]
H = H.merge(BP[cols_bp], on=['match_id', 'index'], how='left', suffixes=('', '_bp'))
print(f'--- (0) Córners directos con foto del saque: {len(H):,} · con foto del contacto {100 * H[CA[0]].notna().mean():.1f}% · por temporada '
      f'{H.temporada.value_counts().sort_index().to_dict()}')
if 's_def_area' in H:
    ig_d = (H[SD].sum(axis=1) == H.s_def_area).mean(); ig_a = (H[SA].sum(axis=1) == H.s_atac_area).mean()
    print(f'    Suma de las 20 celdas = conteo de R8a en el área: defensores {100 * ig_d:.1f}% · atacantes {100 * ig_a:.1f}% de los córners (deben ser ~100)')
print(f'    Liga, por córner: defensores en el área {H[SD].sum(axis=1).mean():.2f} (el reporte dice 8.49) · atacantes {H[SA].sum(axis=1).mean():.2f}')

# ---------- (1) Etapas (igual que G-7) ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
MAP = ptp[['match_id', 'team', 'etapa_id']].drop_duplicates(['match_id', 'team'])
H = H.merge(MAP.rename(columns={'etapa_id': 'et_of'}), on=['match_id', 'team'], how='left')
H = H.merge(MAP.rename(columns={'team': 'rival', 'etapa_id': 'et_de'}), on=['match_id', 'rival'], how='left')
print(f'--- (1) Córners con etapa del que ataca {100 * H.et_of.notna().mean():.1f}% · del que defiende {100 * H.et_de.notna().mean():.1f}% (deben ser ~100)')

# Remates del primer contacto: todos los córners directos (con y sin foto 360)
D = BP[BP.ruta == 'directo'].copy()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
D['rival'] = np.where(D.team == D.match_id.map(loc['local']), D.match_id.map(loc['visita']), D.match_id.map(loc['local']))
D = D.merge(MAP.rename(columns={'etapa_id': 'et_of'}), on=['match_id', 'team'], how='left')
D = D.merge(MAP.rename(columns={'team': 'rival', 'etapa_id': 'et_de'}), on=['match_id', 'rival'], how='left')
D['rem'] = D.contacto_clase == 'remate_atac'
yr = np.where(D.lado_bajo.astype(bool), D.contacto_y, 80 - D.contacto_y)
c = celda(D.contacto_x.to_numpy(float), yr.astype(float)); D['cel'] = np.where(c < 0, 20, c)
print(f'    Córners directos (todos): {len(D):,} · el primer contacto es remate del que cobra en {100 * D.rem.mean():.1f}% · '
      f'de esos remates, dentro del área {100 * (D.loc[D.rem, "cel"] < 20).mean():.1f}%')

# ---------- (2) Agregados ----------
def mitades(d, cols, ref, rng):
    """Estabilidad de la diferencia contra la liga entre dos mitades de partidos (media de N_PART particiones)."""
    g = d.groupby('match_id'); S = g[cols].sum().to_numpy(float); N = g.size().to_numpy(float)
    if len(N) < 8: return None, None
    rs = []
    for _ in range(N_PART):
        m = rng.permutation(len(N)) < len(N) // 2
        a = S[m].sum(0) / N[m].sum() - ref; b = S[~m].sum(0) / N[~m].sum() - ref
        if a.std() > 0 and b.std() > 0: rs.append(np.corrcoef(a, b)[0, 1])
    r = float(np.mean(rs)); return round(r, 3), round(2 * r / (1 + r), 3) if r > 0 else round(r, 3)

def remates(d):
    r = d[d.rem]
    return {'n_dir': int(len(d)), 'rem': np.bincount(r.cel, minlength=21).tolist(),
            'xg': np.round(np.bincount(r.cel, weights=r.xg1.fillna(0), minlength=21), 3).tolist()}

r3 = lambda v: np.round(np.asarray(v, float), 3).tolist()
Hm = H[H[CA[0]].notna()]
LIGA = {'of': {'n': int(len(H)), 'sa': r3(H[SA].mean()), 'n_mov': int(len(Hm)), 'sa_m': r3(Hm[SA].mean()), 'ca': r3(Hm[CA].mean()), **remates(D)},
        'de': {'n': int(len(H)), 'sd': r3(H[SD].mean()), 'sa': r3(H[SA].mean()), **remates(D)}}
ref_of = np.r_[H[SA].mean().to_numpy(), Hm[CA].mean().to_numpy()]; ref_de = H[SD].mean().to_numpy()
OUT = {'version': 'G-9 v1.0', 'grilla': {'xb': [102, 108, 114, 117, 120], 'yb': [18, 30, 36, 44, 50, 62], 'zona': ZONA_CELDA.tolist()},
       'liga': LIGA, 'etapas': {}}
rng = np.random.default_rng(SEMILLA)
filas = []
for e in ETAPAS:
    o, dd = H[H.et_of == e], H[H.et_de == e]; om = o[o[CA[0]].notna()]
    if len(o) == 0 and len(dd) == 0: print('AVISO: sin córners para', e); continue
    so = mitades(om, SA + CA, ref_of, rng); sd_ = mitades(dd, SD, ref_de, rng)
    OUT['etapas'][e] = {
        'of': {'n': int(len(o)), 'partidos': int(o.match_id.nunique()), 'sa': r3(o[SA].mean()), 'n_mov': int(len(om)), 'sa_m': r3(om[SA].mean()),
               'ca': r3(om[CA].mean()), 'sh': so, **remates(D[D.et_of == e])},
        'de': {'n': int(len(dd)), 'partidos': int(dd.match_id.nunique()), 'sd': r3(dd[SD].mean()), 'sa': r3(dd[SA].mean()), 'sh': sd_,
               **remates(D[D.et_de == e])}}
    zo = {z: o[[s for s, zz in zip(SA, ZONA_CELDA) if zz == z]].sum(axis=1).mean() - H[[s for s, zz in zip(SA, ZONA_CELDA) if zz == z]].sum(axis=1).mean() for z in ZONAS}
    zd = {z: dd[[s for s, zz in zip(SD, ZONA_CELDA) if zz == z]].sum(axis=1).mean() - H[[s for s, zz in zip(SD, ZONA_CELDA) if zz == z]].sum(axis=1).mean() for z in ZONAS}
    de_ = D[D.et_de == e]; of_ = D[D.et_of == e]
    filas.append({'etapa': e, 'part_of': o.match_id.nunique(), 'c_favor': len(o), 'c_contra': len(dd),
                  'atac_area': round(o[SA].sum(axis=1).mean(), 2), 'def_area': round(dd[SD].sum(axis=1).mean(), 2),
                  **{f'Δatac_{z[:6]}': round(v, 2) for z, v in zo.items()}, **{f'Δdef_{z[:6]}': round(v, 2) for z, v in zd.items()},
                  'rem_favor_x100': round(100 * of_.rem.mean(), 1), 'rem_contra_x100': round(100 * de_.rem.mean(), 1),
                  'estab_of': so[1] if so[0] is not None else None, 'estab_de': sd_[1] if sd_[0] is not None else None})
T = pd.DataFrame(filas).set_index('etapa')

print(f'\n--- (2) Liga: atacantes / defensores en el área al sacar {H[SA].sum(axis=1).mean():.2f} / {H[SD].sum(axis=1).mean():.2f} · remate en el primer contacto '
      f'{100 * D.rem.mean():.1f} de cada 100 córners')
print('    Atacantes por zona, saque → contacto (mismos córners):', {z[:6]: f"{Hm[[s for s, zz in zip(SA, ZONA_CELDA) if zz == z]].sum(axis=1).mean():.2f} → "
                                                                        f"{Hm[[s for s, zz in zip(CA, ZONA_CELDA) if zz == z]].sum(axis=1).mean():.2f}" for z in ZONAS})
rl = np.bincount(D.loc[D.rem, 'cel'], minlength=21); print('    Remates del primer contacto por celda (liga; filas x 102→120, columnas y 18→62; último = fuera del área):')
print('   ', rl[:20].reshape(4, 5).tolist(), '· fuera', int(rl[20]))
pd.set_option('display.width', 250)
print('\n--- (3) Por etapa (Δ = etapa − liga, jugadores por córner; estab = Spearman-Brown de la diferencia; verde ≥ 0.7, ámbar 0.5-0.7, rojo < 0.5)')
print(T.T.to_string())
poco = [e for e in T.index if T.at[e, 'c_favor'] < MIN_CORNERS or T.at[e, 'c_contra'] < MIN_CORNERS]
print('Etapas con menos de', MIN_CORNERS, 'córners con foto en algún lado:', poco or 'ninguna')
e = 'América · 3'
if e in OUT['etapas']:
    print(f'\nControl {e}: atacantes en el saque por celda (4 filas × 5 columnas) y liga:')
    print('   ', np.round(np.array(OUT['etapas'][e]['of']['sa']).reshape(4, 5), 2).tolist())
    print('   ', np.round(np.array(LIGA['of']['sa']).reshape(4, 5), 2).tolist())
ruta = os.path.join(SAL5, 'reporte_corners360.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False, separators=(',', ':'))
print(f'\nG-9 terminó en {time.time() - t0:.0f} s. Escribió {ruta} ({os.path.getsize(ruta) / 1024:.0f} KB). Súbelo o pégalo en el chat.')

--- (0) Córners directos con foto del saque: 9,081 · con foto del contacto 95.1% · por temporada {'2021/2022': 1765, '2022/2023': 1881, '2023/2024': 1872, '2024/2025': 1827, '2025/2026': 1736}
    Suma de las 20 celdas = conteo de R8a en el área: defensores 98.1% · atacantes 99.4% de los córners (deben ser ~100)
    Liga, por córner: defensores en el área 8.47 (el reporte dice 8.49) · atacantes 5.38
--- (1) Córners con etapa del que ataca 100.0% · del que defiende 100.0% (deben ser ~100)
    Córners directos (todos): 12,893 · el primer contacto es remate del que cobra en 0.0% · de esos remates, dentro del área nan%

--- (2) Liga: atacantes / defensores en el área al sacar 5.38 / 8.47 · remate en el primer contacto 0.0 de cada 100 córners
    Atacantes por zona, saque → contacto (mismos córners): {'primer': '0.59 → 1.37', 'punto_': '2.66 → 2.23', 'area_c': '1.16 → 1.13', 'segund': '0.97 → 1.24'}
    Remates del primer contacto por celda (liga; filas x 102→120, columnas y 18→62; último =

In [3]:
# [G-10 v1.0] Saque de meta y pase largo REALES por etapa: a dónde va el balón (SOLO LECTURA de fw1/fw5; escribe un JSON chico)
# =====================================================================
# Sesión 16 · 2026-10-21 · Chat del reporte · Escenas de construcción con destinos reales en vez de ilustrativos.
# Definiciones (copia literal de R4, para que la escena y el rasgo salgan de lo mismo):
#   - Saque de meta: pase con pass.type = 'Goal Kick'. Corto = termina antes de x = 40 (r4:saque_meta_corto).
#   - Pase largo: pase de juego abierto (fase ≠ 'bp' y no reanudación) de 35 yd o más (r4:pase_largo).
#   - Solo periodos 1 y 2 (D-9). Coordenadas StatsBomb: el equipo ataca hacia x = 120; y de 0 a 80 tal como viene (no se espeja:
#     el lado derecho o izquierdo es parte del estilo).
# Qué hace:
#   (0) Lee los eventos de R1 (fw1, una tabla por temporada) de los partidos de fw5 y chequea los porcentajes de la liga contra el reporte.
#   (1) Por etapa del selector y para la liga, en una grilla de 12 × 8 celdas de 10 × 10 yd:
#       saque de meta → celda donde termina y si se completó; pase largo → celda de origen, celda de destino y si se completó.
#   (2) Estabilidad: ¿la DIFERENCIA del reparto de destinos contra la liga se repite entre dos mitades de los partidos de la etapa?
#       (200 particiones, semilla 2026, Spearman-Brown; mismo criterio que G-9).
# Orden: celda 1 (Drive) → G-10. Tarda ~3-5 min (lee cinco tablas de eventos). Pega TODA la salida y sube
#   salidas/fw5/reporte_construccion.json (o pégalo si pesa poco).
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SEMILLA, N_PART, X_MED, LARGO = 2026, 200, 40, 35
ETAPAS = ['Atlético San Luis · 2', 'América · 3', 'Puebla · 1', 'León · 4', 'Necaxa · 9', 'Cruz Azul · 9',
          'América · 2', 'Monterrey · 3', 'Santos Laguna · 6', 'América · 1']
t0 = time.time()

def celda(x, y):
    """Grilla 12 × 8 (10 yd): índice = fila_x * 8 + col_y; x 0-120, y 0-80. Bordes incluidos en la última celda."""
    xi = np.clip(np.floor(np.asarray(x, float) / 10), 0, 11); yi = np.clip(np.floor(np.asarray(y, float) / 10), 0, 7)
    return (xi * 8 + yi).astype(int)

# ---------- (0) Eventos ----------
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); mids = set(Mm.match_id)
COLS = ['match_id', 'period', 'type.name', 'team.name', 'pass.type.name', 'pass.length', 'pass.outcome.name',
        'loc_x', 'loc_y', 'pass_end_x', 'pass_end_y', 'fase', 'es_reanudacion']
GK, PL = [], []
for temp in sorted(Mm.temporada.unique()):
    ruta = os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet")
    disp = set(pq.read_schema(ruta).names); falt = [c for c in COLS if c not in disp]
    assert not falt, f'A eventos_{temp} le faltan columnas {falt}: pega esta línea a Claude'
    E = pd.read_parquet(ruta, columns=COLS, filters=[('type.name', '==', 'Pass')])
    E = E[E.match_id.isin(mids) & E.period.isin([1, 2])]
    gk = E[E['pass.type.name'] == 'Goal Kick']
    abierto = (E.fase != 'bp') & ~E.es_reanudacion.fillna(False).astype(bool)
    pl = E[abierto & (E['pass.length'] >= LARGO)]
    GK.append(gk.assign(temporada=temp)); PL.append(pl.assign(temporada=temp))
    print(f'  {temp}: saques de meta {len(gk):,} · pases largos {len(pl):,} · pases de juego abierto {int(abierto.sum()):,} ({time.time() - t0:.0f} s)', flush=True)
GK = pd.concat(GK, ignore_index=True); PL = pd.concat(PL, ignore_index=True)
for d in (GK, PL):
    d['ok'] = d['pass.outcome.name'].isna()
    d['dest'] = celda(d.pass_end_x, d.pass_end_y)
PL['ori'] = celda(PL.loc_x, PL.loc_y)
GK['corto'] = GK.pass_end_x < X_MED
print(f'--- (0) Liga: saques de meta {len(GK):,} · cortos (termina antes de x = 40) {100 * GK.corto.mean():.1f}% · completos {100 * GK.ok.mean():.1f}%')
print(f'    Pases largos {len(PL):,} · completos {100 * PL.ok.mean():.1f}% · nulos de destino: saque {int(GK.pass_end_x.isna().sum())} · largo {int(PL.pass_end_x.isna().sum())}')

# ---------- (1) Etapas ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
MAP = ptp[['match_id', 'team', 'etapa_id']].drop_duplicates(['match_id', 'team']).rename(columns={'team': 'team.name'})
GK = GK.merge(MAP, on=['match_id', 'team.name'], how='left'); PL = PL.merge(MAP, on=['match_id', 'team.name'], how='left')
print(f'--- (1) Con etapa: saques {100 * GK.etapa_id.notna().mean():.1f}% · largos {100 * PL.etapa_id.notna().mean():.1f}% (deben ser ~100)')

cnt = lambda s: np.bincount(np.asarray(s, int), minlength=96).tolist()
def mitades(d, col, ref, rng):
    """Estabilidad del reparto de destinos (diferencia contra la liga) entre dos mitades de partidos."""
    M = pd.crosstab(d.match_id, d[col]).reindex(columns=range(96), fill_value=0).to_numpy(float)
    if len(M) < 8: return None, None
    rs = []
    for _ in range(N_PART):
        m = rng.permutation(len(M)) < len(M) // 2
        a = M[m].sum(0) / M[m].sum() - ref; b = M[~m].sum(0) / M[~m].sum() - ref
        if a.std() > 0 and b.std() > 0: rs.append(np.corrcoef(a, b)[0, 1])
    r = float(np.mean(rs)); return round(r, 3), round(2 * r / (1 + r), 3) if r > 0 else round(r, 3)

def bloque(g, p):
    return {'gk': {'n': int(len(g)), 'partidos': int(g.match_id.nunique()), 'corto_pct': round(100 * g.corto.mean(), 2) if len(g) else None,
                   'dest': cnt(g.dest), 'ok': cnt(g.dest[g.ok])},
            'pl': {'n': int(len(p)), 'partidos': int(p.match_id.nunique()), 'ori': cnt(p.ori), 'dest': cnt(p.dest), 'ok': cnt(p.dest[p.ok])}}

OUT = {'version': 'G-10 v1.0', 'grilla': {'nx': 12, 'ny': 8, 'yd': 10}, 'liga': bloque(GK, PL), 'etapas': {}}
ref_gk = np.bincount(GK.dest, minlength=96) / len(GK); ref_pl = np.bincount(PL.dest, minlength=96) / len(PL)
rng = np.random.default_rng(SEMILLA); filas = []
for e in ETAPAS:
    g, p = GK[GK.etapa_id == e], PL[PL.etapa_id == e]
    if len(g) == 0 and len(p) == 0: print('AVISO: sin datos para', e); continue
    b = bloque(g, p); b['gk']['sh'] = mitades(g, 'dest', ref_gk, rng); b['pl']['sh'] = mitades(p, 'dest', ref_pl, rng)
    OUT['etapas'][e] = b
    pp = p.match_id.nunique()
    filas.append({'etapa': e, 'partidos': pp, 'saques': len(g), 'corto%': round(100 * g.corto.mean(), 1), 'saque_ok%': round(100 * g.ok.mean(), 1),
                  'saque_x_fin_med': round(g.pass_end_x.median(), 1), 'largos_x_part': round(len(p) / max(pp, 1), 1), 'largo_ok%': round(100 * p.ok.mean(), 1),
                  'largo_desde_1er_tercio%': round(100 * (p.loc_x < 40).mean(), 1), 'largo_a_ult_tercio%': round(100 * (p.pass_end_x >= 80).mean(), 1),
                  'estab_saque': b['gk']['sh'][1], 'estab_largo': b['pl']['sh'][1]})
T = pd.DataFrame(filas).set_index('etapa')
pd.set_option('display.width', 250)
print(f'\n--- (2) Liga: saque corto {100 * GK.corto.mean():.1f}% · x final mediana del saque {GK.pass_end_x.median():.1f} · largos por equipo-partido '
      f'{len(PL) / PL.groupby(["match_id", "team.name"]).ngroups:.1f} · largo completo {100 * PL.ok.mean():.1f}% · desde el 1er tercio {100 * (PL.loc_x < 40).mean():.1f}%'
      f' · al último tercio {100 * (PL.pass_end_x >= 80).mean():.1f}%')
print('--- (3) Por etapa (estab = Spearman-Brown de la diferencia contra la liga; verde ≥ 0.7, ámbar 0.5-0.7, rojo < 0.5)')
print(T.T.to_string())
ruta = os.path.join(SAL5, 'reporte_construccion.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False, separators=(',', ':'))
print(f'\nG-10 terminó en {time.time() - t0:.0f} s. Escribió {ruta} ({os.path.getsize(ruta) / 1024:.0f} KB). Súbelo o pégalo en el chat.')

  2021/2022: saques de meta 6,066 · pases largos 35,045 · pases de juego abierto 231,907 (4 s)
  2022/2023: saques de meta 6,386 · pases largos 37,067 · pases de juego abierto 255,490 (9 s)
  2023/2024: saques de meta 6,138 · pases largos 35,813 · pases de juego abierto 270,786 (14 s)
  2024/2025: saques de meta 6,190 · pases largos 35,948 · pases de juego abierto 271,522 (19 s)
  2025/2026: saques de meta 6,186 · pases largos 35,305 · pases de juego abierto 260,797 (23 s)
--- (0) Liga: saques de meta 30,966 · cortos (termina antes de x = 40) 50.0% · completos 69.3%
    Pases largos 179,178 · completos 57.0% · nulos de destino: saque 0 · largo 0
--- (1) Con etapa: saques 100.0% · largos 100.0% (deben ser ~100)

--- (2) Liga: saque corto 50.0% · x final mediana del saque 39.9 · largos por equipo-partido 52.7 · largo completo 57.0% · desde el 1er tercio 46.2% · al último tercio 45.0%
--- (3) Por etapa (estab = Spearman-Brown de la diferencia contra la liga; verde ≥ 0.7, ámbar 0.5-0.7, ro

In [4]:
# [G-9b v1.0] Córners: ocupación 360 en grilla fina + remates de primera y segunda jugada, por etapa (SOLO LECTURA; escribe JSON y una caché)
# =====================================================================
# Sesión 16 · 2026-10-21 · Corrige G-9 v1.0:
#   (a) Remates en 0: G-9 buscaba remates en la clase del primer contacto, pero en StatsBomb el cabezazo casi siempre trae antes una
#       recepción en el mismo instante, así que el contacto queda como "control". Aquí los remates son los TIROS de la ventana de R8a (BP-5):
#       primera jugada = tiros del que cobra hasta 1 s después del primer contacto; segunda = de 1 a 10 s, misma posesión, sin penales.
#       Se valida contra xg1 / xg2 de R8a córner por córner.
#   (b) Grilla fina (Arturo): celdas de 3 yd de profundidad × 4 yd de ancho, x 90-120 (10 filas, incluye la frontal) × y 16-64 desde la banda
#       del saque (12 columnas). Las líneas caen en el área chica (114), el punto penal (108), el borde del área (102) y los postes (36, 44).
#       Se vuelven a leer las fotos 360 (R8p solo guardó 20 celdas). Se valida recalculando las 20 celdas de R8p con las mismas fotos.
# Mismo marco que R8p: todo en coordenadas del equipo que COBRA, con y medida desde la banda del saque.
# Capas por córner: atacantes en el saque (sin el que cobra) · defensores en el saque · atacantes y defensores en el primer contacto.
# Estabilidad: igual que G-9 (diferencia contra la liga, dos mitades de partidos, 200 particiones, Spearman-Brown).
# Orden: celda 1 (Drive) → G-9b. Primera vez ~10-15 min (lee ~1,700 fotos 360; imprime avance). Después usa la caché (segundos).
# Pega TODA la salida y sube salidas/fw5/reporte_corners360b.json.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SEMILLA, N_PART = 2026, 200
X0, DX, NX, Y0, DY, NY = 90, 3, 10, 16, 4, 12              # grilla fina: x 90-120 (10 × 3 yd) · y 16-64 (12 × 4 yd)
NC = NX * NY
T_CONTACTO, T_PRIMERA, T_VENTANA = 5.0, 1.0, 10.0           # R8a
ETAPAS = ['Atlético San Luis · 2', 'América · 3', 'Puebla · 1', 'León · 4', 'Necaxa · 9', 'Cruz Azul · 9',
          'América · 2', 'Monterrey · 3', 'Santos Laguna · 6', 'América · 1']
REHACER = False
t0 = time.time()

def fina(px, yr):
    xi = np.floor((np.asarray(px, float) - X0) / DX); yi = np.floor((np.asarray(yr, float) - Y0) / DY)
    xi = np.where(np.asarray(px, float) >= 120, NX - 1, xi)                     # la línea de fondo cuenta en la última fila
    ok = (xi >= 0) & (xi < NX) & (yi >= 0) & (yi < NY)
    return np.where(ok, xi * NY + yi, -1).astype(int)
XB20, YB20 = [102, 108, 114, 117, 120.0001], [18, 30, 36, 44, 50, 62.0001]          # grilla de R8p, solo para validar
def c20(px, yr):
    xi = np.digitize(px, XB20) - 1; yi = np.digitize(yr, YB20) - 1
    return np.where((xi >= 0) & (xi <= 3) & (yi >= 0) & (yi <= 4), xi * 5 + yi, -1)

def capas(P, lado_bajo, girar=False, sin_actor=True):
    tm = P['teammate'].astype('boolean').fillna(False).to_numpy(bool); ac = P['actor'].astype('boolean').fillna(False).to_numpy(bool)
    kp = P['keeper'].astype('boolean').fillna(False).to_numpy(bool)
    px, py = P['x'].to_numpy(float), P['y'].to_numpy(float)
    if girar: px, py, tm = 120 - px, 80 - py, ~tm
    yr = py if lado_bajo else 80 - py
    a = tm & ~kp & (~ac if sin_actor else True); d = ~tm & ~kp
    cf, cv = fina(px, yr), c20(px, yr)
    return (np.bincount(cf[a & (cf >= 0)], minlength=NC), np.bincount(cf[d & (cf >= 0)], minlength=NC),
            np.bincount(cv[a & (cv >= 0)], minlength=20), np.bincount(cv[d & (cv >= 0)], minlength=20))

# ---------- (0) Córners, primer contacto y tiros de la ventana ----------
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
D = BP[(BP.kind == 'corner') & (BP.ruta == 'directo')][['match_id', 'index', 'period', 't', 'team', 'lado_bajo', 'contacto_tipo', 'xg1', 'xg2']].copy()
D['temporada'] = D.match_id.map(loc['temporada'])
D['rival'] = np.where(D.team == D.match_id.map(loc['local']), D.match_id.map(loc['visita']), D.match_id.map(loc['local']))
piezas, tiros = [], []
for temp, g in D.groupby('temporada'):
    ruta = os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"); disp = set(pq.read_schema(ruta).names)
    cols = ['match_id', 'index', 'period', 't', 'possession', 'type.name', 'team.name', 'loc_x', 'loc_y', 'shot.statsbomb_xg', 'shot.type.name',
            'ball_receipt.outcome.name'] + (['shot.body_part.name'] if 'shot.body_part.name' in disp else [])
    E = pd.read_parquet(ruta, columns=cols); E = E[E.match_id.isin(set(g.match_id))]
    g = g.merge(E[['match_id', 'index', 'possession']], on=['match_id', 'index'], how='left')
    F = E[(E['type.name'] != 'Pressure') & ~((E['type.name'] == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())]
    F = F.rename(columns={'index': 'c_index', 'period': 'c_period', 't': 'c_t', 'type.name': 'c_tipo', 'team.name': 'c_team'}); F['index'] = F['c_index']
    x = pd.merge_asof(g.sort_values('index'), F.sort_values('index')[['match_id', 'index', 'c_index', 'c_period', 'c_t', 'c_tipo', 'c_team']],
                      on='index', by='match_id', direction='forward', allow_exact_matches=False)
    val = (x.c_period == x.period) & (x.c_t <= x.t + T_CONTACTO)
    x.loc[~val, ['c_index', 'c_t', 'c_tipo', 'c_team']] = np.nan
    piezas.append(x)
    S = E[(E['type.name'] == 'Shot') & (E['shot.type.name'] != 'Penalty')].rename(columns={'team.name': 'team', 'index': 's_index', 't': 's_t'})
    tiros.append(S.drop(columns=['type.name', 'ball_receipt.outcome.name']))
    print(f'  {temp}: córners directos {len(g):,} ({time.time() - t0:.0f} s)', flush=True)
D = pd.concat(piezas, ignore_index=True); S = pd.concat(tiros, ignore_index=True)
print(f'--- (0) Primer contacto: tipo idéntico a R8a en {100 * (D.c_tipo.fillna("—") == D.contacto_tipo.fillna("—")).mean():.2f}% de {len(D):,} córners (debe ser 100)')
# Tiros de la ventana (réplica de R8a): mismo periodo y posesión, hasta 10 s tras el contacto (o tras el saque si no hubo contacto)
D['t_ref'] = D.c_t.fillna(D.t)
W = D[['match_id', 'index', 'period', 'possession', 'team', 't', 't_ref', 'c_t', 'lado_bajo']].merge(S, on=['match_id', 'period', 'possession', 'team'])
W = W[(W.s_t >= W.t) & (W.s_t <= W.t_ref + T_VENTANA) & (W.s_index > W['index'])].copy()
W['jugada'] = np.where(W.c_t.isna() | (W.s_t <= W.c_t + T_PRIMERA), 1, 2)
W['xg'] = W['shot.statsbomb_xg'].fillna(0.0)
W['cel'] = fina(W.loc_x, np.where(W.lado_bajo.astype(bool), W.loc_y, 80 - W.loc_y)); W.loc[W.cel < 0, 'cel'] = NC   # NC = fuera de la grilla
W['cabeza'] = W['shot.body_part.name'].eq('Head') if 'shot.body_part.name' in W else False
chk = D[['match_id', 'index', 'xg1', 'xg2']].merge(W.pivot_table(index=['match_id', 'index'], columns='jugada', values='xg', aggfunc='sum').reset_index(),
                                                   on=['match_id', 'index'], how='left').fillna(0)
ig1 = np.isclose(chk.xg1, chk.get(1, 0), atol=1e-6).mean(); ig2 = np.isclose(chk.xg2, chk.get(2, 0), atol=1e-6).mean()
print(f'    Tiros en la ventana: {len(W):,} (primera jugada {int((W.jugada == 1).sum()):,} · segunda {int((W.jugada == 2).sum()):,}) · '
      f'xG igual a R8a córner por córner: primera {100 * ig1:.2f}% · segunda {100 * ig2:.2f}% (deben ser ~100)')
print(f'    Remates fuera de la grilla (x < 90 o fuera de y 16-64): {100 * (W.cel == NC).mean():.1f}% · con cabeza: '
      f'{100 * W.cabeza.mean():.1f}%' + ('' if 'shot.body_part.name' in S else ' (no hay columna de parte del cuerpo)'))

# ---------- (1) Fotos 360 en grilla fina (con caché) ----------
H = pd.read_parquet(os.path.join(FW5, 'r8p_corners.parquet'))
H = H.merge(D[['match_id', 'index', 'lado_bajo', 'c_index', 'c_team']], on=['match_id', 'index'], how='left')
CACHE = os.path.join(SAL5, 'g9b_grilla_fina.parquet')
SA, SD, CA = [f'sa{i}' for i in range(20)], [f'sd{i}' for i in range(20)], [f'ca{i}' for i in range(20)]
if os.path.exists(CACHE) and not REHACER:
    G = pd.read_parquet(CACHE); print(f'--- (1) Grilla fina desde caché: {len(G):,} córners')
else:
    filas = []; mids = H.match_id.unique()
    for i, (mid, g) in enumerate(H.groupby('match_id')):
        if i % 200 == 0: print(f'  fotos 360: partido {i:,} de {len(mids):,} · {time.time() - t0:.0f} s', flush=True)
        r = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
        if not os.path.exists(r): continue
        nec = set(g['index']) | set(g['c_index'].dropna().astype(int))
        jj = pd.read_parquet(r, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y']); jj = jj[jj.event_index.isin(nec)]
        JJ = {k: v for k, v in jj.groupby('event_index')}
        for c in g.itertuples(index=False):
            if c.index not in JJ: continue
            A, Dd, A20, D20 = capas(JJ[c.index], bool(c.lado_bajo))
            f = {'match_id': mid, 'index': c.index, 'v_sa': A20.tolist(), 'v_sd': D20.tolist(), 'sa': A.tolist(), 'sd': Dd.tolist()}
            if pd.notna(c.c_index) and int(c.c_index) in JJ:
                A2, D2, _, _ = capas(JJ[int(c.c_index)], bool(c.lado_bajo), girar=c.c_team != c.team, sin_actor=False)
                f.update({'ca': A2.tolist(), 'cd': D2.tolist()})
            filas.append(f)
    G = pd.DataFrame(filas); G.to_parquet(CACHE, index=False)
G = G.merge(H[['match_id', 'index', 'team', 'rival'] + SA + SD + [CA[0]]], on=['match_id', 'index'])
ig_a = np.mean([np.array_equal(np.asarray(v), r) for v, r in zip(G.v_sa, G[SA].to_numpy())])
ig_d = np.mean([np.array_equal(np.asarray(v), r) for v, r in zip(G.v_sd, G[SD].to_numpy())])
print(f'    Validación: las 20 celdas de R8p recalculadas con las mismas fotos coinciden en atacantes {100 * ig_a:.2f}% · defensores {100 * ig_d:.2f}% (deben ser ~100)')
print(f'    Córners con grilla fina: {len(G):,} · con foto del contacto {100 * G.ca.notna().mean():.1f}% · ({time.time() - t0:.0f} s)')

# ---------- (2) Etapas y agregados ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
MAP = ptp[['match_id', 'team', 'etapa_id']].drop_duplicates(['match_id', 'team'])
def etapas(d):
    d = d.merge(MAP.rename(columns={'etapa_id': 'et_of'}), on=['match_id', 'team'], how='left')
    return d.merge(MAP.rename(columns={'team': 'rival', 'etapa_id': 'et_de'}), on=['match_id', 'rival'], how='left')
G = etapas(G); D = etapas(D)
W = W.merge(D[['match_id', 'index', 'et_of', 'et_de']], on=['match_id', 'index'], how='left')
print(f'--- (2) Con etapa: grilla {100 * G.et_of.notna().mean():.1f}% / {100 * G.et_de.notna().mean():.1f}% · tiros {100 * W.et_of.notna().mean():.1f}% (deben ser ~100)')

M = lambda d, c: np.vstack(d[c].map(np.asarray).to_numpy()).astype(float) if len(d) else np.zeros((0, NC))
r3 = lambda v: np.round(np.asarray(v, float), 3).tolist()
def remates(dc, w):
    out = {'n_dir': int(len(dc))}
    for j in (1, 2):
        x = w[w.jugada == j]
        out[f'rem{j}'] = np.bincount(x.cel, minlength=NC + 1).tolist()
        out[f'xg{j}'] = np.round(np.bincount(x.cel, weights=x.xg, minlength=NC + 1), 3).tolist()
        out[f'cab{j}'] = int(x.cabeza.sum())
    return out
def mitades(d, cols, ref, rng):
    if d.match_id.nunique() < 8: return None, None
    X = np.hstack([M(d, c) for c in cols]); ids, inv = np.unique(d.match_id, return_inverse=True)
    S_ = np.zeros((len(ids), X.shape[1])); np.add.at(S_, inv, X); N = np.bincount(inv).astype(float)
    rs = []
    for _ in range(N_PART):
        m = rng.permutation(len(ids)) < len(ids) // 2
        a = S_[m].sum(0) / N[m].sum() - ref; b = S_[~m].sum(0) / N[~m].sum() - ref
        if a.std() > 0 and b.std() > 0: rs.append(np.corrcoef(a, b)[0, 1])
    r = float(np.mean(rs)); return [round(r, 3), round(2 * r / (1 + r), 3) if r > 0 else round(r, 3)]
def bloque(go, gd, do_, dd_, wo, wd, rng=None, ref=None):
    gm = go[go.ca.notna()]; gdm = gd[gd.ca.notna()]
    of = {'n': int(len(go)), 'partidos': int(go.match_id.nunique()), 'sa': r3(M(go, 'sa').mean(0)), 'n_mov': int(len(gm)),
          'sa_m': r3(M(gm, 'sa').mean(0)), 'ca': r3(M(gm, 'ca').mean(0)), **remates(do_, wo)}
    de = {'n': int(len(gd)), 'partidos': int(gd.match_id.nunique()), 'sd': r3(M(gd, 'sd').mean(0)), 'sa': r3(M(gd, 'sa').mean(0)),
          'n_mov': int(len(gdm)), 'cd': r3(M(gdm, 'cd').mean(0)), 'ca': r3(M(gdm, 'ca').mean(0)), **remates(dd_, wd)}
    if rng is not None:
        of['sh'] = mitades(gm, ['sa', 'ca'], ref['of'], rng); de['sh'] = mitades(gd, ['sd'], ref['de'], rng)
    return {'of': of, 'de': de}
LIGA_ = bloque(G, G, D, D, W, W)
ref = {'of': np.r_[LIGA_['of']['sa_m'], LIGA_['of']['ca']], 'de': np.asarray(LIGA_['de']['sd'])}
OUT = {'version': 'G-9b v1.0', 'grilla': {'x0': X0, 'dx': DX, 'nx': NX, 'y0': Y0, 'dy': DY, 'ny': NY}, 'liga': LIGA_, 'etapas': {}}
rng = np.random.default_rng(SEMILLA); filas = []
for e in ETAPAS:
    go, gd = G[G.et_of == e], G[G.et_de == e]
    if len(go) == 0 and len(gd) == 0: print('AVISO: sin córners para', e); continue
    b = bloque(go, gd, D[D.et_of == e], D[D.et_de == e], W[W.et_of == e], W[W.et_de == e], rng, ref); OUT['etapas'][e] = b
    nf, nc = max(b['of']['n_dir'], 1), max(b['de']['n_dir'], 1)
    filas.append({'etapa': e, 'c_favor': b['of']['n'], 'c_contra': b['de']['n'],
                  'rem1_favor_x100': round(100 * sum(b['of']['rem1']) / nf, 1), 'rem2_favor_x100': round(100 * sum(b['of']['rem2']) / nf, 1),
                  'xg_favor_x100': round(100 * (sum(b['of']['xg1']) + sum(b['of']['xg2'])) / nf, 2),
                  'rem1_contra_x100': round(100 * sum(b['de']['rem1']) / nc, 1), 'rem2_contra_x100': round(100 * sum(b['de']['rem2']) / nc, 1),
                  'xg_contra_x100': round(100 * (sum(b['de']['xg1']) + sum(b['de']['xg2'])) / nc, 2),
                  'estab_of': (b['of']['sh'] or [None, None])[1], 'estab_de': (b['de']['sh'] or [None, None])[1]})
T = pd.DataFrame(filas).set_index('etapa')
nl = LIGA_['of']['n_dir']
print(f'\n--- (3) Liga por cada 100 córners directos: remates de primera jugada {100 * sum(LIGA_["of"]["rem1"]) / nl:.1f} · de segunda {100 * sum(LIGA_["of"]["rem2"]) / nl:.1f}'
      f' · xG ×100 {100 * (sum(LIGA_["of"]["xg1"]) + sum(LIGA_["of"]["xg2"])) / nl:.2f} (R8a: {100 * (D.xg1 + D.xg2).mean():.2f})')
r1 = np.array(LIGA_['of']['rem1'][:NC]).reshape(NX, NY)
print('    Remates de primera jugada por fila de profundidad (x 90→120, sumando el ancho):', r1.sum(1).tolist(), '· fuera', LIGA_['of']['rem1'][NC])
r2 = np.array(LIGA_['of']['rem2'][:NC]).reshape(NX, NY)
print('    Remates de segunda jugada por fila de profundidad:', r2.sum(1).tolist(), '· fuera', LIGA_['of']['rem2'][NC])
pd.set_option('display.width', 250)
print('\n--- (4) Por etapa (estab = Spearman-Brown de la diferencia contra la liga, grilla fina; verde ≥ 0.7, ámbar 0.5-0.7, rojo < 0.5)')
print(T.T.to_string())
ruta = os.path.join(SAL5, 'reporte_corners360b.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False, separators=(',', ':'))
print(f'\nG-9b terminó en {time.time() - t0:.0f} s. Escribió {ruta} ({os.path.getsize(ruta) / 1024:.0f} KB). Súbelo al chat.')

  2021/2022: córners directos 2,478 (3 s)
  2022/2023: córners directos 2,721 (5 s)
  2023/2024: córners directos 2,662 (10 s)
  2024/2025: córners directos 2,509 (13 s)
  2025/2026: córners directos 2,523 (15 s)
--- (0) Primer contacto: tipo idéntico a R8a en 100.00% de 12,893 córners (debe ser 100)
    Tiros en la ventana: 5,197 (primera jugada 3,226 · segunda 1,971) · xG igual a R8a córner por córner: primera 100.00% · segunda 100.00% (deben ser ~100)
    Remates fuera de la grilla (x < 90 o fuera de y 16-64): 2.6% · con cabeza: 0.0% (no hay columna de parte del cuerpo)
  fotos 360: partido 0 de 1,665 · 15 s
  fotos 360: partido 200 de 1,665 · 163 s
  fotos 360: partido 400 de 1,665 · 288 s
  fotos 360: partido 600 de 1,665 · 408 s
  fotos 360: partido 800 de 1,665 · 535 s
  fotos 360: partido 1,000 de 1,665 · 661 s
  fotos 360: partido 1,200 de 1,665 · 788 s
  fotos 360: partido 1,400 de 1,665 · 917 s
  fotos 360: partido 1,600 de 1,665 · 1017 s
    Validación: las 20 celdas de R8p